# 🛢️ ROGII H-Blend + Model-Package Sidecar

Base notebook: [nina2025 / ROGII h-blend v1](https://www.kaggle.com/code/nina2025/rogii-h-blend-v1)

This notebook keeps Nina's rank-aware h-blend as the public anchor and adds a sidecar prediction generated from an OOF-weighted model package when the attached CSVs match the active sample.

The sidecar is deliberately treated as a small correction, not a new dominant member. The current default is a 2% late-linear blend because the public anchor already contains a strong saved-model stack.

Core public-probe form:

```text
T_hblend = H(T_HoangPhuc, T_MarkCooper, T_RomanTamrazov)
T_sidecar = F_model_package(current Kaggle test; OOF-fitted package weights)
T_final  = 0.98 * T_hblend + 0.02 * T_sidecar
```

Optional gated candidates only apply sidecar weight where it agrees reasonably with the h-blend anchor:

```text
g_i = lambda_max / (1 + (abs(T_sidecar_i - T_hblend_i) / s)^2)
T_final_i = (1 - g_i) * T_hblend_i + g_i * T_sidecar_i
```

Hidden reruns need one extra guard. Nina's original notebook succeeds on hidden by letting the Hoang Phuc `Model.6` inference-only cell build a current-sample `submission.csv` and then exit. Static public CSVs are not valid if their ids do not match the live sample.

So this notebook has two regimes:

- Public/probe ids match: run the h-blend + model-package sidecar experiment.
- Current sample ids differ: switch to Nina-compatible hidden fallback and prioritize a valid original-model submission.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/ROGII50.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))

## Blend Mechanics

The h-blend is not a plain weighted average. For each row, predictions are sorted, then a small rank correction is added to the base model weight.

```text
row_weight(j, i) = base_weight(j) + rank_correction(rank_i(j))
```

The constraints are kept deliberately simple:

```text
sum(base_weight) = 1
sum(rank_correction) = 0
```

So the row-wise blend remains normalized while still reacting to which submission is high, middle, or low for that row.

```text
T_sort(i)   = sum_j row_weight(j, i) * T_j(i)
T_hblend(i) = asc_weight * T_asc(i) + desc_weight * T_desc(i)
```

The sidecar can enter in three ways:

| Mode | Formula | When to use |
|---|---|---|
| `late_linear` | `(1 - lambda) * T_hblend + lambda * T_sidecar` | default; conservative public probe |
| `gated_late_linear` | row-wise lambda shrinks as sidecar/h-blend disagreement grows | safer diagnostic when sidecar has occasional large shifts |
| `hblend_member` | `H(T_9.537, T_9.765, T_9.956, T_sidecar)` | aggressive probe; sidecar participates in rank correction |
| `off` | `H(T_9.537, T_9.765, T_9.956)` | original-anchor sanity check |


## Experiment Plan

Use this notebook as a public h-blend probe first, with a conservative hidden-safe fallback.

| Probe | Suggested setting | Purpose |
|---|---:|---|
| Public anchor + OOF-weighted sidecar | `RUN_GENERATED_MODEL_CELLS = False` | use attached `9.537/9.765/9.956.csv` plus the current model package when ids match the sample |
| Auto hidden fallback | `AUTO_ORIGINAL_RERUN_ON_ID_MISMATCH = True` | preserve Nina's original current-sample inference path when ids differ |
| Default sidecar | `SIDECAR_INTEGRATION_MODE = "late_linear"`, `SIDECAR_LATE_BLEND_WEIGHT = 0.005` | small correction on top of the strong anchor |
| Candidate probes | `late_linear` 0/0.5/1/2%, gated 1/1.5%, member 0.5/1/2% | compare sidecar strength without editing code cells |

The sidecar model package should have `blend_config.weight_source` or `manifest.weight_source` pointing to an OOF-fitted weight file. The notebook guards this by default so a stale manual-weight package does not get blended accidentally.

Important limitation: the original hidden fallback exits after generating a valid base submission, so it does not apply the sidecar blend. In that hidden/id-mismatch regime, the successful path is best understood as Hoang Phuc `Model.6` artifact inference, not as the full 3-way h-blend. A true hidden-safe h-blend would require non-exiting saved-model packages for all h-blend anchor members, not only static public CSVs.

Dataset note: Hoang Phuc's `ROGII v10 Fresh Artifacts` Dataset is only used when the retained `Model.6` fallback cell actually runs. It provides `manifest.json`, `feature_cols.json`, `inference_config.json`, saved LightGBM/CatBoost models, and TabICL context files for current-sample inference. In the normal public-probe path with `RUN_GENERATED_MODEL_CELLS = False`, the notebook uses the precomputed `9.537.csv` from Nina's h-blend CSV Dataset instead.


## Controls

Edit the first code cell only. It contains the score-changing knobs.

| Knob | Meaning |
|---|---|
| `PUBLIC_PROBE_MODE` | `True` keeps public CSV/artifact blending behavior for public LB probing |
| `AUTO_ORIGINAL_RERUN_ON_ID_MISMATCH` | `True` auto-switches when attached public CSV ids do not match the current sample |
| `ORIGINAL_HIDDEN_COMPATIBILITY_MODE` | `True` preserves Nina's tested hidden path: run `Model.6` inference and exit with `submission.csv` |
| `HIDDEN_SAFE_ORIGINAL_RERUN` | manual switch for the same original-compatible fallback |
| `RUN_GENERATED_MODEL_CELLS` | `False` uses attached CSVs; `True` runs retained model cells |
| `SIDECAR_INTEGRATION_MODE` | `late_linear`, `gated_late_linear`, `hblend_member`, or `off` |
| `HBLEND_*` | original 3-way h-blend weights |
| `SIDECAR_HBLEND_*` | 4-way h-blend weights when sidecar is a ranked member |
| `SIDECAR_LATE_BLEND_WEIGHT` | final linear sidecar weight when using `late_linear`; default is deliberately tiny because the anchor h-blend is already strong |
| `SIDECAR_GATED_MAX_WEIGHT`, `SIDECAR_GATED_SCALE` | row-wise gated sidecar cap and disagreement scale; defaults shrink sidecar aggressively when it disagrees with the anchor |
| `SIDECAR_SOURCE_MODE` | `build_from_model_package` by default; branch `preds` and existing CSV modes remain for diagnostics |
| `SIDECAR_REQUIRE_OOF_WEIGHTED_PACKAGE` | fail fast if the model package does not advertise OOF-fitted weights |
| `SIDECAR_SUBMISSION_NAME` | exact sidecar CSV name, or `None` to auto-select/build |
| `WRITE_ADDITIONAL_SUBMISSION_CANDIDATES` | save member, late-linear, and gated sidecar probes |
| `ROGII v10 Fresh Artifacts` Dataset | needed only for hidden/id-mismatch fallback when `Model.6` is rerun |

If ids mismatch and `ORIGINAL_HIDDEN_COMPATIBILITY_MODE = True`, sidecar blending is deliberately disabled. That is not a bug; it keeps the same hidden-safe behavior as the successful public reference notebook.


In [ ]:
from pathlib import Path

# Model-package sidecar default: validate against the live competition sample first.
PUBLIC_PROBE_MODE = False

# If attached public h-blend CSVs do not match the current sample ids, auto-switch.
# Default True because static public CSVs are invalid for hidden id sets.
AUTO_ORIGINAL_RERUN_ON_ID_MISMATCH = True

# Preserve the successful reference notebook behavior on hidden reruns:
# Model.6 writes a current-sample submission.csv and exits before h-blend/sidecar cells.
ORIGINAL_HIDDEN_COMPATIBILITY_MODE = True

# Set True manually for the same original-compatible hidden fallback.
HIDDEN_SAFE_ORIGINAL_RERUN = False

# Pure blender by default when ids match. The auto guard turns this on only when needed.
RUN_GENERATED_MODEL_CELLS = False

# How to attach the sidecar stack:
#   "late_linear"        -> original 3-way h-blend, then small final linear sidecar correction
#   "gated_late_linear"  -> row-wise late blend; sidecar weight shrinks when disagreement is large
#   "hblend_member"      -> sidecar is the 4th rank-aware h-blend submission
#   "off"                -> original 3-way h-blend only
SIDECAR_INTEGRATION_MODE = "late_linear"

# Standalone sidecar source mode:
#   "build_from_model_package" -> build sidecar from saved models/features for the current sample
#   "build_from_preds"          -> read branch OOF/test artifacts and fit sidecar stack here
#   "existing_submission"       -> require an already-built sidecar submission CSV
#   "existing_or_build"         -> use existing CSV if present, otherwise build from preds
SIDECAR_SOURCE_MODE = "build_from_model_package"

# Use the OOF-fitted step-5 model package by default. Set False only for manual package diagnostics.
SIDECAR_REQUIRE_OOF_WEIGHTED_PACKAGE = True
SIDECAR_EXPECTED_WEIGHT_SOURCE_TOKEN = "oof"

# Sidecar stack inference mode when building from branch predictions.
# Use "test_foldavg" for CV-faithful private-safe behavior; "test_alltrain" if alltrain artifacts exist.
SIDECAR_STACK_TEST_PREDICTION_TYPE = "test_foldavg"
SIDECAR_STACK_TEST_PRIORITY = [
    "test_foldavg",
    "test_alltrain",
    "test_public_aggressive",
    "test_alltrain_public_aggressive",
]
SIDECAR_STACK_MIN_TRAIN_NON_NULL_RATE = 0.99
SIDECAR_STACK_MIN_TEST_NON_NULL_RATE = 0.999
SIDECAR_STACK_L2_GRID = [0.0, 1e-5, 1e-4, 1e-3]
SIDECAR_STACK_REQUIRE_STRONG_BRANCHES = False

# Nina h-blend input CSV roots. These should contain 9.537.csv, 9.765.csv, 9.956.csv.
HBLEND_INPUT_ROOTS = [
    Path("/kaggle/input/datasets/nina2025/rogii-03"),
    Path("/kaggle/input/rogii-03"),
]


# Human-readable registry. The legacy Model.* keys are kept only because Nina's original
# fallback cells use them as execution guards.
HBLEND_MEMBER_REGISTRY = [
    {
        "display_name": "Hoang Phuc v10 Fresh Artifact",
        "author": "Hoang Phuc (K18 HCM)",
        "legacy_model_key": "Model.6",
        "submission_name": "9.537",
        "public_lb": "9.537",
        "notebook": "ROGII v10 Fresh Artifact Infer",
        "url": "https://www.kaggle.com/code/thbdh5765/rogii-v10-fresh-artifact-infer",
        "role": "dominant h-blend anchor",
    },
    {
        "display_name": "Mark Cooper imitation Tasmim",
        "author": "Mark Cooper",
        "legacy_model_key": "Model.7",
        "submission_name": "9.765",
        "public_lb": "9.765",
        "notebook": "imitation-tasmim-lgb-xgb",
        "url": "https://www.kaggle.com/code/markjcooper/imitation-tasmim-lgb-xgb",
        "role": "secondary diversity anchor",
    },
    {
        "display_name": "Roman Tamrazov Better Solution",
        "author": "Roman Tamrazov",
        "legacy_model_key": "Model.4",
        "submission_name": "9.956",
        "public_lb": "9.956",
        "notebook": "[ROGII] BETTER SOLUTION . LB: 9.956",
        "url": "https://www.kaggle.com/code/romantamrazov/rogii-better-solution-lb-9-956",
        "role": "compact Roman physics/formula diversity",
    },
]
HBLEND_MEMBER_BY_SUBMISSION = {m["submission_name"]: m for m in HBLEND_MEMBER_REGISTRY}
HBLEND_MEMBER_BY_LEGACY_KEY = {m["legacy_model_key"]: m for m in HBLEND_MEMBER_REGISTRY}

# Original Nina 3-way h-blend weights.
HBLEND_MODEL6_WEIGHT = 0.81
HBLEND_MODEL7_WEIGHT = 0.15
HBLEND_MODEL4_WEIGHT = 0.04
HBLEND_RANK_CORRECTION_WEIGHTS = [+0.10, -0.03, -0.07]

# Shared asc/desc rank mix.
HBLEND_ASC_WEIGHT = 0.30
HBLEND_DESC_WEIGHT = 0.70

# 4-way h-blend member setting. This is more aggressive than late-linear, so it is no longer the default.
SIDECAR_HBLEND_MODEL6_WEIGHT = 0.77
SIDECAR_HBLEND_MODEL7_WEIGHT = 0.14
SIDECAR_HBLEND_MODEL4_WEIGHT = 0.04
SIDECAR_HBLEND_WEIGHT = 0.05
SIDECAR_HBLEND_NAME = "sidecar_stack"
SIDECAR_HBLEND_RANK_CORRECTION_WEIGHTS = [+0.08, +0.02, -0.03, -0.07]

# Submission notebook output controls.
HBLEND_DETAILS = False
WRITE_ADDITIONAL_SUBMISSION_CANDIDATES = True
SIDECAR_MEMBER_CANDIDATE_WEIGHTS = [0.005, 0.01, 0.02]
LATE_LINEAR_CANDIDATE_WEIGHTS = [0.0, 0.005, 0.01, 0.02]
GATED_LATE_LINEAR_CANDIDATES = [(0.01, 4.0), (0.015, 4.0)]
# Candidate member probes use a softer rank correction because sidecar is only a small add-on.
SIDECAR_CANDIDATE_RANK_CORRECTION_WEIGHTS = [+0.05, +0.01, -0.02, -0.04]

# Late-linear fallback. Used when SIDECAR_INTEGRATION_MODE == "late_linear".
# Keep this small; the public h-blend anchor is strong and sidecar is only a correction.
SIDECAR_LATE_BLEND_WEIGHT = 0.005

# Gated late-linear fallback. Used when SIDECAR_INTEGRATION_MODE == "gated_late_linear".
# The effective sidecar weight is max_weight / (1 + (abs(sidecar - hblend) / scale) ** 2).
# Small cap + small scale makes this a conservative disagreement-aware correction.
SIDECAR_GATED_MAX_WEIGHT = 0.01
SIDECAR_GATED_SCALE = 4.0

# Set this to a specific existing sidecar file name when you want an exact candidate.
# Example: "submission_branch_level_residual_blend_foldavg.csv"
SIDECAR_SUBMISSION_NAME = None

# If True, missing or unusable sidecar artifacts raise an error whenever sidecar mode needs them.
SIDECAR_STRICT = True

# Sidecar artifact dataset roots. The first one is the current attached path.
SIDECAR_ARTIFACT_ROOTS = [
    Path("/kaggle/input/datasets/pilkwang/rogii-branch-artifacts"),
    Path("/kaggle/input/rogii-branch-artifacts"),
    Path("/kaggle/input/rogii-branch-artifacts/rogii_artifacts"),
    Path("/kaggle/input/notebooks/pilkwang/rogii-branch-prediction-factory/rogii_artifacts"),
]

# Hidden-safe sidecar model package roots. The Dataset should contain
# metadata/model_package_manifest.json, feature_builders/, stacking/, and models/.
SIDECAR_MODEL_PACKAGE_ROOTS = [
    Path("/kaggle/input/datasets/pilkwang/rogii-model-package"),
    Path("/kaggle/input/rogii-model-package"),
    Path("/kaggle/input/rogii-branch-artifacts/rogii_model_package"),
]

SIDECAR_PREFERRED_FILES = [
    "submission_branch_level_residual_blend_foldavg.csv",
    "submission_safe_gate_blend_super_top3_foldavg.csv",
    "submission_safe_gate_blend_super_top3_compact_foldavg.csv",
    "submission_safe_gate_blend_top2_physics_foldavg.csv",
    "submission_safe_gate_blend_foldavg.csv",
    "submission_constrained_residual_blend_l2_0e_00_foldavg.csv",
]

_valid_modes = {"hblend_member", "late_linear", "gated_late_linear", "off"}
_valid_source_modes = {"build_from_model_package", "build_from_preds", "existing_submission", "existing_or_build"}
if HIDDEN_SAFE_ORIGINAL_RERUN:
    RUN_GENERATED_MODEL_CELLS = True
    if ORIGINAL_HIDDEN_COMPATIBILITY_MODE:
        SIDECAR_INTEGRATION_MODE = "off"
    elif SIDECAR_SOURCE_MODE != "build_from_model_package":
        SIDECAR_INTEGRATION_MODE = "off"

if SIDECAR_INTEGRATION_MODE not in _valid_modes:
    raise ValueError(f"SIDECAR_INTEGRATION_MODE must be one of {_valid_modes}")
if SIDECAR_SOURCE_MODE not in _valid_source_modes:
    raise ValueError(f"SIDECAR_SOURCE_MODE must be one of {_valid_source_modes}")

print({
    "public_probe_mode": PUBLIC_PROBE_MODE,
    "auto_original_rerun_on_id_mismatch": AUTO_ORIGINAL_RERUN_ON_ID_MISMATCH,
    "hidden_safe_original_rerun": HIDDEN_SAFE_ORIGINAL_RERUN,
    "original_hidden_compatibility_mode": ORIGINAL_HIDDEN_COMPATIBILITY_MODE,
    "run_generated_model_cells": RUN_GENERATED_MODEL_CELLS,
    "sidecar_integration_mode": SIDECAR_INTEGRATION_MODE,
    "sidecar_source_mode": SIDECAR_SOURCE_MODE,
    "sidecar_require_oof_weighted_package": SIDECAR_REQUIRE_OOF_WEIGHTED_PACKAGE,
    "sidecar_expected_weight_source_token": SIDECAR_EXPECTED_WEIGHT_SOURCE_TOKEN,
    "sidecar_stack_test_prediction_type": SIDECAR_STACK_TEST_PREDICTION_TYPE,
    "hblend_asc_desc": [HBLEND_ASC_WEIGHT, HBLEND_DESC_WEIGHT],
    "original_hblend_weights": [HBLEND_MODEL6_WEIGHT, HBLEND_MODEL7_WEIGHT, HBLEND_MODEL4_WEIGHT],
    "original_rank_correction": HBLEND_RANK_CORRECTION_WEIGHTS,
    "sidecar_hblend_weights": [
        SIDECAR_HBLEND_MODEL6_WEIGHT,
        SIDECAR_HBLEND_MODEL7_WEIGHT,
        SIDECAR_HBLEND_MODEL4_WEIGHT,
        SIDECAR_HBLEND_WEIGHT,
    ],
    "sidecar_hblend_rank_correction": SIDECAR_HBLEND_RANK_CORRECTION_WEIGHTS,
    "sidecar_late_blend_weight": SIDECAR_LATE_BLEND_WEIGHT,
    "sidecar_gated_max_weight": SIDECAR_GATED_MAX_WEIGHT,
    "sidecar_gated_scale": SIDECAR_GATED_SCALE,
    "sidecar_member_candidate_weights": SIDECAR_MEMBER_CANDIDATE_WEIGHTS,
    "late_linear_candidate_weights": LATE_LINEAR_CANDIDATE_WEIGHTS,
    "gated_late_linear_candidates": GATED_LATE_LINEAR_CANDIDATES,
    "sidecar_candidate_rank_correction": SIDECAR_CANDIDATE_RANK_CORRECTION_WEIGHTS,
    "hblend_details": HBLEND_DETAILS,
    "write_additional_candidates": WRITE_ADDITIONAL_SUBMISSION_CANDIDATES,
    "sidecar_submission_name": SIDECAR_SUBMISSION_NAME,
    "sidecar_model_package_roots": [str(p) for p in SIDECAR_MODEL_PACKAGE_ROOTS],
})


In [ ]:
import pandas as pd
from IPython.display import display

HBLEND_MEMBER_OVERVIEW = pd.DataFrame(HBLEND_MEMBER_REGISTRY)[[
    "display_name", "author", "submission_name", "public_lb", "legacy_model_key", "role", "notebook"
]]
HBLEND_MEMBER_OVERVIEW["anchor_weight"] = HBLEND_MEMBER_OVERVIEW["submission_name"].map({
    "9.537": HBLEND_MODEL6_WEIGHT,
    "9.765": HBLEND_MODEL7_WEIGHT,
    "9.956": HBLEND_MODEL4_WEIGHT,
})
HBLEND_MEMBER_OVERVIEW["sidecar_member_weight"] = HBLEND_MEMBER_OVERVIEW["submission_name"].map({
    "9.537": SIDECAR_HBLEND_MODEL6_WEIGHT,
    "9.765": SIDECAR_HBLEND_MODEL7_WEIGHT,
    "9.956": SIDECAR_HBLEND_MODEL4_WEIGHT,
})
display(HBLEND_MEMBER_OVERVIEW)

SIDECAR_OVERVIEW = pd.DataFrame([
    {
        "component": "sidecar_stack",
        "enabled": SIDECAR_INTEGRATION_MODE != "off",
        "mode": SIDECAR_INTEGRATION_MODE,
        "source_mode": SIDECAR_SOURCE_MODE,
        "member_weight": SIDECAR_HBLEND_WEIGHT,
        "late_linear_weight": SIDECAR_LATE_BLEND_WEIGHT,
        "note": "small model-package correction; hidden id-mismatch fallback deliberately disables sidecar",
    }
])
display(SIDECAR_OVERVIEW)


In [ ]:
import numpy as np
import pandas as pd

import os,ast,shutil,copy

import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

from bokeh.plotting import figure, gridplot 
from bokeh.io import output_file, show, output_notebook
output_notebook()


def bokeh_show(
        params,
        df_cross,
        show_figures1, 
        show_figures2, wps_fig2,
        color_cross):

    colors = [subm['color'] for subm in params['subm']]
    
    def dossier(js,subms,cols):
        def quant(i,js,subms,cols):
            return {"c" : i, "q" : sum([1 for subm in cols[i] if subm == subms[js]])}
        return {
            'name' : subms[js],
            'q_in' : [quant(i,js,subms,cols) for i in range(len(subms))]
        }
    alls = pd.read_csv(f'tida_desc.csv')
    matrix = [ast.literal_eval(str(row.alls)) for row in alls.itertuples()]
    subms = sorted(matrix[0])
    cols = [[data[i] for data in matrix] for i in range(len(subms))]
    df_subms = pd.DataFrame({f'col_{i}': [x[i] for x in matrix] for i in range(len(subms))})
    dossiers = [dossier(js,subms,cols) for js in range(len(subms))]
    subm_names = [one_dossier['name'] for one_dossier in dossiers]
    figures1,qss,i = [],[],0
    height = 100 if len(colors)==2\
        else 134 if len(colors)==3 else (154 if len(colors)==4 else 174)
    for one_dossier in dossiers: 
        i_col = 'alls. ' + str(one_dossier['q_in'][i]['c'])
        qs = [one['q'] for one in one_dossier['q_in']]
        x_names = [name.replace("Group","").replace("subm_","") for name in subm_names]
        width = 140
        f = figure(x_range=x_names,width=width, height=height, title=i_col)
        f.vbar(x=x_names, width=0.585, top=qs, color=colors)
        figures1.append(f)
        qss.append(qs)
        i+=1
    grid = gridplot([figures1])
    output_file('tida_alls.html')
    if show_figures1 == True: show(grid)
    sub_wts = params['subwts']
    main_wts = [subm['weight'] for subm in params['subm']]
    mms,acc_mass = [],[]
    for j in range(len(dossiers)):
        one_dossier = dossiers[j]
        qs = [one['q'] for one in one_dossier['q_in']]
        mm = [qs[h] * (main_wts[j] + sub_wts[h]) for h in range(len(sub_wts))]
        mass = sum(mm)
        mms.append(mm)
        acc_mass.append(round(mass))                        #subm_names[::-1]
    y_names = [name + " - " + str(mass) for name,mass in zip(subm_names,acc_mass)]
    f1 = figure(y_range=y_names, width=270, height=height, title='relations of general masses')
    f1.hbar(y=y_names, height=0.555, right=acc_mass, left=0, color=colors)
    output_file('tida_alls2.html')
    alls = [f'alls.{i}' for i in range(len(dossiers))]
    subm = [f'sub{i}'   for i in range(len(dossiers))] 
    mmsT  = np.asarray(mms).T
    data = {'cols' : alls}
    for i in range(len(dossiers)): data[f'sub{i}'] = mmsT[i,:]
    f2 = figure(y_range=alls, height=height, width=270, title="relations of columns masses")
    f2.hbar_stack(subm, y='cols', height=0.555, color=colors, source=data)
    qssT  = np.asarray(qss).T
    data = {'cols' : alls}
    for i in range(len(dossiers)): data[f'sub{i}'] = qssT[i,:]
    f3 = figure(y_range=alls, height=height, width=245, title="ratios in columns")
    f3.hbar_stack(subm, y='cols', height=0.555, color=colors, source=data)
    grid = gridplot([[f3,f2,f1]])
    show(grid)
    if show_figures2 == True:
        def read(params,i):
            FiN = params["path"] + params["subm"][i]["name"] + ".csv"
            target_name_back = {'target':params["target"],'pred':params["target"]}
            return pd.read_csv(FiN).rename(columns=target_name_back)
        dfs = [read(params,i) for i in range(len(params["subm"]))] + [df_cross]
        _height = 358 if len(params["subm"]) == 11 else 254
        f   = figure(width=785, height=_height)
        f.title.text = 'Click on legend entries to mute the corresponding lines'
        b,e        = 21000,21154
        line_x     = [dfs[i][b:e]['id']         for i in range(len(dfs))]
        line_y     = [dfs[i][b:e]['tvt'] for i in range(len(dfs))]
        color      = colors + [color_cross]
        alpha      = [0.8 for i in range(len(dfs)-1)] + [0.95]
        lws        = [1.0 for i in range(len(dfs)-1)] + [1.00]
        legend = subm_names + ['cross']
        for i in range(len(legend)):
            f.line(line_x[i], line_y[i], line_width=lws[i], color=color[i], alpha=alpha[i],
                   muted_color='white',legend_label=legend[i])
        f.legend.location = "top_left"
        f.legend.click_policy="mute"
        show(f)

# An example of working with Seaborn is taken from a notebook:
# https://www.kaggle.com/code/likithagedipudi/kaggle-success-factors
# Presented by an expert from Buffalo, New York, United States - Likitha Gedipudi
# https://www.kaggle.com/likithagedipudi

def seaborn_display_1(params,
                      df_cross,
                      show_figures1, show_figures2, color_cross):

    colors = [subm['color'] for subm in params['subm']]
    
    def dossier(js,subms,cols):
        def quant(i,js,subms,cols):
            return {"c" : i, "q" : sum([1 for subm in cols[i] if subm == subms[js]])}
        return {
            'name' : subms[js],
            'q_in' : [quant(i,js,subms,cols) for i in range(len(subms))]
        }
    alls = pd.read_csv(f'tida_desc.csv')
    matrix = [ast.literal_eval(str(row.alls)) for row in alls.itertuples()]
    subms = sorted(matrix[0])
    cols = [[data[i] for data in matrix] for i in range(len(subms))]
    df_subms = pd.DataFrame({f'col_{i}': [x[i] for x in matrix] for i in range(len(subms))})
    dossiers = [dossier(js,subms,cols) for js in range(len(subms))]
    subm_names = [one_dossier['name'] for one_dossier in dossiers]
    
    nqs,qss,i = [],[],0
    
    for one_dossier in dossiers: 
        qs = [one['q'] for one in one_dossier['q_in']]
        x_names = [name.replace("Group","").replace("subm_","") for name in subm_names]
        qss.append(qs)
        nqs.append({'n':x_names, 'q':qs})
        i+=1
    
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['font.size']      = 7
    plt.rcParams['axes.titlesize'] = 9
    plt.rcParams['axes.labelsize'] = 8

    len_nqs = len(nqs) if len(nqs) > 3 else 4

    fig, axes = plt.subplots(2, len_nqs, figsize=(2.1*len_nqs, 4))
    
    def ric(j, i, n, q, colors):
        ax1 = axes[j, i]
        bars1 = ax1.bar(n, q, color=colors)
        ax1.set_title(f'ratios in alls.col {i}', fontweight='bold')
        ax1.set_xticklabels(n, rotation=45, ha='right')
        i = 0
        for bar, val in zip(bars1, q):
            if i != 0:
                ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
                         f'{val:,.0f}', ha='center', va='bottom', fontsize=8)
            i += 1

    lm_max = 0
    for i in range(len(nqs)):
        for q in nqs[i]['q']:
            if q > lm_max: lm_max = q  
    lm_colors = ['whitesmoke'] + colors
    for nq in nqs:
        nq['n'].insert(0,'')
        nq['q'].insert(0, lm_max*1.13)

    for i in range(len(nqs)): ric(0, i, nqs[i]['n'], nqs[i]['q'], lm_colors)

    ax10 = axes[1, 0]
    #--------------------------------------------------------------------------------
    sub_wts = params['subwts']
    main_wts = [subm['weight'] for subm in params['subm']]
    mms,acc_mass = [],[]
    for j in range(len(dossiers)):
        one_dossier = dossiers[j]
        qs = [one['q'] for one in one_dossier['q_in']]
        mm = [qs[h] * (main_wts[j] + sub_wts[h]) for h in range(len(sub_wts))]
        mass = sum(mm)
        mms.append(mm)
        acc_mass.append(round(mass))
    acc_mass = acc_mass
    y_names = [str(mass) + " - " + name for name,mass in zip(subm_names,acc_mass)]
    colors  .insert(0,'whitesmoke')
    y_names .insert(0,'')
    acc_mass.insert(0, max(acc_mass) * 1.13)
    irbis = ax10.barh(y_names[::-1], acc_mass[::-1], color=colors[::-1])
    ax10.set_title('relations of g.masses', fontweight='bold')
    ax10.set_xlabel('weights')


    ax11 = axes[1, 1]
    #--------------------------------------------------------------------------------
    corr_matrix = alls[subm_names].corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    mask = 0.5*mask
    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', 
                center=0,fmt='.3f', square=True, linewidths=0.5, ax=ax11,
                cbar_kws={'shrink': 0.8})
    ax11.set_title('Correlation Matrix', fontweight='bold', fontsize=9)
    ax11.set_xticklabels(subm_names, rotation=45, ha='right')

    ax13 = axes[1,3]
    #--------------------------------------------------------------------------------
    ip1 = len(nqs)-4
    ip2 = len(nqs)-2
    sample  = alls.sample(min(100_000, len(alls)), random_state=42)
    scatter = ax13.scatter(np.log10(sample[subm_names[ip2]] + 1), 
                           np.log10(sample[subm_names[ip1]] + 1), 
                           c=sample['tvt'], cmap='viridis', alpha=0.5, s=10)
    ax13.set_title(f'{subm_names[ip2]} vs {subm_names[ip1]}', fontweight='bold', fontsize=8)
    plt.colorbar(scatter, ax=ax13, label='Score')

    if len(nqs)>=5:
        ax14 = axes[1,4]
        #--------------------------------------------------------------------------------
        ip1 = len(nqs)-1
        ip2 = len(nqs)-4
        sample  = alls.sample(min(100_000, len(alls)), random_state=42)
        scatter = ax14.scatter(np.log10(sample[subm_names[ip2]] + 1), 
                               np.log10(sample[subm_names[ip1]] + 1), 
                               c=sample['tvt'], cmap='viridis', alpha=0.5, s=10)
        ax14.set_title(f'{subm_names[ip2]} vs {subm_names[ip1]}', fontweight='bold', fontsize=8)
        plt.colorbar(scatter, ax=ax14, label='Score')
        
    if len(nqs)>=6:
        ax15 = axes[1,5]
        #--------------------------------------------------------------------------------
        ip1 = 1
        ip2 = 3
        sample  = alls.sample(min(100_000, len(alls)), random_state=42)
        scatter = ax15.scatter(np.log10(sample[subm_names[ip2]] + 1), 
                               np.log10(sample[subm_names[ip1]] + 1), 
                               c=sample['tvt'], cmap='viridis', alpha=0.5, s=10)
        ax15.set_title(f'{subm_names[ip2]} vs {subm_names[ip1]}', fontweight='bold', fontsize=8)
        plt.colorbar(scatter, ax=ax15, label='Score')

    
    ax12 = axes[1, 2]
    #--------------------------------------------------------------------------------
    colors = [subm['color'] for subm in params['subm']]
    
    for subm,color in zip(subm_names,colors):
        subm_data = alls[subm]
        ax12.hist(np.log10(subm_data + 1), bins=30, alpha=0.5, label=subm, color=color)
    ax12.set_title('(Log Scale)', fontweight='bold')
    ax12.set_xlabel('log10')
    ax12.set_ylabel('Frequency')
    ax12.legend()
 
    plt.tight_layout()
    plt.show()


def seaborn_display_2(params, file_name_cross=""):
    plt.figure(figsize=(9, 2.5))
    for subm in params['subm']:
        pred = pd.read_csv(params['path']+subm['name']+'.csv')[params['id_target'][1]]
        sns.kdeplot(pred, label = subm['name'], linewidth = 0.5)
    if file_name_cross != '':
        pred = pd.read_csv(file_name_cross)[params['id_target'][1]]
        sns.kdeplot(pred, label = 'blend', linewidth = 1, linestyle = 'dashed')
    plt.title("KDE")
    plt.xlabel("target")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def matrix_vs(path,fs_names):
    def load(path,fs_names):
        dfs = [pd.read_csv(path + name_subm +'.csv') for name_subm in fs_names]
        for i in range(len(dfs)):
            dfs[i] = dfs[i].rename(columns={"tvt": f'{fs_names[i]}'})
        dfsm = pd.merge(dfs[0], dfs[1], on="id")
        for i in range(2,len(dfs)):
            dfsm = pd.merge(dfsm,dfs[i],on='id')
        return dfsm   
    def make_list_vs(fs_names):
        list = []
        for i in range(0,len(fs_names)-1):
            for j in range(i+1,len(fs_names)):
                list.append(fs_names[i] + "_vs_" + fs_names[j])
        return list
    def get_mvs(dfs, list_vs):
        def get_abs_distance(x,t1,t2):
            return abs(x[t1]-x[t2])
        for vs in list_vs:
            t = vs.split('_vs_')
            dfs[vs] = dfs.apply(lambda x: get_abs_distance(x,t[0],t[1]), axis=1)
        return dfs   
    def distance_vs(name, st_names, list_vs, dfs):
        distances = []
        for st in st_names:
            vs_between = name + "_vs_" + st
            if vs_between not in list_vs:
                distances.append(0)
            else: distances.append(round(dfs[vs_between].sum()))
        return distances
    dfs = load(path,fs_names)
    list_vs = make_list_vs(fs_names)
    mvs = get_mvs(dfs, list_vs)
    m1 = pd.DataFrame({'subm':fs_names})
    m2 = pd.DataFrame({ name :distance_vs(name, fs_names, list_vs, mvs) for name in fs_names})
    matrix = pd.concat([m1,m2],axis=1)
    return matrix


def display_distances(params):
    files = [subm['name'] for subm in params['subm']]
    distances = matrix_vs ( params['path'], files )            
    display(distances)


def arr_colors(color):
    dskb,mvr = 'deepskyblue','mediumvioletred'
    sg = ['darkgray','silver','gainsboro']
    if color=='red'   or color=='R': return ['firebrick','red','crimson','tomato']     + sg
    if color=='Red'   or color=='r': return ['red','tomato','crimson']                 + sg
    if color=='Green' or color=='G': return ['darkgreen','limegreen','green','lime']   + sg
    if color=='Blue'  or color=='B': return ['midnightblue','blue','mediumblue',dskb]  + sg
    if color=='RGB'   or color=='S': return ['mediumblue','darkgreen','crimson']       + sg
    if color=='RGBM'  or color=='M': return [mvr,'darkorchid','darkmagenta','magenta'] + sg
    return ['black','dimgray','gray'] + sg


def convert(schema):
    colors = arr_colors(schema[2])
    dicts  = [
        {'name': schema[0][i],'weight':schema[1][i],'color':colors[i]} 
        for i in range(len(schema[0]))
    ]
    return {'subm':dicts}


def h_blend(
        params, _update={},
        cross='silver',
        details=False,
        fig1=False, fig2=False, wf2=555, 
        dtls=False, dist=False, subm=''):

    if isinstance(params, list): params = convert(params)

    if 'path' in _update: params.update(_update)
    
    color_cross, dk  = cross, copy.deepcopy(params)

    if details == True:
        dist = True
        show_details,show_figures1,show_figures2 = True,True,True
    else:
        show_details,show_figures1,show_figures2 = dtls,fig1,fig2
        
    file_short_names = [subm['name'] for subm in params['subm']]
    type_sort    = params['type_sort'][0]
    dk['asc']    = params['type_sort'][1]
    dk['desc']   = params['type_sort'][2]
    dk['id']     = params['id_target'][0]
    dk['target'] = params['id_target'][1]

    if 'task' in params: dk['task'] = params['task']
    
    def read(dk,i):
        tnm = dk["subm"][i]["name"]
        FiN = dk["path"] + tnm + ".csv"
        df = pd.read_csv(FiN).rename(columns={'target':tnm, 'pred':tnm, dk["target"]:tnm})
        if 'task' in dk and dk['task'][0]=='round':
            df[tnm] = df[tnm].round(dk['task'][1])
        return df 
        
    def merge(dfs_subm):
        df_subms = pd.merge(dfs_subm[0],  dfs_subm[1], on=[dk['id']])
        for i in range(2, len(dk["subm"])): 
            df_subms = pd.merge(df_subms, dfs_subm[i], on=[dk['id']])
        return df_subms
        
    def da(dk,sorting_direction,show_details):
        
        df_subms = merge([read(dk,i) for i in range(len(dk["subm"]))])
        cols = [col for col in df_subms.columns if col != dk['id']]
        short_name_cols = [c for c in cols]
        
        def alls1(x, sd=sorting_direction,cs=cols):
            reverse = True if sd=='desc' else False
            tes = {c: x[c] for c in cs}.items()
            subms_sorted = [t[0] for t in sorted(tes,key=lambda k:k[1],reverse=reverse)]
            return subms_sorted

        import random

        def alls2(x, sd=sorting_direction,cs=cols):
            reverse = True if sd=='desc' else False
            tes = {c: x[c] for c in cs}.items()
            subms_random = [t[0] for t in tes]
            random.shuffle(subms_random)
            return subms_random

        alls = alls1 if type_sort == 'asc/desc' else alls2
            
        def summa(x,cs,wts,ic_alls): 
            return sum([x[cs[j]] * (wts[0][j] + wts[1][ic_alls[j]]) for j in range(len(cs))])
            
        wts = [[[e['weight'] for e in dk["subm"]], [w for w in dk["subwts"]]]]
          
        def correct(x, cs=cols, wts=wts):
            i = [x['alls'].index(c) for c in short_name_cols]
            return summa(x,cs,wts[0],i)

        if len(wts) == 1:
            correct_sub_weights = [wt for wt in dk["subwts"]]
            weights = [subm['weight'] for subm in dk["subm"]]
            def correct(x, cs=cols, w=weights, cw=correct_sub_weights):
                ic = [x['alls'].index(c) for c in short_name_cols]
                cS = [x[cols[j]] * (w[j] + cw[ic[j]]) for j in range(len(cols))]
                return sum(cS)
                
        if len(wts) > 1 or "subwts2" in dk:

            wts = [
                [[e['weight'] for e in dk["subm"]], [w for w in dk["subwts" ]]],
                [[e['weight'] for e in dk["subm2"]],[w for w in dk["subwts2"]]],
                [[e['weight'] for e in dk["subm3"]],[w for w in dk["subwts3"]]],
            ]

            def correct(x, cs=cols, wts=wts, diff=dk['different'], seg=dk['segment']): 
                i = [x['alls'].index(c) for c in short_name_cols]
                if   seg[0][0] < x['mx-m'] <= seg[0][1]: return summa(x,cs,wts[diff[0]],i)
                if   seg[1][0] < x['mx-m'] <= seg[1][1]: return summa(x,cs,wts[diff[1]],i)
                else:                                    return summa(x,cs,wts[diff[2]],i)

                   
        def amxm(x, cs=cols):
            list_values = x[cs].to_list()
            mxm = abs(max(list_values)-min(list_values))
            return mxm

        if len(wts) > 1 or "subwts2" in dk or 'task' in dk and 'mxm' in dk['task']:
            df_subms['mx-m']   = df_subms.apply(lambda x: amxm   (x), axis=1)
        df_subms['alls']       = df_subms.apply(lambda x: alls   (x), axis=1)
        df_subms[dk["target"]] = df_subms.apply(lambda x: correct(x), axis=1)
        schema_rename = { old_nc:new_shnc for old_nc, new_shnc in zip(cols, short_name_cols) }
        df_subms = df_subms.rename(columns=schema_rename)
        df_subms = df_subms.rename(columns={dk["target"]:"ensemble"})
        df_subms.insert(loc=1, column=' _ ', value=['   '] * len(df_subms))
        df_subms[' _ '] = df_subms[' _ '].astype(str)
        pd.set_option('display.max_rows',100)
        pd.set_option('display.float_format', '{:.5f}'.format)
        if len(wts) > 1 or 'task' in dk and 'mxm' in dk['task']: 
            vcols = [dk['id']] + [' _ '] + short_name_cols + [' _ '] + ['mx-m'] + [' _ '] +\
                      ['alls'] + [' _ '] + ['ensemble']
        else:
            vcols = [dk['id']] + [' _ '] + short_name_cols + [' _ '] +\
                      ['alls'] + [' _ '] + ['ensemble']
        df_subms = df_subms[vcols]
        if show_details and sorting_direction=='desc': display(df_subms.head(5))
        pd.set_option('display.float_format', '{:.5f}'.format)
        df_subms = df_subms.rename(columns={"ensemble":dk["target"]})
        if sorting_direction=='desc': 
            df_subms.to_csv(f'tida_{sorting_direction}.csv', index=False)
        return df_subms[[dk['id'],dk['target']]]
   
    def ensemble_da(dk,        show_details): 
        dfD    = da(dk,'desc', show_details)
        dfA    = da(dk,'asc',  show_details)
        dfA[dk['target']] = dk['desc']*dfD[dk['target']] + dfA[dk['target']]*dk['asc']
        return dfA

    da = ensemble_da(dk,show_details)

    # bokeh_show(dk, da, show_figures1, False, wf2, color_cross)

    if subm != '': da.to_csv(subm, index=False)

    seaborn_display_1(dk, da, show_figures1, False, color_cross)

    display_distances(params)

    seaborn_display_2(params, file_name_cross=subm)
    
    return  da

## h-blend

In [ ]:
params_h_blend = {
  'path'      : f'/kaggle/input/datasets/nina2025/rogii-03/',
  'id_target' : ['id', "tvt"],
  'type_sort' : ['asc/desc', HBLEND_ASC_WEIGHT, HBLEND_DESC_WEIGHT],
  'subwts'    : list(HBLEND_RANK_CORRECTION_WEIGHTS),
  'subm'      : [
      {'model': f'Model.6', 'name': f'9.537', 'weight': HBLEND_MODEL6_WEIGHT, 'color': f'navy'},
      {'model': f'Model.7', 'name': f'9.765', 'weight': HBLEND_MODEL7_WEIGHT, 'color': f'darkorange'},
      {'model': f'Model.4', 'name': f'9.956', 'weight': HBLEND_MODEL4_WEIGHT, 'color': f'darkgreen'},
  ],
}

print('Initial h-blend params ready. Legacy Model.6/7/4 keys map to Hoang Phuc / Mark Cooper / Roman Tamrazov.')


In [ ]:
import os
subm_names = [subm['name'] for subm in params_h_blend['subm']]

for file in subm_names + 'cross,tida_desc,demo_submission'.split(','):
    if os.path.isfile(file + '.csv'):
        os.remove(file + '.csv')


In [ ]:
from pathlib import Path
import pandas as pd


def _early_read_csv_ids(path: Path):
    if not path.exists():
        return None
    frame = pd.read_csv(path, usecols=['id'])
    return set(frame['id'].astype(str))


def _early_find_current_sample() -> Path | None:
    candidates = [
        Path('/kaggle/input/rogii-wellbore-geology-prediction/sample_submission.csv'),
        Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv'),
    ]
    root = Path('/kaggle/input')
    if root.exists():
        candidates.extend(sorted(root.glob('*/sample_submission.csv')))
        candidates.extend(sorted(root.glob('*/*/sample_submission.csv')))
        candidates.extend(sorted(root.glob('**/sample_submission.csv')))
    seen = set()
    for path in candidates:
        key = path.as_posix()
        if key in seen or not path.exists():
            continue
        seen.add(key)
        # Prefer a real competition/data directory over artifact metadata.
        parent = path.parent
        if 'rogii-wellbore-geology-prediction' in key.lower() or ((parent / 'train').exists() and (parent / 'test').exists()):
            return path
    for path in candidates:
        if path.exists():
            return path
    return None


def _early_find_hblend_csv(name: str) -> Path | None:
    file_name = f'{name}.csv'
    for root in HBLEND_INPUT_ROOTS:
        path = Path(root) / file_name
        if path.exists():
            return path
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.glob(f'**/{file_name}'))
        if matches:
            return matches[0]
    return None


def _early_hblend_inputs_match_current_sample() -> tuple[bool, dict]:
    sample_path = _early_find_current_sample()
    report = {'sample_path': str(sample_path) if sample_path else '', 'checked': []}
    if sample_path is None:
        return True, report
    sample_ids = _early_read_csv_ids(sample_path)
    for name in ['9.537', '9.765', '9.956']:
        path = _early_find_hblend_csv(name)
        row = {'name': name, 'path': str(path) if path else '', 'matches': False, 'missing': None, 'extra': None}
        if path is None:
            report['checked'].append(row)
            return False, report
        ids = _early_read_csv_ids(path)
        row['missing'] = len(sample_ids - ids)
        row['extra'] = len(ids - sample_ids)
        row['matches'] = row['missing'] == 0 and row['extra'] == 0
        report['checked'].append(row)
        if not row['matches']:
            return False, report
    return True, report


if not HIDDEN_SAFE_ORIGINAL_RERUN and AUTO_ORIGINAL_RERUN_ON_ID_MISMATCH:
    ok, early_report = _early_hblend_inputs_match_current_sample()
    print('early_hblend_id_compatibility:', early_report)
    if not ok:
        print('Attached h-blend CSVs do not match the current sample ids; switching to Nina-compatible hidden fallback.')
        HIDDEN_SAFE_ORIGINAL_RERUN = True
        RUN_GENERATED_MODEL_CELLS = True
        if ORIGINAL_HIDDEN_COMPATIBILITY_MODE:
            SIDECAR_INTEGRATION_MODE = 'off'
        elif SIDECAR_SOURCE_MODE != 'build_from_model_package':
            SIDECAR_INTEGRATION_MODE = 'off'

if HIDDEN_SAFE_ORIGINAL_RERUN:
    RUN_GENERATED_MODEL_CELLS = True
    if ORIGINAL_HIDDEN_COMPATIBILITY_MODE:
        SIDECAR_INTEGRATION_MODE = 'off'
    elif SIDECAR_SOURCE_MODE != 'build_from_model_package':
        SIDECAR_INTEGRATION_MODE = 'off'

print({
    'after_early_guard_hidden_safe_original_rerun': HIDDEN_SAFE_ORIGINAL_RERUN,
    'after_early_guard_original_hidden_compatibility_mode': ORIGINAL_HIDDEN_COMPATIBILITY_MODE,
    'after_early_guard_run_generated_model_cells': RUN_GENERATED_MODEL_CELLS,
    'after_early_guard_sidecar_integration_mode': SIDECAR_INTEGRATION_MODE,
})


In [ ]:
print('Model-cell gate ready. RUN_GENERATED_MODEL_CELLS controls whether heavy model cells execute.')


# Models

In [ ]:
ensemble_of_solutions = [subm['model'] for subm in params_h_blend['subm']] if RUN_GENERATED_MODEL_CELLS else []

print('RUN_GENERATED_MODEL_CELLS:', RUN_GENERATED_MODEL_CELLS)
print('ensemble_of_solutions:', ensemble_of_solutions)


## Hoang Phuc (K18 HCM) | 9.537 fallback branch

Legacy execution key: `Model.6`

Source: [ROGII v10 Fresh Artifact Infer](https://www.kaggle.com/code/thbdh5765/rogii-v10-fresh-artifact-infer) by [Hoang Phuc (K18 HCM)](https://www.kaggle.com/thbdh5765). This cell is disabled by default and kept only for explicit Nina-style reruns when precomputed CSVs do not match the active sample.


In [ ]:
if 'Model.6' in ensemble_of_solutions:
        
    print('\n\n','Model.6','\n\n')
    
    import gc
    import json
    import multiprocessing
    import os
    import sys
    import time
    import traceback
    import warnings
    from functools import lru_cache
    from pathlib import Path
    from typing import Optional
    from concurrent.futures import ThreadPoolExecutor
    
    import lightgbm as lgb
    import numpy as np
    import pandas as pd
    from catboost import CatBoostRegressor, Pool
    from scipy.interpolate import interp1d
    from scipy.signal import savgol_filter
    from scipy.spatial import cKDTree
    from sklearn.linear_model import Ridge
    from sklearn.metrics import root_mean_squared_error
    from sklearn.model_selection import GroupKFold
    
    warnings.filterwarnings("ignore")
    
    # Kaggle inference notebook defaults: load the artifact dataset we created locally.
    os.environ.setdefault("ROGII_INFERENCE_ONLY", "1")
    os.environ.setdefault("ROGII_SAVE_ARTIFACTS", "0")
    os.environ.setdefault("ROGII_RUN_TABICL", "1")
    
    # ── optional numba JIT ──────────────────────────────────────────────────────
    try:
        from numba import njit as _njit
        _NUMBA = True
    except ImportError:
        def _njit(*a, **kw):
            def _wrap(f): return f
            return _wrap
        _NUMBA = False
    
    print("NUMBA:", _NUMBA)
    
    SEED = 42
    np.random.seed(SEED)
    # Use ALL available cores — Kaggle typically gives 4 (sometimes 2×4 on multi-GPU)
    def env_flag(name: str, default: bool = False) -> bool:
        value = os.environ.get(name)
        if value is None:
            return bool(default)
        return value.strip().lower() not in {"0", "false", "no", "off", ""}
    
    
    def env_int(name: str, default: int) -> int:
        value = os.environ.get(name)
        if value is None or not str(value).strip():
            return int(default)
        return int(value)
    
    
    RUNNING_ON_KAGGLE = (
        Path("/kaggle/input").exists()
        or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
        or bool(os.environ.get("KAGGLE_URL_BASE"))
    )
    
    NCPU = max(1, min(multiprocessing.cpu_count(), env_int("ROGII_NCPU", multiprocessing.cpu_count())))
    print(f"NCPU={NCPU}")
    
    # ══════════════════════════════════════════════════════════════════════════════
    # DEBUG CONFIG
    # ══════════════════════════════════════════════════════════════════════════════
    DBG_VERBOSE        = env_flag("ROGII_DEBUG_VERBOSE", False)
    DBG_SINGLE_WELL    = env_flag("ROGII_SMOKE_WELL", False)
    DBG_SINGLE_TEST    = env_flag("ROGII_SMOKE_TEST", False)
    DBG_PARALLEL_STATS = env_flag("ROGII_PARALLEL_STATS", True)
    DBG_NAN_AUDIT      = env_flag("ROGII_NAN_AUDIT", False)
    DBG_FEATURE_AUDIT  = env_flag("ROGII_FEATURE_AUDIT", False)
    DBG_MAX_ERRORS     = 20
    _dbg_error_count   = 0
    
    _T0_GLOBAL = time.time()
    
    
    def _dbg(msg: str, arr: np.ndarray = None, level: str = "INFO") -> None:
        if not DBG_VERBOSE:
            return
        elapsed = time.time() - _T0_GLOBAL
        prefix  = f"[{elapsed:8.1f}s][{level}] "
        print(prefix + msg, file=sys.stderr, flush=True)
        if arr is not None and isinstance(arr, np.ndarray) and arr.size > 0:
            try:
                nans = int(np.isnan(arr.astype(float)).sum())
                infs = int(np.isinf(arr.astype(float)).sum())
                print(
                    f"{prefix}  shape={arr.shape} dtype={arr.dtype} "
                    f"min={np.nanmin(arr):.4g} max={np.nanmax(arr):.4g} "
                    f"nan={nans} inf={infs}",
                    file=sys.stderr, flush=True,
                )
            except Exception:
                print(f"{prefix}  shape={arr.shape} dtype={arr.dtype} (stats failed)",
                      file=sys.stderr, flush=True)
    
    
    def _dbg_df(tag: str, df: pd.DataFrame) -> None:
        if not DBG_VERBOSE:
            return
        elapsed = time.time() - _T0_GLOBAL
        nan_cols = df.isnull().sum()
        nan_cols = nan_cols[nan_cols > 0]
        print(
            f"[{elapsed:8.1f}s][DF] {tag}: shape={df.shape}  "
            f"nan_cols={len(nan_cols)}/{len(df.columns)}",
            file=sys.stderr, flush=True,
        )
        if len(nan_cols) > 0 and DBG_VERBOSE:
            top = nan_cols.nlargest(10)
            print(f"  top nan cols: {dict(top)}", file=sys.stderr, flush=True)
    
    
    def _log_error(ctx: str, exc: Exception) -> None:
        global _dbg_error_count
        _dbg_error_count += 1
        if _dbg_error_count > DBG_MAX_ERRORS:
            if _dbg_error_count == DBG_MAX_ERRORS + 1:
                print(f"[ERROR] Max error log limit ({DBG_MAX_ERRORS}) reached.",
                      file=sys.stderr, flush=True)
            return
        elapsed = time.time() - _T0_GLOBAL
        tb = traceback.format_exc()
        print(
            f"[{elapsed:8.1f}s][ERROR] {ctx}: {type(exc).__name__}: {exc}\n{tb}",
            file=sys.stderr, flush=True,
        )
    
    
    # ── data paths ──────────────────────────────────────────────────────────────
    def _find() -> Path:
        candidates = []
        if os.environ.get("ROGII_DATA_DIR"):
            candidates.append(Path(os.environ["ROGII_DATA_DIR"]))
        candidates.extend([
            Path("/kaggle/input/rogii-wellbore-geology-prediction"),
            Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
            Path.cwd(),
            Path.cwd() / "rogii-wellbore-geology-prediction",
            Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd(),
            Path(__file__).resolve().parents[2] / "rogii-wellbore-geology-prediction" if "__file__" in globals() else Path.cwd(),
        ])
        for p in candidates:
            if (p / "train").is_dir() and (p / "test").is_dir() and (p / "sample_submission.csv").is_file():
                return p
        input_root = Path("/kaggle/input")
        if input_root.exists():
            for sample in input_root.glob("**/sample_submission.csv"):
                p = sample.parent
                if (p / "train").is_dir() and (p / "test").is_dir():
                    return p
        raise FileNotFoundError("Data not found")
    
    
    def _default_output_dir() -> Path:
        if RUNNING_ON_KAGGLE:
            return Path("/kaggle/working")
        if os.environ.get("ROGII_OUTPUT_DIR"):
            return Path(os.environ["ROGII_OUTPUT_DIR"])
        return Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    
    
    DATA       = _find()
    TRAIN_DIR  = DATA / "train"
    TEST_DIR   = DATA / "test"
    SAMPLE     = DATA / "sample_submission.csv"
    OUTPUT_DIR = _default_output_dir()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    OUT        = OUTPUT_DIR / "submission.csv"
    
    SAVE_ARTIFACTS = env_flag("ROGII_SAVE_ARTIFACTS", False)
    ARTIFACT_DIR = Path(os.environ.get("ROGII_ARTIFACT_DIR", OUTPUT_DIR / "model_artifacts"))
    ARTIFACT_MANIFEST = {
        "version": "v10_fresh_artifact_train",
        "goal": "public_score_9.5",
        "lgb": [],
        "catboost": [],
        "tabicl_contexts": [],
        "created_outputs": {},
    }
    
    
    def _json_default(obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, Path):
            return str(obj)
        raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")
    
    
    def _artifact_rel(path: Path) -> str:
        try:
            return str(path.relative_to(ARTIFACT_DIR)).replace("\\", "/")
        except ValueError:
            return str(path).replace("\\", "/")
    
    
    def save_json(path: Path, data) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(data, indent=2, default=_json_default), encoding="utf-8")
    
    
    def load_json(path: Path):
        return json.loads(path.read_text(encoding="utf-8"))
    
    
    def find_artifact_dir() -> Path:
        candidates: list[Path] = []
        if os.environ.get("ROGII_ARTIFACT_DIR"):
            candidates.append(Path(os.environ["ROGII_ARTIFACT_DIR"]))
        candidates.append(ARTIFACT_DIR)
        for root in [Path("/kaggle/input"), DATA, OUTPUT_DIR]:
            if root.exists():
                candidates.extend(p.parent for p in root.glob("**/manifest.json"))
        for candidate in dict.fromkeys(candidates):
            if (candidate / "manifest.json").exists():
                return candidate
        raise FileNotFoundError(
            "No artifact manifest found. Set ROGII_ARTIFACT_DIR to the model_artifacts folder."
        )
    
    
    if SAVE_ARTIFACTS:
        ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Artifact saving enabled: {ARTIFACT_DIR}", flush=True)
    
    
    def env_float(name: str, default: float) -> float:
        value = os.environ.get(name)
        if value is None or not str(value).strip():
            return float(default)
        return float(value)
    
    
    def apply_exact_train_coordinate_blend(sub: pd.DataFrame, data_dir: Path) -> pd.DataFrame:
        """Optional public train-coordinate blend; disabled with ROGII_EXACT_OVERLAP=0."""
        if os.environ.get("ROGII_EXACT_OVERLAP", "1").strip().lower() in {"0", "false", "no"}:
            print("Exact train-coordinate overlap override disabled.", flush=True)
            return sub
        blend_weight = env_float("ROGII_EXACT_BLEND_WEIGHT", 0.28)
        blend_weight = float(np.clip(blend_weight, 0.0, 1.0))
    
        train_parts = []
        for p in sorted((data_dir / "train").glob("*__horizontal_well.csv")):
            try:
                cur = pd.read_csv(p, usecols=["X", "Y", "Z", "TVT"])
            except Exception:
                continue
            cur = cur[cur["TVT"].notna()].copy()
            if not cur.empty:
                train_parts.append(cur)
        if not train_parts:
            print("Exact overlap: no train coordinate rows found.", flush=True)
            return sub
    
        train_all = pd.concat(train_parts, ignore_index=True)
        for col in ["X", "Y", "Z"]:
            train_all[col + "_r"] = train_all[col].round(2)
        train_map = (
            train_all
            .drop_duplicates(subset=["X_r", "Y_r", "Z_r"])
            .set_index(["X_r", "Y_r", "Z_r"])["TVT"]
            .to_dict()
        )
    
        coord_parts = []
        for p in sorted((data_dir / "test").glob("*__horizontal_well.csv")):
            wid = p.name.split("__")[0]
            try:
                cur = pd.read_csv(p, usecols=["X", "Y", "Z", "TVT_input"])
            except Exception:
                continue
            mask = cur["TVT_input"].isna().to_numpy()
            if not mask.any():
                continue
            row_idx = np.arange(len(cur))[mask]
            part = cur.loc[mask, ["X", "Y", "Z"]].copy()
            part["id"] = [f"{wid}_{int(i)}" for i in row_idx]
            coord_parts.append(part)
        if not coord_parts:
            print("Exact overlap: no test prediction rows found.", flush=True)
            return sub
    
        coord = pd.concat(coord_parts, ignore_index=True)
        coord["key"] = list(zip(coord["X"].round(2), coord["Y"].round(2), coord["Z"].round(2)))
        coord["exact_tvt"] = coord["key"].map(train_map)
        exact = coord[coord["exact_tvt"].notna()][["id", "exact_tvt"]]
        if exact.empty:
            print("Exact overlap: 0 rows replaced.", flush=True)
            return sub
    
        out = sub.merge(exact, on="id", how="left")
        mask = out["exact_tvt"].notna()
        out.loc[mask, "tvt"] = (
            (1.0 - blend_weight) * out.loc[mask, "tvt"].astype(float)
            + blend_weight * out.loc[mask, "exact_tvt"].astype(float)
        )
        print(
            f"Exact overlap blend: blended {int(mask.sum())}/{len(out)} rows "
            f"with weight={blend_weight:.3f}.",
            flush=True,
        )
        return out[["id", "tvt"]]
    
    TRAIN_CORE_CACHE_NAME = "aeroridge_train_core_df.pkl"
    CACHE_ONLY            = env_flag("ROGII_CACHE_ONLY", False)
    USE_TRAIN_CORE_CACHE  = env_flag("ROGII_USE_TRAIN_CORE_CACHE", True)
    WRITE_TRAIN_CORE_CACHE = env_flag("ROGII_WRITE_TRAIN_CORE_CACHE", not RUNNING_ON_KAGGLE)
    RUN_TABICL            = env_flag("ROGII_RUN_TABICL", True)
    INFERENCE_ONLY        = env_flag("ROGII_INFERENCE_ONLY", False)
    SAVE_FEATURE_FRAMES   = env_flag("ROGII_SAVE_FEATURE_FRAMES", False)
    USE_HILL_STACK        = env_flag("ROGII_USE_HILL_STACK", True)
    HILL_PRECISION        = env_float("ROGII_HILL_PRECISION", 0.01)
    DEBUG_MAX_TRAIN_WELLS = env_int("ROGII_DEBUG_MAX_TRAIN_WELLS", 0)
    DEBUG_MAX_TEST_WELLS  = env_int("ROGII_DEBUG_MAX_TEST_WELLS", 0)
    
    
    def _cache_candidates() -> list[Path]:
        paths = []
        if os.environ.get("ROGII_TRAIN_CORE_CACHE"):
            paths.append(Path(os.environ["ROGII_TRAIN_CORE_CACHE"]))
        for root in [Path("/kaggle/input"), OUTPUT_DIR]:
            if root.exists():
                paths.extend(root.glob(f"**/{TRAIN_CORE_CACHE_NAME}"))
        return list(dict.fromkeys(paths))
    
    
    def _train_core_cache_path_for_write() -> Path:
        if os.environ.get("ROGII_TRAIN_CORE_CACHE_OUT"):
            return Path(os.environ["ROGII_TRAIN_CORE_CACHE_OUT"])
        return OUTPUT_DIR / TRAIN_CORE_CACHE_NAME
    
    
    print(f"DATA={DATA}")
    print(f"OUTPUT_DIR={OUTPUT_DIR}")
    
    FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
    PLANE_K    = 10
    DENSE_SPW  = 60
    DENSE_K    = 20
    N_SPLITS   = 5
    N_AUG_SPLITS      = 1#3
    MIN_KNOWN_FOR_AUG = 20
    
    '''BEAMS = [
        (10, 20.0, 144.0, 3, "cons"),
        (10,  8.0,  64.0, 3, "loose"),
        ( 8, 35.0, 220.0, 1, "vcons"),
        (10, 14.0,  90.0, 5, "sm5"),
        (20,  4.0,  36.0, 3, "vloose"),
        (12, 12.0, 100.0, 3, "mid"),
        (15, 25.0, 180.0, 2, "stiff"),
    ]'''
    BEAMS = [
        (10, 20.0, 144.0, 3, "cons"),
        (10,  8.0,  64.0, 3, "loose"),
        (10, 14.0,  90.0, 5, "sm5"),
    ]
    
    PF_N = 150; ANCC_N = 150
    PF_MOM = 0.993; PF_VN = 0.005; PF_PN = 0.01
    PF_GR_SIG_MIN = 10.0; PF_GR_SIG_MAX = 60.0; PF_GR_SIG_DEF = 30.0
    PF_INIT_V_STD = 0.02; PF_INIT_SPR = 0.5; PF_RESAMP = 0.5
    PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003; PF_GR_WIN = 5; PF_GR_WT = 0.3
    ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005
    ANCC_IR = 0.01; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001
    
    NOTEBOOK_RUN_VERSION = "ROGII_v26_combinedbest_lgbfix_cb3_tabicl_splitgpu_2026_05_10"
    
    LGB_P = dict(
        boosting_type="gbdt",
        learning_rate=0.04,
        num_leaves=127,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5.0,
        reg_alpha=0.1,
        objective="regression",
        verbose=-1,
        n_jobs=-1,
        device_type="gpu",
        gpu_use_dp=False,
        max_bin=255,
    )
    LGB_SEEDS = [42, 7, 123]
    
    import subprocess as _s
    
    
    def _gpu_names() -> list[str]:
        try:
            out = _s.run(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                capture_output=True, text=True, check=False,
            ).stdout.strip()
            return [line.strip() for line in out.splitlines() if line.strip()]
        except Exception:
            return []
    
    
    GPU_NAMES = _gpu_names()
    GPU_COUNT = len(GPU_NAMES)
    CATBOOST_DEVICES = os.environ.get(
        "ROGII_CATBOOST_DEVICES",
        ":".join(str(i) for i in range(GPU_COUNT)) if GPU_COUNT else "0",
    )
    
    CB_P = dict(
        iterations=5000, learning_rate=0.04, depth=8, l2_leaf_reg=3.0,
        min_data_in_leaf=20, loss_function="RMSE",
        task_type="GPU", devices=CATBOOST_DEVICES, od_type="Iter", od_wait=150, verbose=0,
    )
    FORCE_CPU = env_flag("ROGII_FORCE_CPU", False)
    if FORCE_CPU:
        LGB_P["device_type"] = "cpu"
        LGB_P.pop("gpu_use_dp", None)
        LGB_P.pop("max_bin", None)
        CB_P["task_type"] = "CPU"
        CB_P.pop("devices", None)
    
    print("GPUs:", "\n".join(GPU_NAMES) if GPU_NAMES else "(none)")
    print(f"CatBoost devices: {CATBOOST_DEVICES}")
    print(f"FORCE_CPU={FORCE_CPU}")
    print(f"CPUs={NCPU}  train={len(list(TRAIN_DIR.glob('*__horizontal_well.csv')))} wells")
    
    
    # ═══════════════════════════════════════════════════════════════════════════════
    # Helpers
    # ═══════════════════════════════════════════════════════════════════════════════
    
    def nn_idx(arr: np.ndarray, v: float) -> int:
        i = int(np.searchsorted(arr, v, "left"))
        n = len(arr)
        if i >= n:
            return n - 1
        if i > 0 and abs(arr[i - 1] - v) <= abs(arr[i] - v):
            return i - 1
        return i
    
    
    def robust_slope(x: np.ndarray, y: np.ndarray) -> float:
        x = np.asarray(x, float); y = np.asarray(y, float)
        m = np.isfinite(x) & np.isfinite(y)
        if m.sum() < 2 or np.std(x[m]) < 1e-6:
            return 0.0
        return float(np.polyfit(x[m], y[m], 1)[0])
    
    
    def affine_cal(kgr: np.ndarray, tw_at_k: np.ndarray, min_pts: int = 20):
        v = np.isfinite(kgr) & np.isfinite(tw_at_k)
        if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
            bias = float(np.nanmean(kgr) - np.nanmean(tw_at_k)) if v.any() else 0.0
            return 1.0, bias
        a, b = np.polyfit(tw_at_k[v], kgr[v], 1)
        return float(a), float(b)
    
    
    def wls_b_well(ktvt: np.ndarray, kz: np.ndarray, form_kn_col: np.ndarray,
                   decay: float = 0.02) -> float:
        n = len(ktvt)
        if n < 3:
            return float(np.median(ktvt + kz - form_kn_col))
        w = np.exp(decay * np.arange(n, dtype=np.float64))
        w /= w.sum()
        return float(np.dot(w, ktvt + kz - form_kn_col))
    
    
    def safe_interp(x, xp, fp):
        xp = np.asarray(xp, float); fp = np.asarray(fp, float)
        m = np.isfinite(xp) & np.isfinite(fp)
        if m.sum() < 2:
            return np.full(len(np.asarray(x)), np.nan)
        o = np.argsort(xp[m])
        return np.interp(np.asarray(x, float), xp[m][o], fp[m][o],
                         left=np.nan, right=np.nan)
    
    
    def _rolling_std_cumsum(a: np.ndarray, w: int) -> np.ndarray:
        s = pd.Series(a.astype(np.float64))
        return s.rolling(w, center=True, min_periods=1).std().fillna(0.0).to_numpy(np.float32)
    
    
    def _build_gr_rolls(gr_vals: np.ndarray, ev_iloc: np.ndarray):
        """
        Compute all GR rolling features for the eval rows in one pass.
        Single Series construction; all window sizes computed from it.
        """
        s = pd.Series(gr_vals.astype(np.float64))
        out = {}
        # All rolling windows in one loop to share the Series object
        for w in (5, 21, 51, 101):
            rm = s.rolling(w, center=True, min_periods=1)
            mean_arr = rm.mean().to_numpy(np.float32)
            std_arr  = rm.std().fillna(0.0).to_numpy(np.float32)
            out[f"grm{w}"] = mean_arr[ev_iloc]
            out[f"grs{w}"] = std_arr[ev_iloc]
        # Lags / leads — computed from the base series, not rolling
        for lag in (1, 5, 15, 30):
            out[f"glag{lag}"]  = s.shift(lag ).bfill().to_numpy(np.float32)[ev_iloc]
            out[f"glead{lag}"] = s.shift(-lag).ffill().to_numpy(np.float32)[ev_iloc]
        diff1 = s.diff().fillna(0.0).to_numpy(np.float32)
        diff2 = s.diff().diff().fillna(0.0).to_numpy(np.float32)
        out["gr_d1"] = diff1[ev_iloc]
        out["gr_d2"] = diff2[ev_iloc]
        return out
    
    
    # ═══════════════════════════════════════════════════════════════════════════════
    # Multi-scale self-correlation
    # ═══════════════════════════════════════════════════════════════════════════════
    
    def multi_scale_sc(kgr: np.ndarray, ktvt: np.ndarray, hgr: np.ndarray,
                       hws=(8, 15, 25), stride: int = 3):
        fallback = float(ktvt[-1]) if len(ktvt) > 0 else 0.0
        nh = len(hgr); nk = len(kgr)
        results = []
    
        # Smooth once; share across all scales
        kg_sm = (pd.Series(kgr).rolling(5, center=True, min_periods=1)
                 .mean().to_numpy(np.float32))
        hg_sm = (pd.Series(hgr).rolling(5, center=True, min_periods=1)
                 .mean().to_numpy(np.float32))
    
        # Build eval Hankel matrix once for the largest window; slice for smaller ones
        max_hw = max(hws)
        max_win = 2 * max_hw + 1
        _hp_max = np.pad(hg_sm, max_hw, mode="edge")
        # Pre-build the full H matrix for max window (reused via slicing below)
        if nh > 0 and nk >= max_win + 1:
            _H_max = _hp_max[np.arange(nh)[:, None] + np.arange(max_win)[None, :]].astype(np.float32)
        else:
            _H_max = None
    
        for hw_sc in hws:
            win = 2 * hw_sc + 1
            if nk < win + 1 or nh == 0:
                results.append((np.full(nh, fallback, np.float32),
                                np.zeros(nh, np.float32)))
                continue
    
            sts = np.arange(0, nk - win + 1, stride, dtype=np.int32)
            if len(sts) == 0:
                results.append((np.full(nh, fallback, np.float32),
                                np.zeros(nh, np.float32)))
                continue
    
            # Known-side Hankel
            idx_mat = sts[:, None] + np.arange(win, dtype=np.int32)[None, :]
            C  = kg_sm[idx_mat].astype(np.float32)
            mu = C.mean(1, keepdims=True); sd = C.std(1, keepdims=True) + 1e-6
            Cn = (C - mu) / sd
    
            # Eval-side Hankel: reuse _H_max by slicing center columns when possible
            if _H_max is not None and hw_sc == max_hw:
                H = _H_max
            else:
                # For smaller windows, re-pad efficiently
                pad_h = hw_sc
                hp = np.pad(hg_sm, pad_h, mode="edge")
                H  = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]].astype(np.float32)
    
            mu_h = H.mean(1, keepdims=True); sd_h = H.std(1, keepdims=True) + 1e-6
            Hn   = (H - mu_h) / sd_h
    
            # Matrix NCC: (nh × ns) — use float32 matmul (faster on GPU-less CPU)
            ncc  = np.dot(Hn, Cn.T) / win          # shape (nh, ns)
            best  = ncc.argmax(1)
            score = ncc.max(1).astype(np.float32)
            ctrs  = np.clip(sts[best] + hw_sc, 0, nk - 1)
            results.append((ktvt[ctrs].astype(np.float32), score))
    
        return results
    
    
    def gr_envelope(gr: np.ndarray, w: int = 21) -> np.ndarray:
        return (pd.Series(gr).rolling(w, center=True, min_periods=1)
                .max().to_numpy(np.float32))
    
    
    def gr_energy(gr: np.ndarray, w: int = 21) -> np.ndarray:
        sq = gr.astype(np.float64) ** 2
        return np.sqrt(
            pd.Series(sq).rolling(w, center=True, min_periods=1)
            .mean().to_numpy().clip(0)
        ).astype(np.float32)
    
    
    # ═══════════════════════════════════════════════════════════════════════════════
    # Beam Search  (vectorised; ±2 delta; boolean seen-array instead of set)
    # ═══════════════════════════════════════════════════════════════════════════════
    
    _DELTAS = np.array([-2, -1, 0, 1, 2], dtype=np.int32)
    _ND     = len(_DELTAS)
    
    
    def beam_search(gr_h: np.ndarray, tw_tvt: np.ndarray, tw_gr: np.ndarray,
                    start_tvt: float, bs: int = 10, mc: float = 20.0,
                    es: float = 144.0, r: int = 2) -> np.ndarray:
        tw_tvt = np.asarray(tw_tvt, np.float32)
        tw_gr  = np.asarray(tw_gr,  np.float32)
        T  = len(tw_tvt)
        fb = float(np.nanmean(tw_gr))
    
        sg = pd.Series(gr_h, dtype="float32").interpolate(
            limit_direction="both").fillna(fb)
        if r > 0:
            sg = sg.rolling(r * 2 + 1, center=True, min_periods=1).mean()
        sg = sg.to_numpy(np.float32)
    
        si = nn_idx(tw_tvt, start_tvt)
        ns = len(sg)
    
        bps = np.empty((ns, bs), np.int32)
        bpb = np.empty((ns, bs), np.int32)
    
        bi  = np.full(bs, si, np.int32)
        bc  = np.zeros(bs, np.float64)
    
        # Pre-allocate movement penalty (constant across steps)
        mv = mc * np.abs(_DELTAS).astype(np.float64)   # shape (ND,)
    
        # Boolean seen array — O(T) reset but avoids set hashing
        _seen = np.zeros(T, dtype=np.bool_)
    
        for s, gv in enumerate(sg):
            ci = np.clip(bi[:, None] + _DELTAS[None, :], 0, T - 1)   # (bs, ND)
            em = (gv - tw_gr[ci]) ** 2 / es                            # (bs, ND)
            cc = bc[:, None] + em + mv[None, :]                        # (bs, ND)
    
            fi  = ci.ravel()       # (bs*ND,)
            fc  = cc.ravel()       # (bs*ND,)
            fp  = np.repeat(np.arange(bs, dtype=np.int32), _ND)
    
            ord_ = np.argsort(fc, kind="stable")
    
            # Reset only the positions we used last step (O(bs*ND) not O(T))
            _seen[:] = False
            kept = []
            for o in ord_:
                t = int(fi[o])
                if not _seen[t]:
                    _seen[t] = True
                    kept.append(o)
                if len(kept) == bs:
                    break
            while len(kept) < bs:
                kept.append(kept[-1])
            kept_arr = np.array(kept, np.int32)
    
            bps[s] = fp[kept_arr]
            bpb[s] = fi[kept_arr]
            bi = fi[kept_arr].astype(np.int32)
            bc = fc[kept_arr]
    
        path = np.empty(ns, np.int32)
        cb   = int(np.argmin(bc))
        for s in range(ns - 1, -1, -1):
            path[s] = bpb[s, cb]
            cb      = bps[s, cb]
    
        return tw_tvt[path]
    
    
    def run_all_beams(hw: pd.DataFrame, tw_tvt: np.ndarray, tw_gr: np.ndarray,
                      last_tvt: float, gr_filled: pd.Series,
                      eval_start_idx: int) -> dict:
        hgr    = gr_filled.iloc[eval_start_idx:].to_numpy(np.float32)
        n_eval = int((hw["TVT_input"].isna()).sum())
        result = {}
        for bs, mc, es, r, tag in BEAMS:
            pred = beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
            result[tag] = pred[:n_eval].astype(np.float32)
        return result
    
    
    # ═══════════════════════════════════════════════════════════════════════════════
    # Particle Filters
    # ═══════════════════════════════════════════════════════════════════════════════
    
    def _cal_gr_sigma(hw: pd.DataFrame, tw_tvt: np.ndarray, tw_gr: np.ndarray) -> float:
        kn = hw[hw["TVT_input"].notna() & hw["GR"].notna()]
        if len(kn) < 20:
            return PF_GR_SIG_DEF
        ex = np.interp(kn["TVT_input"].values, tw_tvt, tw_gr)
        return float(np.clip(np.std(kn["GR"].values - ex),
                             PF_GR_SIG_MIN, PF_GR_SIG_MAX))
    
    
    def _z_beta(hw: pd.DataFrame):
        kn = hw[hw["TVT_input"].notna()]
        if len(kn) < 30:
            return -1.0, 0.0, 0.1
        dz   = np.diff(kn["Z"].values)
        dtvt = np.diff(kn["TVT_input"].values)
        dmd  = np.diff(kn["MD"].values)
        m    = dmd > 0
        if m.sum() < 10:
            return -1.0, 0.0, 0.1
        vz = dz[m] / dmd[m]; vt = dtvt[m] / dmd[m]
        A  = np.column_stack([vz, np.ones_like(vz)])
        c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        return float(c[0]), float(c[1]), max(float(np.std(vt - (c[0] * vz + c[1]))), 0.001)
    
    
    def _init_v(hw: pd.DataFrame) -> float:
        kn = hw[hw["TVT_input"].notna()]
        if len(kn) < 10:
            return 0.0
        tail = kn.tail(20)
        dtvt = np.diff(tail["TVT_input"].values)
        dmd  = np.diff(tail["MD"].values)
        m    = dmd > 0
        return 0.0 if m.sum() < 3 else float(np.median(dtvt[m] / dmd[m]))
    
    
    # ── numba-compiled PF loop cores ─────────────────────────────────────────────
    
    @_njit(cache=True)
    def _numba_interp1(xp: np.ndarray, fp: np.ndarray, x: float) -> float:
        n = len(xp)
        if x <= xp[0]:  return fp[0]
        if x >= xp[-1]: return fp[-1]
        lo = 0; hi = n - 1
        while hi - lo > 1:
            mid = (lo + hi) >> 1
            if xp[mid] <= x: lo = mid
            else:             hi = mid
        t = (x - xp[lo]) / (xp[hi] - xp[lo])
        return fp[lo] + t * (fp[hi] - fp[lo])
    
    
    @_njit(cache=True)
    def _pf_z_loop(md_v, gr_v, z_v,
                   tw_tvt, tw_gr, tw_s,
                   pos, vel, w,
                   gs, beta, icpt, zsig,
                   PF_MOM, PF_VN, PF_PN, tmin, tmax,
                   PF_GR_WT, PF_RESAMP, PF_ROUGH_P, PF_ROUGH_V,
                   gr_sm_v):
        N   = len(pos)
        n_e = len(md_v)
        pts = np.empty(n_e, np.float32)
        std = np.empty(n_e, np.float32)
        pm  = md_v[0]
        for i in range(n_e):
            dm  = max(md_v[i] - pm, 1.0)
            dzd = (z_v[i] - (z_v[i - 1] if i > 0 else z_v[0])) / dm
    
            noise_v = np.random.normal(0.0, PF_VN, N)
            noise_p = np.random.normal(0.0, PF_PN, N)
            _lo = tmin - 50.0; _hi = tmax + 50.0
            for j in range(N):
                vel[j] = PF_MOM * vel[j] + noise_v[j]
                _raw   = pos[j] + vel[j] * dm + noise_p[j]
                pos[j] = min(max(_raw, _lo), _hi)
    
            gv = gr_v[i]
            if not np.isnan(gv):
                for j in range(N):
                    ep = _numba_interp1(tw_tvt, tw_gr, pos[j])
                    lp = np.exp(-0.5 * ((gv - ep) / gs) ** 2)
                    gs_sm_j = gr_sm_v[i] if not np.isnan(gr_sm_v[i]) else np.nan
                    if not np.isnan(gs_sm_j):
                        es_ = _numba_interp1(tw_tvt, tw_s, pos[j])
                        ls  = np.exp(-0.5 * ((gs_sm_j - es_) / (gs * 1.5)) ** 2)
                        lk  = (1.0 - PF_GR_WT) * lp + PF_GR_WT * ls
                    else:
                        lk = lp
                    w[j] *= max(lk, 1e-300)
                ws = np.sum(w)
                if ws > 0.0:
                    for j in range(N): w[j] /= ws
                else:
                    for j in range(N): w[j] = 1.0 / N
    
            zs = max(zsig * 2.0, 0.005)
            for j in range(N):
                ve = beta * dzd + icpt
                lz = max(np.exp(-0.5 * ((vel[j] - ve) / zs) ** 2), 1e-300)
                w[j] *= lz
            ws = np.sum(w)
            if ws > 0.0:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1.0 / N
    
            ne = 1.0 / np.sum(w * w)
            if ne < PF_RESAMP * N:
                cum = np.cumsum(w)
                u_  = (np.arange(N) + np.random.uniform()) / N
                ix  = np.searchsorted(cum, u_)
                new_pos = pos[ix].copy(); new_vel = vel[ix].copy()
                noise_rp = np.random.normal(0.0, PF_ROUGH_P, N)
                noise_rv = np.random.normal(0.0, PF_ROUGH_V, N)
                for j in range(N):
                    pos[j] = new_pos[j] + noise_rp[j]
                    vel[j] = new_vel[j] + noise_rv[j]
                    w[j]   = 1.0 / N
    
            mu_ = 0.0
            for j in range(N): mu_ += w[j] * pos[j]
            var_ = 0.0
            for j in range(N): var_ += w[j] * (pos[j] - mu_) ** 2
            pts[i] = np.float32(mu_)
            std[i] = np.float32(np.sqrt(var_))
            pm = md_v[i]
    
        return pts, std
    
    
    @_njit(cache=True)
    def _pf_ancc_loop(md_v, gr_v, z_v,
                      tw_tvt, tw_gr,
                      pos, rate, w, gs,
                      ANCC_ALPHA, ANCC_RN, ANCC_PN,
                      tmin, tmax, PF_RESAMP,
                      ANCC_RP, ANCC_RR):
        N   = len(pos)
        n_e = len(md_v)
        pts = np.empty(n_e, np.float32)
        std = np.empty(n_e, np.float32)
        pm  = md_v[0]
        for i in range(n_e):
            dm = max(md_v[i] - pm, 1.0)
            noise_r = np.random.normal(0.0, ANCC_RN, N)
            noise_p = np.random.normal(0.0, ANCC_PN, N)
            for j in range(N):
                rate[j] = ANCC_ALPHA * rate[j] + noise_r[j]
                pos[j] += rate[j] * dm + noise_p[j]
            _lo = tmin - 50.0; _hi = tmax + 50.0
            tvt_e = np.empty(N, np.float64)
            _zvi  = z_v[i]
            for j in range(N):
                _raw    = pos[j] - _zvi
                tvt_e[j] = min(max(_raw, _lo), _hi)
            for j in range(N): pos[j] = tvt_e[j] + _zvi
    
            gv = gr_v[i]
            if not np.isnan(gv):
                for j in range(N):
                    eg  = _numba_interp1(tw_tvt, tw_gr, tvt_e[j])
                    lk  = max(np.exp(-0.5 * ((gv - eg) / gs) ** 2), 1e-300)
                    w[j] *= lk
                ws = np.sum(w)
                if ws > 0.0:
                    for j in range(N): w[j] /= ws
                else:
                    for j in range(N): w[j] = 1.0 / N
    
            ne = 1.0 / np.sum(w * w)
            if ne < PF_RESAMP * N:
                cum = np.cumsum(w)
                u_  = (np.arange(N) + np.random.uniform()) / N
                ix  = np.searchsorted(cum, u_)
                new_pos  = pos[ix].copy(); new_rate = rate[ix].copy()
                noise_rp = np.random.normal(0.0, ANCC_RP, N)
                noise_rr = np.random.normal(0.0, ANCC_RR, N)
                for j in range(N):
                    pos[j]  = new_pos[j]  + noise_rp[j]
                    rate[j] = new_rate[j] + noise_rr[j]
                    w[j]    = 1.0 / N
    
            tvt_e2 = pos - z_v[i]
            mu_ = 0.0
            for j in range(N): mu_ += w[j] * tvt_e2[j]
            var_ = 0.0
            for j in range(N): var_ += w[j] * (tvt_e2[j] - mu_) ** 2
            pts[i] = np.float32(mu_)
            std[i] = np.float32(np.sqrt(var_))
            pm = md_v[i]
    
        return pts, std
    
    
    def run_pf_z(hw: pd.DataFrame, tw_tvt: np.ndarray, tw_gr: np.ndarray,
                 N: int = PF_N):
        tw_s = (pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1)
                .mean().to_numpy(np.float32))
        tmin, tmax = float(tw_tvt.min()), float(tw_tvt.max())
        gs = _cal_gr_sigma(hw, tw_tvt, tw_gr)
        beta, icpt, zsig = _z_beta(hw)
        kn = hw[hw["TVT_input"].notna()]
        ev = hw[hw["TVT_input"].isna()]
        if len(ev) == 0:
            return np.array([], np.float32), np.array([], np.float32)
    
        gr_sm_full = (hw["GR"].rolling(PF_GR_WIN, center=True, min_periods=1)
                      .mean().to_numpy(np.float32))
        ev_locs   = [hw.index.get_loc(idx) for idx in ev.index]
        gr_sm_ev  = gr_sm_full[ev_locs].astype(np.float32)
    
        pos  = float(kn["TVT_input"].iloc[-1]) + np.random.normal(0, PF_INIT_SPR, N).astype(np.float32)
        vel  = (_init_v(hw) + np.random.normal(0, PF_INIT_V_STD, N)).astype(np.float32)
        w    = np.full(N, 1.0 / N, np.float64)
    
        md_v  = ev["MD"].to_numpy(np.float32)
        gr_v  = ev["GR"].to_numpy(np.float32)
        z_v   = ev["Z"].to_numpy(np.float32)
    
        if _NUMBA:
            pts, std = _pf_z_loop(
                md_v, gr_v, z_v,
                tw_tvt.astype(np.float64), tw_gr.astype(np.float64),
                tw_s.astype(np.float64), pos.astype(np.float64), vel.astype(np.float64),
                w,
                float(gs), float(beta), float(icpt), float(zsig),
                float(PF_MOM), float(PF_VN), float(PF_PN),
                float(tmin), float(tmax),
                float(PF_GR_WT), float(PF_RESAMP), float(PF_ROUGH_P), float(PF_ROUGH_V),
                gr_sm_ev.astype(np.float64),
            )
        else:
            tf_p  = interp1d(tw_tvt, tw_gr, bounds_error=False,
                             fill_value=(tw_gr[0], tw_gr[-1]))
            tf_s  = interp1d(tw_tvt, tw_s,  bounds_error=False,
                             fill_value=(tw_s[0],  tw_s[-1]))
            pts_l = np.empty(len(ev)); std_l = np.empty(len(ev))
            pm_py = float(kn["MD"].iloc[-1]); pz = float(kn["Z"].iloc[-1])
            for i, idx in enumerate(ev.index):
                dm  = max(md_v[i] - pm_py, 1.0)
                dzd = (z_v[i] - pz) / dm
                vel = PF_MOM * vel + np.random.normal(0, PF_VN, N)
                pos = pos + vel * dm + np.random.normal(0, PF_PN, N)
                pos = np.clip(pos, tmin - 50, tmax + 50)
                if not np.isnan(gr_v[i]):
                    ep = tf_p(pos)
                    lp = np.exp(-0.5 * ((gr_v[i] - ep) / gs) ** 2)
                    gs_sm = gr_sm_ev[i]
                    if not np.isnan(gs_sm):
                        es_ = tf_s(pos)
                        ls  = np.exp(-0.5 * ((gs_sm - es_) / (gs * 1.5)) ** 2)
                        lk  = (1 - PF_GR_WT) * lp + PF_GR_WT * ls
                    else:
                        lk = lp
                    lk = np.maximum(lk, 1e-300); w *= lk
                    ws = w.sum(); w = (w / ws) if ws > 0 else np.full(N, 1.0 / N)
                ve = beta * dzd + icpt; zs = max(zsig * 2.0, 0.005)
                lz = np.exp(-0.5 * ((vel - ve) / zs) ** 2)
                lz = np.maximum(lz, 1e-300); w *= lz
                ws = w.sum(); w = (w / ws) if ws > 0 else np.full(N, 1.0 / N)
                ne = 1.0 / np.sum(w ** 2)
                if ne < PF_RESAMP * N:
                    cum = np.cumsum(w); u = (np.arange(N) + np.random.uniform()) / N
                    ix  = np.searchsorted(cum, u)
                    pos = pos[ix]; vel = vel[ix]; w[:] = 1.0 / N
                    pos += np.random.normal(0, PF_ROUGH_P, N)
                    vel += np.random.normal(0, PF_ROUGH_V, N)
                pts_l[i] = np.average(pos, weights=w)
                std_l[i] = np.sqrt(np.average((pos - pts_l[i]) ** 2, weights=w))
                pm_py = md_v[i]; pz = z_v[i]
            pts, std = pts_l, std_l
    
        return pts.astype(np.float32), std.astype(np.float32)
    
    
    def run_pf_ancc(hw: pd.DataFrame, tw_tvt: np.ndarray, tw_gr: np.ndarray,
                    N: int = ANCC_N):
        tmin, tmax = float(tw_tvt.min()), float(tw_tvt.max())
        gs  = _cal_gr_sigma(hw, tw_tvt, tw_gr)
        kn  = hw[hw["TVT_input"].notna()]
        ev  = hw[hw["TVT_input"].isna()]
        if len(ev) == 0:
            return np.array([], np.float32), np.array([], np.float32)
    
        ls  = float(kn["TVT_input"].iloc[-1] + kn["Z"].iloc[-1])
        tail = kn.tail(30)
        dt   = np.diff(tail["TVT_input"].values)
        dz   = np.diff(tail["Z"].values)
        dm   = np.diff(tail["MD"].values)
        m    = dm > 0
        ir   = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0
    
        pos  = (ls + np.random.normal(0, ANCC_IS, N)).astype(np.float64)
        rate = (ir  + np.random.normal(0, ANCC_IR, N)).astype(np.float64)
        w    = np.full(N, 1.0 / N, np.float64)
    
        md_v = ev["MD"].to_numpy(np.float32)
        z_v  = ev["Z"].to_numpy(np.float32)
        gr_v = ev["GR"].to_numpy(np.float32)
    
        if _NUMBA:
            pts, std = _pf_ancc_loop(
                md_v.astype(np.float64), gr_v.astype(np.float64),
                z_v.astype(np.float64),
                tw_tvt.astype(np.float64), tw_gr.astype(np.float64),
                pos, rate, w, float(gs),
                float(ANCC_ALPHA), float(ANCC_RN), float(ANCC_PN),
                float(tmin), float(tmax), float(PF_RESAMP),
                float(ANCC_RP), float(ANCC_RR),
            )
        else:
            pm    = float(kn["MD"].iloc[-1])
            pts_l = np.empty(len(ev)); std_l = np.empty(len(ev))
            for i in range(len(ev)):
                dm_s = max(md_v[i] - pm, 1.0)
                rate = ANCC_ALPHA * rate + np.random.normal(0, ANCC_RN, N)
                pos  = pos + rate * dm_s + np.random.normal(0, ANCC_PN, N)
                tvt_e = np.clip(pos - z_v[i], tmin - 50, tmax + 50)
                pos   = tvt_e + z_v[i]
                if not np.isnan(gr_v[i]):
                    eg  = np.interp(tvt_e, tw_tvt, tw_gr)
                    lk  = np.exp(-0.5 * ((gr_v[i] - eg) / gs) ** 2)
                    lk  = np.maximum(lk, 1e-300); w *= lk
                    ws  = w.sum(); w = (w / ws) if ws > 0 else np.full(N, 1.0 / N)
                ne = 1.0 / np.sum(w ** 2)
                if ne < PF_RESAMP * N:
                    cum = np.cumsum(w); u = (np.arange(N) + np.random.uniform()) / N
                    ix  = np.searchsorted(cum, u)
                    pos = pos[ix]; rate = rate[ix]; w[:] = 1.0 / N
                    pos  += np.random.normal(0, ANCC_RP, N)
                    rate += np.random.normal(0, ANCC_RR, N)
                tv = float(np.average(pos - z_v[i], weights=w))
                pts_l[i] = tv
                std_l[i] = np.sqrt(np.average((pos - z_v[i] - tv) ** 2, weights=w))
                pm = md_v[i]
            pts, std = pts_l, std_l
    
        return pts.astype(np.float32), std.astype(np.float32)
    
    
    # ═══════════════════════════════════════════════════════════════════════════════
    # Spatial Imputers
    # ═══════════════════════════════════════════════════════════════════════════════
    
    class FormationPlaneKNN:
        def __init__(self, well_ids, data_dir: Path):
            rows = []
            for wid in well_ids:
                p = data_dir / f"{wid}__horizontal_well.csv"
                try:
                    df = pd.read_csv(p, usecols=["X", "Y"] + FORMATIONS).dropna()
                except Exception:
                    continue
                if len(df) == 0:
                    continue
                row = {"wid": wid,
                       "x": float(df["X"].median()),
                       "y": float(df["Y"].median())}
                for c in FORMATIONS:
                    row[f"{c}_m"] = float(df[c].median())
                rows.append(row)
    
            self.df   = pd.DataFrame(rows)
            self.wmap = {w: i for i, w in enumerate(self.df["wid"])}
            xy        = self.df[["x", "y"]].to_numpy(np.float64)
            self.scale = np.where(xy.std(0) < 1e-3, 1.0, xy.std(0))
            self.tree  = cKDTree(xy / self.scale)
            self.xa    = self.df["x"].to_numpy(np.float64)
            self.ya    = self.df["y"].to_numpy(np.float64)
            self.fa    = self.df[[f"{c}_m" for c in FORMATIONS]].to_numpy(np.float64)
            self._fa_mean = self.fa.mean(0)
    
        def impute(self, xy_q: np.ndarray, self_wid=None, k: int = PLANE_K):
            q    = xy_q / self.scale
            nf   = min(k + 5, len(self.df))
            dist, idx = self.tree.query(q, k=nf, workers=-1)
    
            if self_wid in self.wmap:
                dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
    
            if nf > k:
                ord_ = np.argpartition(dist, k - 1, axis=1)[:, :k]
                dk   = np.take_along_axis(dist, ord_, axis=1)
                ik   = np.take_along_axis(idx,  ord_, axis=1)
            else:
                dk, ik = dist, idx
    
            vk  = np.isfinite(dk)
            w   = np.where(vk, 1.0 / (dk + 1e-3), 0.0)
    
            xn = self.xa[ik]; yn = self.ya[ik]
            wx = w * xn;      wy = w * yn
    
            A = np.zeros((len(q), 3, 3), np.float64)
            A[:, 0, 0] = (wx * xn).sum(1);  A[:, 0, 1] = (wx * yn).sum(1)
            A[:, 0, 2] = wx.sum(1)
            A[:, 1, 0] = A[:, 0, 1];        A[:, 1, 1] = (wy * yn).sum(1)
            A[:, 1, 2] = wy.sum(1)
            A[:, 2, 0] = A[:, 0, 2];        A[:, 2, 1] = A[:, 1, 2]
            A[:, 2, 2] = w.sum(1)
            A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
    
            fn  = self.fa[ik]
            rhs = np.stack([
                (wx[:, :, None] * fn).sum(1),
                (wy[:, :, None] * fn).sum(1),
                (w[:, :, None]  * fn).sum(1),
            ], axis=1)
    
            try:
                coef = np.linalg.solve(A, rhs)
            except np.linalg.LinAlgError:
                coef = np.zeros((len(q), 3, 6))
                for r in range(len(q)):
                    try:
                        coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                    except Exception:
                        pass
    
            Xq   = xy_q[:, 0]; Yq = xy_q[:, 1]
            pred = (Xq[:, None] * coef[:, 0, :]
                    + Yq[:, None] * coef[:, 1, :]
                    + coef[:, 2, :]).astype(np.float32)
    
            no_nbr = ~vk.any(1)
            pred[no_nbr] = self._fa_mean.astype(np.float32)
    
            min_dist = np.where(vk, dk, np.inf).min(1).astype(np.float32)
            return pred, min_dist
    
    
    class DenseANCCImputer:
        def __init__(self, well_ids, data_dir: Path, spw: int = DENSE_SPW):
            xs, ys, anccs, wids = [], [], [], []
            for wid in well_ids:
                p = data_dir / f"{wid}__horizontal_well.csv"
                try:
                    df = pd.read_csv(p, usecols=["X", "Y", "ANCC"]).dropna()
                except Exception:
                    continue
                if len(df) == 0:
                    continue
                ix = np.linspace(0, len(df) - 1, min(spw, len(df)), dtype=int)
                s  = df.iloc[ix]
                xs.append(s["X"].values); ys.append(s["Y"].values)
                anccs.append(s["ANCC"].values)
                wids.extend([wid] * len(s))
    
            self.xy   = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
            self.ancc = np.concatenate(anccs).astype(np.float32)
            self.wids = np.array(wids)
            self.scale = np.where(self.xy.std(0) < 1e-3, 1.0, self.xy.std(0))
            self.tree  = cKDTree(self.xy / self.scale)
            self._mean = float(self.ancc.mean())
    
        def impute(self, xy_q: np.ndarray, self_wid=None,
                   k: int = DENSE_K, nfetch: int = 500):
            xy_q = np.atleast_2d(xy_q)
            q    = xy_q / self.scale
            nf   = min(nfetch, len(self.ancc))
    
            dist, idx = self.tree.query(q, k=nf, workers=-1)
            if self_wid is not None:
                dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
    
            if nf > k:
                ord_ = np.argpartition(dist, k - 1, axis=1)[:, :k]
                dk   = np.take_along_axis(dist, ord_, axis=1)
                ik   = np.take_along_axis(idx,  ord_, axis=1)
            else:
                dk, ik = dist, idx
    
            vk  = np.isfinite(dk)
            w   = np.where(vk, 1.0 / (dk + 1e-3), 0.0)
            sw  = w.sum(1); safe = np.where(sw < 1e-9, 1.0, sw)
    
            an  = self.ancc[ik]
            ap  = (an * w).sum(1) / safe
            ap  = np.where(sw < 1e-9, self._mean, ap)
            var = ((an - ap[:, None]) ** 2 * w).sum(1) / safe
    
            return (ap.astype(np.float32),
                    np.sqrt(np.maximum(var, 0.0)).astype(np.float32),
                    np.where(vk, dk, np.inf).min(1).astype(np.float32))
    
    
    # ═══════════════════════════════════════════════════════════════════════════════
    # Feature Builder — split into I/O wrapper + pure compute core
    # ═══════════════════════════════════════════════════════════════════════════════
    
    ANCH_OFFS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80],  dtype=np.float32)
    BEAM_OFFS = np.array([-40, -20, -10,  -5, -3, 0, 3,  5, 10, 20, 40],  dtype=np.float32)
    SC_OFFS   = np.array([-30, -15,  -8,  -4, -2, 0, 2,  4,  8, 15, 30],  dtype=np.float32)
    
    # Process-global imputers — set once per worker via pool initializer
    _FI: Optional[FormationPlaneKNN] = None
    _DI: Optional[DenseANCCImputer]  = None
    
    
    def _worker_init(fi: FormationPlaneKNN, di: DenseANCCImputer) -> None:
        """Called once per worker process; stores imputers in process-local globals."""
        global _FI, _DI
        _FI = fi
        _DI = di
    
    
    USE_PROCESS_POOL = env_flag("ROGII_USE_PROCESS_POOL", os.name != "nt")
    USE_THREAD_POOL  = env_flag("ROGII_USE_THREAD_POOL", os.name == "nt")
    
    
    def _map_with_imputers(func, items, processes):
        """Use process pools on Kaggle/Linux and thread pools on Windows local runs."""
        if USE_PROCESS_POOL and processes > 1:
            with multiprocessing.Pool(
                processes=processes,
                initializer=_worker_init,
                initargs=(FI, DI),
            ) as pool:
                return pool.map(func, items)
        _worker_init(FI, DI)
        if USE_THREAD_POOL and processes > 1:
            with ThreadPoolExecutor(max_workers=processes) as pool:
                return list(pool.map(func, items))
        return [func(item) for item in items]
    
    
    def _starmap_with_imputers(func, args, processes):
        """Use process pools on Kaggle/Linux and thread pools on Windows local runs."""
        if USE_PROCESS_POOL and processes > 1:
            with multiprocessing.Pool(
                processes=processes,
                initializer=_worker_init,
                initargs=(FI, DI),
            ) as pool:
                return pool.starmap(func, args)
        _worker_init(FI, DI)
        if USE_THREAD_POOL and processes > 1:
            with ThreadPoolExecutor(max_workers=processes) as pool:
                return list(pool.map(lambda arg: func(*arg), args))
        return [func(*arg) for arg in args]
    
    
    # ── Pre-computed GR stats cache (per well, shared across aug splits) ──────────
    
    class _WellGRCache:
        """
        Holds expensive per-well arrays that don't change across augmentation splits
        (the full GR array, rolling stats, envelope, energy).
        Constructed once in process_train_well and passed into build_augmented.
        """
        __slots__ = ("gr_arr", "roll_feats_full", "gr_env_full", "gr_nrg_full")
    
        def __init__(self, hw: pd.DataFrame, gr_mean: float):
            gr_full = (hw["GR"].astype(float)
                       .interpolate(limit_direction="both")
                       .fillna(gr_mean))
            self.gr_arr         = gr_full.to_numpy(np.float32)
            # Compute rolling stats over the FULL well once
            all_idx             = np.arange(len(hw), dtype=np.int64)
            self.roll_feats_full = _build_gr_rolls(self.gr_arr, all_idx)
            self.gr_env_full     = gr_envelope(self.gr_arr)
            self.gr_nrg_full     = gr_energy(self.gr_arr)
    
    
    def _build_well_from_df(
        hw: pd.DataFrame,
        tw: pd.DataFrame,
        is_train: bool,
        wid: str,
        gr_cache: Optional[_WellGRCache] = None,
    ) -> Optional[pd.DataFrame]:
        """
        Pure-compute feature builder. No disk I/O.
        gr_cache: if provided, reuses pre-computed GR rolling stats (avoids recompute
                  across augmentation splits of the same well).
        """
        if _FI is None or _DI is None:
            _dbg(f"_build_well_from_df({wid}): imputers not set!", level="ERROR")
            return None
    
        required_hw = {"MD", "GR", "X", "Y", "Z", "TVT_input"}
        required_tw = {"TVT", "GR"}
        missing_hw  = required_hw - set(hw.columns)
        missing_tw  = required_tw - set(tw.columns)
        if missing_hw:
            _dbg(f"_build_well_from_df({wid}): missing hw cols {missing_hw}", level="WARN")
            return None
        if missing_tw:
            _dbg(f"_build_well_from_df({wid}): missing tw cols {missing_tw}", level="WARN")
            return None
    
        if is_train and "TVT" not in hw.columns:
            _dbg(f"_build_well_from_df({wid}): is_train but no TVT column", level="WARN")
            return None
    
        kn = hw[hw["TVT_input"].notna()]
        ev = hw[hw["TVT_input"].isna()]
        if len(ev) == 0:
            _dbg(f"_build_well_from_df({wid}): no eval rows", level="WARN")
            return None
        if len(kn) < 10:
            _dbg(f"_build_well_from_df({wid}): too few known rows ({len(kn)})", level="WARN")
            return None
        if is_train and hw["TVT"].isna().all():
            _dbg(f"_build_well_from_df({wid}): is_train but all TVT NaN", level="WARN")
            return None
    
        tw_tvt = tw["TVT"].to_numpy(np.float32)
        tw_gr  = tw["GR"].to_numpy(np.float32)
        if len(tw_tvt) < 3:
            _dbg(f"_build_well_from_df({wid}): typewell too short", level="WARN")
            return None
    
        try:
            lk       = kn.iloc[-1]
            last_tvt = float(lk["TVT_input"])
            gr_mean  = float(np.nanmean(tw_gr))
    
            # ── GR arrays — use cache if available, else compute ──────────────
            if gr_cache is not None:
                gr_arr = gr_cache.gr_arr
            else:
                gr_full = (hw["GR"].astype(float)
                           .interpolate(limit_direction="both")
                           .fillna(gr_mean))
                gr_arr = gr_full.to_numpy(np.float32)
    
            # iloc positions of eval rows (needed for rolling feature slicing)
            ev_idx_arr = np.array([hw.index.get_loc(i) for i in ev.index], dtype=np.int64)
    
            hgr = gr_arr[ev_idx_arr]
            kgr = gr_arr[:len(kn)]
    
            # ── GR rolling features ───────────────────────────────────────────
            if gr_cache is not None:
                # Slice pre-computed full-well rolling arrays to eval positions
                roll_feats = {
                    k: v[ev_idx_arr]
                    for k, v in gr_cache.roll_feats_full.items()
                }
                hgr_env = gr_cache.gr_env_full[ev_idx_arr]
                hgr_nrg = gr_cache.gr_nrg_full[ev_idx_arr]
            else:
                roll_feats = _build_gr_rolls(gr_arr, ev_idx_arr)
                hgr_env    = gr_envelope(gr_arr)[ev_idx_arr]
                hgr_nrg    = gr_energy(gr_arr)[ev_idx_arr]
    
            gr_d1 = roll_feats.pop("gr_d1")
            gr_d2 = roll_feats.pop("gr_d2")
    
            # Particle filters
            pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr)
            pf_z, std_z = run_pf_z(hw, tw_tvt, tw_gr)
            if len(pf_a) == 0:
                _dbg(f"_build_well_from_df({wid}): pf_ancc returned empty", level="WARN")
                return None
    
            pf_use  = pf_a; std_use = std_a
            has_z   = (len(pf_z) == len(pf_a)) and not np.any(np.isnan(pf_z))
    
            # Beam search (7 configs)
            # eval_start_idx: first iloc position of the eval block
            eval_start_iloc = int(ev_idx_arr[0])
            gr_filled_series = pd.Series(gr_arr)
            bpaths    = run_all_beams(hw, tw_tvt, tw_gr, last_tvt,
                                      gr_filled_series, eval_start_iloc)
            beam_vals = np.stack(list(bpaths.values()), axis=1)
            beam_ref  = (bpaths["cons"] + bpaths["sm5"]) * 0.5
    
            # Multi-scale self-correlation
            ktvt = kn["TVT_input"].to_numpy(np.float32)
            sc_results  = multi_scale_sc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
            sc8,  sc8s  = sc_results[0]
            sc15, sc15s = sc_results[1]
            sc25, sc25s = sc_results[2]
            sc_consensus = ((sc8 + sc15 + sc25) * (1.0 / 3.0)).astype(np.float32)
            sc_trust = float(np.clip(len(kn) / 200.0, 0.0, 0.6))
            hyb_ref  = ((1 - sc_trust) * beam_ref + sc_trust * sc15).astype(np.float32)
    
            # Affine calibration
            tw_at_k = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32)
            a_cal, b_cal = affine_cal(kgr, tw_at_k)
    
            # Prefix statistics
            kmd      = kn["MD"].to_numpy(np.float32)
            kz       = kn["Z"].to_numpy(np.float32)
            pfx_rmse = float(np.sqrt(np.mean((kgr - tw_at_k) ** 2)))
            slp_all  = robust_slope(kmd,       ktvt)
            slp_50   = robust_slope(kmd[-50:], ktvt[-50:])
            slp_z    = robust_slope(kz,        ktvt)
    
            # Spatial imputers
            swid    = wid if is_train else None
            xy_ev   = ev[["X", "Y"]].to_numpy(np.float64)
            xy_kn   = kn[["X", "Y"]].to_numpy(np.float64)
            form_ev, knn_d  = _FI.impute(xy_ev, self_wid=swid)
            form_kn, _      = _FI.impute(xy_kn, self_wid=swid)
            z_kn    = kn["Z"].to_numpy(np.float32)
            z_ev    = ev["Z"].to_numpy(np.float32)
    
            # Per-formation TVT formulas + RMSE features
            ktvt_plus_zkn = ktvt + z_kn
            residuals_all = ktvt_plus_zkn[:, None] - form_kn
    
            b_all_v  = np.median(residuals_all, axis=0)
            b_wls_v  = np.array([wls_b_well(ktvt, z_kn, form_kn[:, fi])
                                 for fi in range(6)], dtype=np.float32)
            b_50_v   = (np.median(residuals_all[-50:], axis=0).astype(np.float32)
                        if len(ktvt) >= 5 else b_all_v.astype(np.float32))
    
            tvt_form_mat  = (-z_ev[:, None] + form_ev + b_all_v[None, :]).astype(np.float32)
            tvt_formw_mat = (-z_ev[:, None] + form_ev + b_wls_v[None, :]).astype(np.float32)
    
            kn_pred_mat   = (-z_kn[:, None] + form_kn + b_all_v[None, :]).astype(np.float32)
            form_rmse_v   = np.sqrt(np.mean((ktvt[:, None] - kn_pred_mat) ** 2, axis=0))
    
            form_consistency = tvt_form_mat
            form_mean_d  = (form_consistency.mean(1) - last_tvt).astype(np.float32)
            form_std_d   = form_consistency.std(1).astype(np.float32)
            form_range_d = (form_consistency.max(1) - form_consistency.min(1)).astype(np.float32)
    
            # Dense ANCC
            d_ancc, d_std, d_dist         = _DI.impute(xy_ev, self_wid=swid)
            d_kn,   d_std_kn, _           = _DI.impute(xy_kn, self_wid=swid)
            res_kn   = ktvt + z_kn - d_kn
            b_d      = float(np.median(res_kn))
            b_dw     = wls_b_well(ktvt, z_kn, d_kn)
            b_d50    = float(np.median(res_kn[-50:])) if len(ktvt) >= 5 else b_d
            tvt_dense = (-z_ev + d_ancc + b_d).astype(np.float32)
            tvt_dw    = (-z_ev + d_ancc + b_dw).astype(np.float32)
            tvt_d50   = (-z_ev + d_ancc + b_d50).astype(np.float32)
            d_rmse    = float(np.sqrt(np.mean(res_kn ** 2)))
            d_bias    = float(np.mean(res_kn))
            d_nb_std  = float(np.mean(d_std_kn))
    
            # Inter-signal consensus std
            all_sig = [pf_use, tvt_form_mat[:, 0], tvt_dense,
                       *bpaths.values(), sc8, sc15, sc25]
            valid_sig = [s for s in all_sig
                         if len(s) == len(ev) and not np.any(np.isnan(s))]
            signal_std = (np.stack(valid_sig, axis=1).std(1).astype(np.float32)
                          if len(valid_sig) >= 2 else np.zeros(len(ev), np.float32))
    
            # Slope baselines
            hmd        = ev["MD"].to_numpy(np.float32)
            md_since   = hmd - float(lk["MD"])
            slp_base_all = (last_tvt + slp_all * md_since).astype(np.float32)
            slp_base_50  = (last_tvt + slp_50  * md_since).astype(np.float32)
    
            nh   = len(ev)
            frac = (np.arange(nh) / max(nh - 1, 1)).astype(np.float32)
    
            def sc(v: float) -> np.ndarray:
                return np.full(nh, np.float32(v), np.float32)
    
            # Trajectory derivatives
            mdd   = hw["MD"].diff().replace(0, np.nan)
            dzdmd = (hw["Z"].diff() / mdd).iloc[ev.index].to_numpy(np.float32)
            dxdmd = (hw["X"].diff() / mdd).iloc[ev.index].to_numpy(np.float32)
            dydmd = (hw["Y"].diff() / mdd).iloc[ev.index].to_numpy(np.float32)
    
            # Pre-compute interp lookups (all at once, single np.interp calls)
            anch_twgr       = np.interp(float(last_tvt) + ANCH_OFFS, tw_tvt, tw_gr).astype(np.float32)
            beam_ref_tw     = np.interp(beam_ref,  tw_tvt, tw_gr).astype(np.float32)
            sc15_tw         = np.interp(sc15,      tw_tvt, tw_gr).astype(np.float32)
    
            # Offset lookups — vectorised: shape (len(BEAM_OFFS), nh) → transpose
            beam_offsets_tw = np.array([
                np.interp(beam_ref + o, tw_tvt, tw_gr) for o in BEAM_OFFS
            ], dtype=np.float32).T
            sc_offsets_tw = np.array([
                np.interp(sc15 + o, tw_tvt, tw_gr) for o in SC_OFFS
            ], dtype=np.float32).T
    
            # ── Assemble feature dict ────────────────────────────────────────
            feats: dict = {
                "well": wid,
                "id":   [f"{wid}_{i}" for i in ev.index],
                "last_known_tvt": sc(last_tvt),
                "pf_ancc":       pf_use,
                "pf_ancc_std":   std_use,
                "pf_ancc_delta": (pf_use - last_tvt).astype(np.float32),
                "pf_z":          pf_z.astype(np.float32) if has_z else sc(last_tvt),
                "pf_z_delta":    (pf_z - last_tvt).astype(np.float32) if has_z else sc(0.0),
                "pf_vs_z":       (pf_use - pf_z.astype(np.float32)) if has_z else sc(0.0),
                "pf_std_trend":  (std_use - std_use[0]).astype(np.float32) if len(std_use) > 0 else sc(0.0),
                **{f"beam_{t}_d": (p - np.float32(last_tvt)).astype(np.float32)
                   for t, p in bpaths.items()},
                "beam_mean_d": (beam_vals - last_tvt).mean(1).astype(np.float32),
                "beam_std_d":  (beam_vals - last_tvt).std(1).astype(np.float32),
                "beam_med_d":  np.median(beam_vals - last_tvt, axis=1).astype(np.float32),
                "sc8_d":    (sc8  - np.float32(last_tvt)),
                "sc8_score": sc8s,
                "sc15_d":   (sc15 - np.float32(last_tvt)),
                "sc15_score": sc15s,
                "sc25_d":   (sc25 - np.float32(last_tvt)),
                "sc25_score": sc25s,
                "sc_cons_d": (sc_consensus - np.float32(last_tvt)),
                "sc_trust":  sc(sc_trust),
                "hyb_d":     (hyb_ref - np.float32(last_tvt)).astype(np.float32),
                **{fn:                    tvt_form_mat[:, fi]  for fi, fn in enumerate(FORMATIONS)},
                **{fn + "_wls":           tvt_formw_mat[:, fi] for fi, fn in enumerate(FORMATIONS)},
                **{f"b_{fn}":             sc(float(b_all_v[fi]))  for fi, fn in enumerate(FORMATIONS)},
                **{f"bw_{fn}":            sc(float(b_wls_v[fi]))  for fi, fn in enumerate(FORMATIONS)},
                **{f"b50_{fn}":           sc(float(b_50_v[fi]))   for fi, fn in enumerate(FORMATIONS)},
                **{f"tvtF_{fn}_d":       (tvt_form_mat[:,  fi] - last_tvt).astype(np.float32)
                   for fi, fn in enumerate(FORMATIONS)},
                **{f"tvtFw_{fn}_d":      (tvt_formw_mat[:, fi] - last_tvt).astype(np.float32)
                   for fi, fn in enumerate(FORMATIONS)},
                **{f"form_rmse_{fn}":    sc(float(form_rmse_v[fi]))
                   for fi, fn in enumerate(FORMATIONS)},
                "form_mean_d":   form_mean_d,
                "form_std_d":    form_std_d,
                "form_range_d":  form_range_d,
                "spatial_knn_dist": knn_d,
                "dense_ancc":    d_ancc,
                "dense_std":     d_std,
                "dense_dist":    d_dist,
                "tvt_dense_d":   (tvt_dense - last_tvt).astype(np.float32),
                "tvt_dw_d":      (tvt_dw    - last_tvt).astype(np.float32),
                "tvt_d50_d":     (tvt_d50   - last_tvt).astype(np.float32),
                "dense_rmse":    sc(d_rmse),
                "dense_bias":    sc(d_bias),
                "dense_nb_std":  sc(d_nb_std),
                "pf_vs_spatial":      (pf_use - tvt_form_mat[:, 0]).astype(np.float32),
                "pf_vs_dense":        (pf_use - tvt_dense).astype(np.float32),
                "spatial_vs_dense":   (tvt_form_mat[:, 0] - tvt_dense).astype(np.float32),
                "beam_vs_spatial":    (bpaths["cons"] - tvt_form_mat[:, 0]).astype(np.float32),
                "sc15_vs_beam_cons":  (sc15 - bpaths["cons"]).astype(np.float32),
                "signal_std":  signal_std,
                "cal_a": sc(a_cal), "cal_b": sc(b_cal),
                "pfx_rmse":   sc(pfx_rmse), "known_len": sc(len(kn)), "eval_len": sc(nh),
                "slp_all":    sc(slp_all),  "slp_50":    sc(slp_50),  "slp_z":    sc(slp_z),
                "slp_base_d_all": (slp_base_all - last_tvt).astype(np.float32),
                "slp_base_d_50":  (slp_base_50  - last_tvt).astype(np.float32),
                "ktvt_range": sc(float(np.ptp(ktvt))), "ktvt_std": sc(float(ktvt.std())),
                "md_since": md_since, "frac": frac, "frac2": frac ** 2,
                "sqrt_frac": np.sqrt(frac),
                "z":  z_ev,
                "dx": (ev["X"] - float(lk["X"])).to_numpy(np.float32),
                "dy": (ev["Y"] - float(lk["Y"])).to_numpy(np.float32),
                "dz": (z_ev    - float(lk["Z"])).astype(np.float32),
                "dxy": np.sqrt(
                    (ev["X"] - float(lk["X"])) ** 2 +
                    (ev["Y"] - float(lk["Y"])) ** 2
                ).to_numpy(np.float32),
                "dzdmd": dzdmd, "dxdmd": dxdmd, "dydmd": dydmd,
                "gr":       hgr,
                "gr_d1":    gr_d1,
                "gr_d2":    gr_d2,
                "gr_env":   hgr_env.astype(np.float32),
                "gr_energy": hgr_nrg.astype(np.float32),
                "gr_vs_tw_anc":  hgr - float(np.interp(last_tvt, tw_tvt, tw_gr)),
                "gr_vs_slp_all": hgr - np.interp(slp_base_all, tw_tvt, tw_gr).astype(np.float32),
                **{f"tda{int(o)}": hgr - anch_twgr[i] for i, o in enumerate(ANCH_OFFS)},
                **{f"tdbc{int(o)}": hgr - beam_offsets_tw[:, i]
                   for i, o in enumerate(BEAM_OFFS)},
                **{f"tdsc{int(o)}": hgr - sc_offsets_tw[:, i]
                   for i, o in enumerate(SC_OFFS)},
                "tw_range":   sc(float(np.ptp(tw_tvt))),
                "tw_gr_mean": sc(float(tw_gr.mean())),
            }
            feats.update(roll_feats)
    
            result = pd.DataFrame(feats)
    
            if DBG_VERBOSE:
                _dbg(f"_build_well_from_df({wid}): rows={len(result)} cols={len(result.columns)}")
    
            if is_train:
                if "TVT" not in ev.columns or ev["TVT"].isna().all():
                    _dbg(f"_build_well_from_df({wid}): is_train but ev TVT all NaN", level="WARN")
                    return None
                result["target"] = (ev["TVT"].to_numpy(np.float32) - np.float32(last_tvt))
    
            return result
    
        except Exception as exc:
            _log_error(f"_build_well_from_df({wid})", exc)
            return None
    
    
    def build_well(hw_path: str, tw_path: str, is_train: bool) -> Optional[pd.DataFrame]:
        """I/O wrapper — reads CSVs then delegates to _build_well_from_df."""
        if _FI is None or _DI is None:
            _dbg(f"build_well({Path(hw_path).stem}): imputers not set!", level="ERROR")
            return None
        wid = Path(hw_path).stem.replace("__horizontal_well", "")
        try:
            hw = pd.read_csv(hw_path)
            tw = pd.read_csv(tw_path).sort_values("TVT")
        except Exception as exc:
            _log_error(f"build_well({wid}) read_csv", exc)
            return None
        return _build_well_from_df(hw, tw, is_train=is_train, wid=wid)
    
    
    # ═══════════════════════════════════════════════════════════════════════════════
    # Cal-Zone Augmentation — no temp files, reuses GR cache
    # ═══════════════════════════════════════════════════════════════════════════════
    
    def build_augmented(
        hw: pd.DataFrame,
        tw_tvt: np.ndarray,
        tw_gr: np.ndarray,
        wid: str,
        gr_cache: Optional[_WellGRCache] = None,
        n_splits: int = N_AUG_SPLITS,
        min_known: int = MIN_KNOWN_FOR_AUG,
    ) -> pd.DataFrame:
        known_all = hw[hw["TVT_input"].notna()]
        n_cal = len(known_all)
        if n_cal < min_known + 5:
            _dbg(f"build_augmented({wid}): too few known rows ({n_cal})", level="WARN")
            return pd.DataFrame()
    
        split_ks = np.unique(np.linspace(min_known, n_cal - 2, n_splits).astype(int))
        tw_mock  = pd.DataFrame({"TVT": tw_tvt, "GR": tw_gr})
        parts    = []
    
        for k in split_ks:
            hw_m = hw.copy()
            mask_start = known_all.index[k]
            hw_m.loc[mask_start:, "TVT_input"] = np.nan
    
            kn_m  = hw_m[hw_m["TVT_input"].notna()]
            ev_m  = hw_m[hw_m["TVT_input"].isna()]
            n_new = n_cal - k
    
            if len(ev_m) == 0 or len(kn_m) < 10 or n_new == 0:
                _dbg(f"build_augmented({wid}) k={k}: skipping", level="DEBUG")
                continue
    
            try:
                # ── Direct in-memory call — no temp files ──────────────────────
                feat = _build_well_from_df(
                    hw_m, tw_mock, is_train=True, wid=wid, gr_cache=gr_cache
                )
            except Exception as exc:
                _log_error(f"build_augmented({wid}) k={k}", exc)
                continue
    
            if feat is None or len(feat) == 0:
                _dbg(f"build_augmented({wid}) k={k}: build returned empty", level="DEBUG")
                continue
    
            feat_new = feat.iloc[:n_new].copy()
            feat_new["aug_k"] = np.int32(k)
            parts.append(feat_new)
            _dbg(f"build_augmented({wid}) k={k}: added {len(feat_new)} aug rows", level="DEBUG")
    
        if not parts:
            _dbg(f"build_augmented({wid}): no augmented parts produced", level="WARN")
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    
    
    def process_train_well(hw_path: Path) -> pd.DataFrame:
        """
        Top-level per-well train worker.
        Builds GR cache once, passes it to both the original call and all aug splits.
        """
        wid     = hw_path.stem.replace("__horizontal_well", "")
        tw_path = TRAIN_DIR / f"{wid}__typewell.csv"
        if not tw_path.exists():
            _dbg(f"process_train_well({wid}): typewell not found", level="WARN")
            return pd.DataFrame()
        try:
            hw = pd.read_csv(hw_path)
            tw = pd.read_csv(tw_path).sort_values("TVT")
            tw_tvt = tw["TVT"].to_numpy(np.float32)
            tw_gr  = tw["GR"].to_numpy(np.float32)
    
            # Build GR cache once — shared across original + all aug splits
            gr_mean  = float(np.nanmean(tw_gr))
            gr_cache = _WellGRCache(hw, gr_mean)
    
            # Original (full cal-zone) features
            feat_orig = _build_well_from_df(
                hw, tw, is_train=True, wid=wid, gr_cache=gr_cache
            )
            parts: list = []
            if feat_orig is not None and len(feat_orig) > 0:
                feat_orig["aug_k"] = np.int32(-1)
                parts.append(feat_orig)
            else:
                _dbg(f"process_train_well({wid}): build_well returned None/empty", level="WARN")
    
            # Augmented splits — pass same gr_cache to avoid recomputation
            feat_aug = build_augmented(hw, tw_tvt, tw_gr, wid, gr_cache=gr_cache)
            if len(feat_aug) > 0:
                parts.append(feat_aug)
    
            if not parts:
                _dbg(f"process_train_well({wid}): returning empty", level="WARN")
                return pd.DataFrame()
    
            r = pd.concat(parts, ignore_index=True)
            r["well_id"] = wid
            _dbg(f"process_train_well({wid}): done rows={len(r)}", level="DEBUG")
            return r
        except Exception as exc:
            _log_error(f"process_train_well({wid})", exc)
            return pd.DataFrame()
    
    
    def process_test_train(hw_path: Path) -> pd.DataFrame:
        """Online training: augment from test well calibration zone."""
        wid     = hw_path.stem.replace("__horizontal_well", "")
        tw_path = TEST_DIR / f"{wid}__typewell.csv"
        if not tw_path.exists():
            _dbg(f"process_test_train({wid}): typewell not found", level="WARN")
            return pd.DataFrame()
        try:
            hw = pd.read_csv(hw_path)
            tw = pd.read_csv(tw_path)
            if "TVT" not in tw.columns or "GR" not in tw.columns:
                _dbg(f"process_test_train({wid}): typewell missing TVT/GR", level="WARN")
                return pd.DataFrame()
            known = hw[hw["TVT_input"].notna()]
            if len(known) < MIN_KNOWN_FOR_AUG + 5:
                _dbg(f"process_test_train({wid}): too few known rows", level="WARN")
                return pd.DataFrame()
            hw_aug = hw.copy()
            hw_aug["TVT"] = hw_aug["TVT_input"]
            tw_tvt = tw["TVT"].to_numpy(np.float32)
            tw_gr  = tw["GR"].to_numpy(np.float32)
    
            # Build GR cache for test well augmentation
            gr_mean  = float(np.nanmean(tw_gr))
            gr_cache = _WellGRCache(hw_aug, gr_mean)
    
            feat_aug = build_augmented(hw_aug, tw_tvt, tw_gr, wid, gr_cache=gr_cache)
            if len(feat_aug) > 0:
                feat_aug["well_id"] = wid
                _dbg(f"process_test_train({wid}): aug rows={len(feat_aug)}", level="DEBUG")
            else:
                _dbg(f"process_test_train({wid}): no aug rows", level="WARN")
            return feat_aug
        except Exception as exc:
            _log_error(f"process_test_train({wid})", exc)
            return pd.DataFrame()
    
    
    def build_dataset(paths, is_train: bool, label: str) -> pd.DataFrame:
        args = [
            (str(p),
             str(p.parent / f"{p.stem.replace('__horizontal_well','')}__typewell.csv"),
             is_train)
            for p in paths
            if (p.parent / f"{p.stem.replace('__horizontal_well','')}__typewell.csv").exists()
        ]
        print(f"  {label}: {len(args)} wells | {NCPU} workers | process_pool={USE_PROCESS_POOL}")
        res = _starmap_with_imputers(build_well, args, NCPU)
        parts = [r for r in res if r is not None and len(r) > 0]
        print(f"  {label}: OK={len(parts)} skipped={len(args)-len(parts)}")
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    
    
    # ═══════════════════════════════════════════════════════════════════════════════
    # Main pipeline
    # ═══════════════════════════════════════════════════════════════════════════════
    
    hw_paths   = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
    if DEBUG_MAX_TRAIN_WELLS > 0:
        print(f"[DEBUG] Limiting train wells to first {DEBUG_MAX_TRAIN_WELLS}", flush=True)
        hw_paths = hw_paths[:DEBUG_MAX_TRAIN_WELLS]
    train_wids = [p.stem.replace("__horizontal_well", "") for p in hw_paths]
    print(f"Building imputers from {len(train_wids)} wells...")
    t0 = time.time()
    FI = FormationPlaneKNN(train_wids, TRAIN_DIR)
    DI = DenseANCCImputer(train_wids, TRAIN_DIR)
    # Set in main process too (for smoke tests)
    _FI = FI; _DI = DI
    print(f"  FI: {len(FI.df)} centroids | DI: {len(DI.ancc):,} pts  ({time.time()-t0:.0f}s)")
    
    
    # ── DEBUG: single-well smoke tests (run in main process, imputers already set) ──
    
    if DBG_SINGLE_WELL and len(hw_paths) > 0:
        _smoke_path = hw_paths[0]
        _smoke_wid  = _smoke_path.stem.replace("__horizontal_well", "")
        _smoke_tw   = TRAIN_DIR / f"{_smoke_wid}__typewell.csv"
        print(f"[SMOKE] Testing train well: {_smoke_wid}", flush=True)
        try:
            _smoke_result = build_well(str(_smoke_path), str(_smoke_tw), is_train=True)
            if _smoke_result is None:
                print("[SMOKE] build_well returned None", flush=True)
            else:
                print(f"[SMOKE] build_well OK: shape={_smoke_result.shape}", flush=True)
                if DBG_FEATURE_AUDIT:
                    print(f"[SMOKE] columns ({len(_smoke_result.columns)}):",
                          list(_smoke_result.columns), flush=True)
                nan_check = _smoke_result.isnull().sum()
                nan_check = nan_check[nan_check > 0]
                if len(nan_check):
                    print(f"[SMOKE] NaN columns: {dict(nan_check)}", flush=True)
                else:
                    print("[SMOKE] No NaN columns.", flush=True)
        except Exception as _e:
            print(f"[SMOKE] EXCEPTION: {_e}", flush=True)
            traceback.print_exc()
    
    test_paths = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
    if DEBUG_MAX_TEST_WELLS > 0:
        print(f"[DEBUG] Limiting test wells to first {DEBUG_MAX_TEST_WELLS}", flush=True)
        test_paths = test_paths[:DEBUG_MAX_TEST_WELLS]
    
    if DBG_SINGLE_TEST and len(test_paths) > 0:
        _smoke_tp   = test_paths[0]
        _smoke_twid = _smoke_tp.stem.replace("__horizontal_well", "")
        _smoke_twp  = TEST_DIR / f"{_smoke_twid}__typewell.csv"
        print(f"[SMOKE] Testing test well: {_smoke_twid}", flush=True)
        try:
            _smoke_t = build_well(str(_smoke_tp), str(_smoke_twp), is_train=False)
            if _smoke_t is None:
                print("[SMOKE] test build_well returned None", flush=True)
            else:
                print(f"[SMOKE] test build_well OK: shape={_smoke_t.shape}", flush=True)
        except Exception as _e:
            print(f"[SMOKE] test EXCEPTION: {_e}", flush=True)
            traceback.print_exc()
    
    
    if INFERENCE_ONLY:
        print("\nROGII_INFERENCE_ONLY=1; loading saved artifacts.", flush=True)
        artifact_dir = find_artifact_dir()
        manifest = load_json(artifact_dir / "manifest.json")
        config = load_json(artifact_dir / manifest.get("inference_config", "inference_config.json"))
        feature_cols = load_json(artifact_dir / manifest.get("feature_cols", "feature_cols.json"))
        result_keys = list(config.get("result_keys") or [])
        print(f"Artifact dir: {artifact_dir}", flush=True)
        print(f"Artifact result keys: {result_keys}", flush=True)
    
        test_df = build_dataset(test_paths, is_train=False, label="test")
        if test_df.empty:
            raise RuntimeError("No test features were built.")
        missing_features = [c for c in feature_cols if c not in test_df.columns]
        if missing_features:
            print(f"[WARN] Filling {len(missing_features)} missing artifact features with NaN.", flush=True)
            for col in missing_features:
                test_df[col] = np.nan
        Xt = test_df[feature_cols]
        pred_by_key: dict[str, np.ndarray] = {}
    
        def _entry_path(entry: dict) -> Path:
            return artifact_dir / str(entry["path"]).replace("/", os.sep)
    
        lgb_groups: dict[str, list[dict]] = {}
        for entry in manifest.get("lgb", []):
            lgb_groups.setdefault(f"lgb{entry['seed']}", []).append(entry)
        for key, entries in sorted(lgb_groups.items()):
            pred = np.zeros(len(test_df), dtype=np.float32)
            for entry in sorted(entries, key=lambda e: e.get("fold", 0)):
                booster = lgb.Booster(model_file=str(_entry_path(entry)))
                best_iter = int(entry.get("best_iteration") or booster.best_iteration or booster.current_iteration())
                pred += booster.predict(Xt, num_iteration=best_iter).astype(np.float32) / len(entries)
            pred_by_key[key] = pred
            print(f"Loaded {len(entries)} LGB folds for {key}", flush=True)
    
        cb_groups: dict[str, list[dict]] = {}
        for entry in manifest.get("catboost", []):
            cb_groups.setdefault(f"cb{entry['seed']}", []).append(entry)
        for key, entries in sorted(cb_groups.items()):
            pred = np.zeros(len(test_df), dtype=np.float32)
            for entry in sorted(entries, key=lambda e: e.get("fold", 0)):
                model = CatBoostRegressor()
                model.load_model(str(_entry_path(entry)))
                pred += model.predict(Xt.values).astype(np.float32) / len(entries)
            pred_by_key[key] = pred
            print(f"Loaded {len(entries)} CatBoost folds for {key}", flush=True)
    
        tabicl_needed = any(k.startswith("tabicl") for k in result_keys)
        tabicl_entries = manifest.get("tabicl_contexts", [])
        if tabicl_needed and tabicl_entries:
            if not RUN_TABICL:
                raise RuntimeError("Artifacts expect TabICL predictions, but ROGII_RUN_TABICL=0.")
            import subprocess as _sp
            from pathlib import Path as _Path
    
            tabicl_roots = [_Path("/kaggle/input")]
            if os.environ.get("ROGII_TABICL_DIR"):
                tabicl_roots.insert(0, _Path(os.environ["ROGII_TABICL_DIR"]))
            _wheels, _ckpts = [], []
            for _root in tabicl_roots:
                if _root.exists():
                    _wheels.extend(_root.rglob("tabicl-*.whl"))
                    _ckpts.extend(_root.rglob("tabicl-regressor*.ckpt"))
            if not _wheels or not _ckpts:
                raise RuntimeError("TabICL wheel/checkpoint not found for artifact inference.")
            _sp.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(_wheels[0])], check=True)
            from tabicl import TabICLRegressor
    
            tabicl_features = load_json(artifact_dir / manifest.get("tabicl_features", "tabicl_features.json"))
            Xt_top = Xt[tabicl_features].values
            label_groups: dict[str, list[dict]] = {}
            for entry in tabicl_entries:
                label = str(entry["label"])
                if label.startswith("tabicl_A"):
                    key = "tabicl_A"
                elif label.startswith("tabicl_B"):
                    key = "tabicl_B"
                else:
                    key = label
                label_groups.setdefault(key, []).append(entry)
    
            device = "cpu" if FORCE_CPU else "cuda"
            chunk = 50_000
            for key, entries in sorted(label_groups.items()):
                pred = np.zeros(len(test_df), dtype=np.float32)
                for entry in sorted(entries, key=lambda e: (e.get("fold", 0), e.get("seed", 0))):
                    ctx = np.load(_entry_path(entry))
                    reg = TabICLRegressor(
                        model_path=str(_ckpts[0]),
                        device=device,
                        random_state=int(entry.get("seed", 0)),
                        n_estimators=int(entry.get("n_estimators", 4)),
                        n_jobs=1,
                        verbose=False,
                        use_amp="auto",
                        batch_size=4,
                    )
                    reg.fit(ctx["X_ctx"], ctx["y_ctx"])
                    burnin_path = entry.get("burnin_path")
                    if burnin_path:
                        burn = np.load(artifact_dir / str(burnin_path).replace("/", os.sep))["X_va"]
                        print(
                            f"   burn {key} fold{entry.get('fold')} seed{entry.get('seed')}: "
                            f"{len(burn)} rows",
                            flush=True,
                        )
                        for i in range(0, len(burn), chunk):
                            e = min(i + chunk, len(burn))
                            _ = reg.predict(burn[i:e])
                        del burn
                    cur = np.empty(len(test_df), dtype=np.float32)
                    for i in range(0, len(Xt_top), chunk):
                        e = min(i + chunk, len(Xt_top))
                        cur[i:e] = reg.predict(Xt_top[i:e]).astype(np.float32)
                    pred += cur / len(entries)
                pred_by_key[key] = pred
                print(f"Loaded {len(entries)} TabICL contexts for {key}", flush=True)
    
        if not result_keys:
            result_keys = list(pred_by_key.keys())
        missing_results = [k for k in result_keys if k not in pred_by_key]
        if missing_results:
            raise RuntimeError(f"Missing artifact predictions for result keys: {missing_results}")
    
        St = np.column_stack([pred_by_key[k] for k in result_keys])
        stacker = config.get("stacker", {})
        if stacker.get("coef") is not None:
            coef = np.asarray(stacker["coef"], dtype=np.float32)
            if len(coef) != St.shape[1]:
                raise RuntimeError(f"Stack coef length {len(coef)} does not match predictions {St.shape[1]}.")
            final_test = St @ coef + float(stacker.get("intercept", 0.0))
            print(f"Using saved {stacker.get('kind', 'linear')} stack.", flush=True)
        elif config.get("use_ridge", False):
            coef = np.asarray(config["ridge_coef"], dtype=np.float32)
            if len(coef) != St.shape[1]:
                raise RuntimeError(f"Ridge coef length {len(coef)} does not match predictions {St.shape[1]}.")
            final_test = St @ coef + float(config.get("ridge_intercept", 0.0))
            print("Using saved ridge stack.", flush=True)
        else:
            final_test = St.mean(axis=1)
            print("Using saved simple-average stack.", flush=True)
    
        alpha = float(config.get("postproc_alpha", 1.0))
        tau = config.get("postproc_tau", None)
        w_pf = float(config.get("postproc_w_pf", 0.0))
        pf_delta = (test_df["pf_ancc"].values - test_df["last_known_tvt"].values).astype(np.float32)
        delta = ((1.0 - w_pf) * final_test.astype(np.float32) + w_pf * pf_delta).astype(np.float32)
        if tau is not None:
            delta *= 1.0 - np.exp(-np.maximum(test_df["md_since"].values, 0.0) / float(tau))
        test_df2 = test_df.copy()
        test_df2["pred"] = test_df2["last_known_tvt"].values + delta * alpha
    
        sample = pd.read_csv(SAMPLE)
        sub = sample[["id"]].merge(
            test_df2[["id", "pred"]].rename(columns={"pred": "tvt"}),
            on="id", how="left",
        )
        fallback_tvt = float(config.get("fallback_tvt", test_df2["pred"].mean()))
        sub["tvt"] = sub["tvt"].fillna(fallback_tvt)
        print(
            f"[SUB AUDIT] rows={len(sub)} null={int(sub['tvt'].isnull().sum())} "
            f"fallback_filled={int((sub['tvt'] == fallback_tvt).sum())}",
            flush=True,
        )
    
        if config.get("exact_overlap_enabled", True):
            os.environ["ROGII_EXACT_OVERLAP"] = "1"
            os.environ["ROGII_EXACT_BLEND_WEIGHT"] = str(config.get("exact_blend_weight", 0.28))
        else:
            os.environ["ROGII_EXACT_OVERLAP"] = "0"
        sub = apply_exact_train_coordinate_blend(sub[["id", "tvt"]], DATA)
        sub[["id", "tvt"]].to_csv(OUT, index=False)
        print(f"\n✅  {OUT}  {len(sub)} rows (artifact inference)")
        print(sub.head(8).to_string(index=False))
        print(f"=== ARTIFACT INFERENCE COMPLETE  total={time.time()-_T0_GLOBAL:.0f}s ===")
        raise SystemExit(0)
    
    
    # ── Build train features — process pool with shared imputers ─────────────────
    print("\nLoading/building official train-core features...")
    t0 = time.time()
    
    train_core_df = None
    if USE_TRAIN_CORE_CACHE and DEBUG_MAX_TRAIN_WELLS <= 0:
        for cache_path in _cache_candidates():
            if cache_path.exists():
                print(f"Loading train-core cache: {cache_path}", flush=True)
                train_core_df = pd.read_pickle(cache_path)
                print(f"  train_core_df={train_core_df.shape} loaded in {time.time()-t0:.0f}s", flush=True)
                break
    
    if train_core_df is None:
        print("\nBuilding official train-core features...")
        res = _map_with_imputers(process_train_well, hw_paths, NCPU)
    
        if DBG_PARALLEL_STATS:
            n_none  = sum(1 for r in res if r is None)
            n_empty = sum(1 for r in res if r is not None and len(r) == 0)
            n_ok    = sum(1 for r in res if r is not None and len(r) > 0)
            row_dist = [len(r) for r in res if r is not None and len(r) > 0]
            print(f"[PARALLEL STATS train] None={n_none} empty={n_empty} ok={n_ok}", flush=True)
            if row_dist:
                print(f"  rows per well: min={min(row_dist)} max={max(row_dist)} "
                      f"mean={np.mean(row_dist):.0f} total={sum(row_dist)}", flush=True)
            else:
                print("  *** All results None or empty! ***", flush=True)
    
        parts_core = [r for r in res if r is not None and len(r) > 0]
        if len(parts_core) == 0:
            print("[FATAL] No valid official train DataFrames.", flush=True)
            raise RuntimeError("All process_train_well calls returned None or empty.")
    
        train_core_df = pd.concat(parts_core, ignore_index=True)
        del res, parts_core
        gc.collect()
        print(f"  train_core_df={train_core_df.shape} built in {time.time()-t0:.0f}s", flush=True)
    
        if WRITE_TRAIN_CORE_CACHE or CACHE_ONLY:
            cache_out = _train_core_cache_path_for_write()
            cache_out.parent.mkdir(parents=True, exist_ok=True)
            train_core_df.to_pickle(cache_out)
            preview_cols = [c for c in ["well", "id", "last_known_tvt", "target", "aug_k"] if c in train_core_df.columns]
            train_core_df[preview_cols].head(1000).to_csv(
                cache_out.with_name("aeroridge_train_core_preview_1000.csv"), index=False
            )
            schema = pd.DataFrame({
                "column": train_core_df.columns,
                "dtype": [str(train_core_df[c].dtype) for c in train_core_df.columns],
                "non_null_count": [int(train_core_df[c].notna().sum()) for c in train_core_df.columns],
                "null_count": [int(train_core_df[c].isna().sum()) for c in train_core_df.columns],
            })
            schema["null_rate"] = schema["null_count"] / max(len(train_core_df), 1)
            schema.to_csv(cache_out.with_name("aeroridge_train_core_schema.csv"), index=False)
            print(f"  wrote train-core cache: {cache_out}", flush=True)
    
    if CACHE_ONLY:
        print("ROGII_CACHE_ONLY=1, stopping after train-core cache build.", flush=True)
        raise SystemExit(0)
    
    parts = [train_core_df]
    
    # ── Online training on test wells ────────────────────────────────────────────
    print(f"\nOnline training on {len(test_paths)} test wells...")
    t1 = time.time()
    
    res_t = _map_with_imputers(process_test_train, test_paths, min(NCPU, max(1, len(test_paths))))
    
    if DBG_PARALLEL_STATS:
        n_none_t  = sum(1 for r in res_t if r is None)
        n_empty_t = sum(1 for r in res_t if r is not None and len(r) == 0)
        n_ok_t    = sum(1 for r in res_t if r is not None and len(r) > 0)
        print(f"[PARALLEL STATS test-online] None={n_none_t} empty={n_empty_t} ok={n_ok_t}",
              flush=True)
    
    parts.extend([r for r in res_t if r is not None and len(r) > 0])
    del res_t
    
    # ── Pre-concat guard ─────────────────────────────────────────────────────────
    if len(parts) == 0:
        print("[FATAL] No valid DataFrames to concatenate.", flush=True)
        raise RuntimeError(
            "All process_train_well / process_test_train calls returned None or empty."
        )
    
    train_df = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    print(f"\ntrain_df: {train_df.shape} | {time.time()-t0:.0f}s")
    orig = (train_df["aug_k"] == -1).sum()
    aug  = (train_df["aug_k"] >=  0).sum()
    print(f"  Original: {orig:,}  Augmented: {aug:,} (+{aug/max(orig,1)*100:.0f}%)")
    
    _dbg_df("train_df", train_df)
    
    SKIP = {"well", "well_id", "id", "target", "aug_k"}
    feature_cols = [c for c in train_df.columns if c not in SKIP]
    print(f"Features: {len(feature_cols)}")
    if SAVE_ARTIFACTS:
        save_json(ARTIFACT_DIR / "feature_cols.json", feature_cols)
        ARTIFACT_MANIFEST["feature_cols"] = "feature_cols.json"
        ARTIFACT_MANIFEST["data"] = {
            "train_df_shape": list(train_df.shape),
            "test_df_shape": None,
            "n_features": len(feature_cols),
            "n_splits": N_SPLITS,
            "seed": SEED,
            "used_train_core_cache": str(os.environ.get("ROGII_TRAIN_CORE_CACHE", "")),
        }
    
    if DBG_FEATURE_AUDIT:
        print(f"[FEATURE AUDIT] {len(feature_cols)} feature columns:", flush=True)
        for _fc in feature_cols:
            _arr = train_df[_fc].to_numpy()
            _nans = int(np.isnan(_arr.astype(float)).sum()) if np.issubdtype(_arr.dtype, np.number) else 0
            if _nans > 0:
                print(f"  NaN: {_fc}  count={_nans}/{len(_arr)}", flush=True)
    
    X  = train_df[feature_cols]
    y  = train_df["target"]
    g  = train_df["well"]
    
    if DBG_NAN_AUDIT:
        _dbg("NaN audit on X (train features)")
        _nan_X = X.isnull().sum()
        _nan_X = _nan_X[_nan_X > 0]
        if len(_nan_X):
            print(f"[NaN AUDIT] X has NaN in {len(_nan_X)} cols:", flush=True)
            print(_nan_X.to_string(), flush=True)
        else:
            print("[NaN AUDIT] X: clean (no NaN)", flush=True)
        _nan_y = int(y.isnull().sum())
        print(f"[NaN AUDIT] y: {_nan_y} NaN  "
              f"min={y.min():.2f} max={y.max():.2f} mean={y.mean():.2f}", flush=True)
        _inf_X = np.isinf(X.to_numpy(dtype=float)).sum()
        print(f"[NaN AUDIT] X Inf count: {_inf_X}", flush=True)
    
    # ── Test features ────────────────────────────────────────────────────────────
    test_df = build_dataset(test_paths, is_train=False, label="test")
    Xt = test_df[feature_cols]
    if SAVE_ARTIFACTS:
        ARTIFACT_MANIFEST.setdefault("data", {})["test_df_shape"] = list(test_df.shape)
        ARTIFACT_MANIFEST["created_outputs"]["test_feature_preview"] = None
    
    if DBG_NAN_AUDIT:
        _nan_Xt = Xt.isnull().sum()
        _nan_Xt = _nan_Xt[_nan_Xt > 0]
        if len(_nan_Xt):
            print(f"[NaN AUDIT] Xt has NaN in {len(_nan_Xt)} cols:", flush=True)
            print(_nan_Xt.to_string(), flush=True)
        else:
            print("[NaN AUDIT] Xt: clean (no NaN)", flush=True)
        _inf_Xt = np.isinf(Xt.to_numpy(dtype=float)).sum()
        print(f"[NaN AUDIT] Xt Inf count: {_inf_Xt}", flush=True)
    
    gc.collect()
    _dbg(f"Feature matrix ready: X={X.shape} Xt={Xt.shape}")
    
    _unique_wells = np.unique(g.values)
    _rng_cv       = np.random.RandomState(SEED)
    _shuffled     = _rng_cv.permutation(_unique_wells)
    _fold_map     = {w: i % N_SPLITS for i, w in enumerate(_shuffled)}
    _fold_ids     = np.array([_fold_map[w] for w in g.values])
    
    # Boolean mask: True for original rows (not augmented)
    _is_orig = (train_df["aug_k"].values == -1)
    
    splits: list = []
    for _f in range(N_SPLITS):
        _tr_idx = np.where(_fold_ids != _f)[0]                      # all rows not in fold f
        _va_all = np.where(_fold_ids == _f)[0]                      # fold-f rows
        _va_idx = _va_all[_is_orig[_va_all]]                        # keep only original rows
        if len(_va_idx) == 0:
            print(f"[CV] WARNING: fold {_f} has zero original val rows — skipping", flush=True)
            continue
        splits.append((_tr_idx, _va_idx))
    
    print(f"[CV] {len(splits)} folds | "
          f"train aug+orig per fold ≈ {np.mean([len(t) for t,_ in splits]):.0f} | "
          f"val orig-only per fold ≈ {np.mean([len(v) for _,v in splits]):.0f}")
    
    
    
    def run_lgb(seed: int, gpu_id: int = 0):
        p   = dict(LGB_P, n_estimators=5000, seed=seed)
        if not FORCE_CPU:
            p["gpu_device_id"] = gpu_id
        oof = np.zeros(len(train_df), np.float32)
        tp  = np.zeros(len(test_df),  np.float32)
        for fold, (tr, va) in enumerate(splits):
            _dbg(f"LGB seed={seed} fold={fold} starting")
            ds_tr = lgb.Dataset(X.iloc[tr], label=y.iloc[tr])
            ds_va = lgb.Dataset(X.iloc[va], label=y.iloc[va], reference=ds_tr)
            m = lgb.train(
                p,
                ds_tr,
                valid_sets=[ds_va],
                num_boost_round=p["n_estimators"],
                callbacks=[
                    lgb.early_stopping(125, verbose=False),
                    lgb.log_evaluation(250),
                ],
            )
            ni = m.best_iteration
            oof[va] = m.predict(X.iloc[va], num_iteration=ni).astype(np.float32)
            tp      += m.predict(Xt, num_iteration=ni).astype(np.float32) / len(splits)
            fold_rmse = root_mean_squared_error(y.iloc[va], oof[va])
            print(f"   LGB{seed} fold{fold}: best_iter={ni} rmse={fold_rmse:.4f}")
            if SAVE_ARTIFACTS:
                model_path = ARTIFACT_DIR / "lgb" / f"seed{seed}_fold{fold}.txt"
                model_path.parent.mkdir(parents=True, exist_ok=True)
                m.save_model(str(model_path), num_iteration=ni)
                ARTIFACT_MANIFEST["lgb"].append({
                    "seed": int(seed),
                    "fold": int(fold),
                    "best_iteration": int(ni),
                    "rmse": float(fold_rmse),
                    "path": _artifact_rel(model_path),
                })
            _dbg(f"LGB seed={seed} fold={fold} done rmse={fold_rmse:.4f}")
        # OOF only over rows that were in some validation set
        va_all = np.concatenate([va for _, va in splits])
        r = root_mean_squared_error(y.iloc[va_all], oof[va_all])
        print(f"   LGB{seed} OOF={r:.4f}")
        return oof, tp, r
    
    
    def run_cb(seed: int = 42):
        p   = dict(CB_P, random_seed=seed)
        oof = np.zeros(len(train_df), np.float32)
        tp  = np.zeros(len(test_df),  np.float32)
        for fold, (tr, va) in enumerate(splits):
            _dbg(f"CB fold={fold} starting")
            m = CatBoostRegressor(**p)
            m.fit(Pool(X.iloc[tr].values, label=y.iloc[tr].values),
                  eval_set=Pool(X.iloc[va].values, label=y.iloc[va].values),
                  use_best_model=True)
            oof[va] = m.predict(X.iloc[va].values).astype(np.float32)
            tp      += m.predict(Xt.values).astype(np.float32) / len(splits)
            fold_rmse = root_mean_squared_error(y.iloc[va], oof[va])
            print(f"   CB{seed} fold{fold}: rmse={fold_rmse:.4f}")
            if SAVE_ARTIFACTS:
                model_path = ARTIFACT_DIR / "catboost" / f"seed{seed}_fold{fold}.cbm"
                model_path.parent.mkdir(parents=True, exist_ok=True)
                m.save_model(str(model_path))
                ARTIFACT_MANIFEST["catboost"].append({
                    "seed": int(seed),
                    "fold": int(fold),
                    "rmse": float(fold_rmse),
                    "path": _artifact_rel(model_path),
                })
            _dbg(f"CB fold={fold} done rmse={fold_rmse:.4f}")
        va_all = np.concatenate([va for _, va in splits])
        r = root_mean_squared_error(y.iloc[va_all], oof[va_all])
        print(f"   CB{seed} OOF={r:.4f}")
        return oof, tp, r
    
    
    import threading
    
    _lgb_results = {}
    _lgb_exc     = [None]
    
    def _run_lgb_parallel():
        """LGB[42,7] on GPU 0; LGB[123] on GPU 1 — parallel threads."""
        try:
            import threading as _t
    
            def _gpu0():
                for s in [42, 7]:
                    o, t, r = run_lgb(s, gpu_id=0)
                    _lgb_results[f'lgb{s}'] = {'oof': o, 'test': t, 'rmse': r}
    
            def _gpu1():
                o, t, r = run_lgb(123, gpu_id=1)
                _lgb_results['lgb123'] = {'oof': o, 'test': t, 'rmse': r}
    
            t0 = _t.Thread(target=_gpu0)
            t1 = _t.Thread(target=_gpu1)
            t0.start(); t1.start()
            t0.join();  t1.join()
        except Exception as e:
            _lgb_exc[0] = e
    
    print(">> GPU0: LGB[42,7]  |  GPU1: LGB[123]  (parallel)")
    _run_lgb_parallel()
    if _lgb_exc[0]: raise _lgb_exc[0]
    
    results = {}
    results.update(_lgb_results)
    
    # CB×3 seeds run serially. Device string is detected from available Kaggle GPUs.
    for seed in [42, 7, 123]:
        oof_cb, tp_cb, r_cb = run_cb(seed=seed)
        results[f'cb{seed}'] = {'oof': oof_cb, 'test': tp_cb, 'rmse': r_cb}
    
    _va_union = np.concatenate([va for _, va in splits])
    
    # ── TabICL setup ─────────────────────────────────────────────────────────────
    if RUN_TABICL:
        import subprocess as _sp, sys as _sys
        from pathlib import Path as _Path
    
        tabicl_roots = [_Path("/kaggle/input")]
        if os.environ.get("ROGII_TABICL_DIR"):
            tabicl_roots.insert(0, _Path(os.environ["ROGII_TABICL_DIR"]))
        _wheels = []
        _ckpts = []
        for _root in tabicl_roots:
            if _root.exists():
                _wheels.extend(_root.rglob("tabicl-*.whl"))
                _ckpts.extend(_root.rglob("tabicl-regressor*.ckpt"))
        print(f"found wheels: {_wheels}")
        print(f"found ckpts:  {_ckpts}")
        if not _wheels or not _ckpts:
            _msg = (
                "TabICL artifacts not found. Add the public dataset "
                "needless090/rogii-tabicl-mirror, or set ROGII_RUN_TABICL=0."
            )
            if RUNNING_ON_KAGGLE:
                raise RuntimeError(_msg)
            print(_msg + " Skipping TabICL for this local run.", flush=True)
        else:
            _sp.run([_sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                     str(_wheels[0])], check=True)
            TABICL_CKPT = str(_ckpts[0])
            print(f"TabICL ckpt: {TABICL_CKPT}")
    
            print(">> Selecting top-50 features for TabICL", flush=True)
            _tr0, _ = splits[0]
            _samp = np.random.RandomState(42).choice(_tr0, size=min(200_000, len(_tr0)), replace=False)
            _lgb_sel_params = dict(
                n_estimators=300, learning_rate=0.05, num_leaves=63,
                n_jobs=-1, verbosity=-1,
            )
            if not FORCE_CPU:
                _lgb_sel_params["device_type"] = "gpu"
            _lgb_sel = lgb.LGBMRegressor(**_lgb_sel_params)
            _lgb_sel.fit(X.iloc[_samp].values, y.iloc[_samp].values)
            _imp = pd.Series(_lgb_sel.feature_importances_, index=feature_cols).sort_values(ascending=False)
            TABICL_FEATS = _imp.head(50).index.tolist()
            print(f"   top-5: {TABICL_FEATS[:5]}")
            if SAVE_ARTIFACTS:
                save_json(ARTIFACT_DIR / "tabicl_features.json", TABICL_FEATS)
                ARTIFACT_MANIFEST["tabicl_features"] = "tabicl_features.json"
    
            X_top  = X[TABICL_FEATS].values
            Xt_top = Xt[TABICL_FEATS].values
            CHUNK  = 50_000
    
            def run_tabicl(ctx_n, n_estimators, seeds, label):
                from tabicl import TabICLRegressor
                print(f"\n>> {label}: ctx={ctx_n} n_est={n_estimators} seeds={seeds}", flush=True)
                n_per = len(seeds)
                oof = np.zeros(len(train_df), dtype=np.float32)
                test_pred = np.zeros(len(test_df), dtype=np.float32)
                for fold, (tr, va) in enumerate(splits):
                    oof_fold = np.zeros(len(va), dtype=np.float32)
                    for sd in seeds:
                        ctx_idx = np.random.RandomState(sd * 1000 + fold).choice(
                            tr, size=min(ctx_n, len(tr)), replace=False)
                        X_ctx = X_top[ctx_idx]
                        y_ctx = y.iloc[ctx_idx].values.astype(np.float32)
                        if SAVE_ARTIFACTS:
                            ctx_path = (
                                ARTIFACT_DIR / "tabicl_contexts"
                                / f"{label}_fold{fold}_seed{sd}.npz"
                            )
                            ctx_path.parent.mkdir(parents=True, exist_ok=True)
                            np.savez_compressed(
                                ctx_path,
                                X_ctx=X_ctx.astype(np.float32),
                                y_ctx=y_ctx.astype(np.float32),
                            )
                            ARTIFACT_MANIFEST["tabicl_contexts"].append({
                                "label": label,
                                "fold": int(fold),
                                "seed": int(sd),
                                "ctx_n": int(ctx_n),
                                "n_estimators": int(n_estimators),
                                "path": _artifact_rel(ctx_path),
                            })
                        X_va  = X_top[va]
                        t0_ = time.time()
                        reg = TabICLRegressor(model_path=TABICL_CKPT, device="cuda",
                                              random_state=sd, n_estimators=n_estimators,
                                              n_jobs=1, verbose=False, use_amp="auto", batch_size=4)
                        reg.fit(X_ctx, y_ctx)
                        pv = np.empty(len(va), dtype=np.float32)
                        for i in range(0, len(va), CHUNK):
                            e = min(i + CHUNK, len(va))
                            pv[i:e] = reg.predict(X_va[i:e]).astype(np.float32)
                        oof_fold += pv / n_per
                        tf = np.empty(len(test_df), dtype=np.float32)
                        for i in range(0, len(Xt_top), CHUNK):
                            e = min(i + CHUNK, len(Xt_top))
                            tf[i:e] = reg.predict(Xt_top[i:e]).astype(np.float32)
                        test_pred += tf / N_SPLITS / n_per
                        print(f"   fold{fold} seed{sd}: [{time.time()-t0_:.1f}s]", flush=True)
                    oof[va] = oof_fold
                    rmse_fold = root_mean_squared_error(y.iloc[va].values, oof_fold)
                    print(f"   fold{fold} avg RMSE = {rmse_fold:.4f}", flush=True)
                r_val = root_mean_squared_error(y.iloc[_va_union].values, oof[_va_union])
                print(f"   {label} OOF RMSE (val rows) = {r_val:.4f}", flush=True)
                return oof, test_pred, float(r_val)
    
            # TabICL A: ctx=4096, 4 estimators, 5 seeds (~74 min)
            oof_A, test_A, rmse_A = run_tabicl(4096, 4, [0, 1, 2, 3, 4], "tabicl_A_4096_5seed")
            results["tabicl_A"] = {"oof": oof_A, "test": test_A, "rmse": rmse_A}
    
            # TabICL B: ctx=8192, 4 estimators, 1 seed (~19 min)
            oof_B, test_B, rmse_B = run_tabicl(8192, 4, [42], "tabicl_B_8192_1seed")
            results["tabicl_B"] = {"oof": oof_B, "test": test_B, "rmse": rmse_B}
    else:
        print("ROGII_RUN_TABICL=0; skipping TabICL models.", flush=True)
    
    # Ridge stacking — fit and score only over rows that were actually validated
    result_keys = list(results.keys())
    Sx = np.column_stack([v["oof"] for v in results.values()]).astype(np.float32)
    St = np.column_stack([v["test"] for v in results.values()]).astype(np.float32)
    y_val_stack = y.iloc[_va_union].values.astype(np.float32)
    Sx_val = Sx[_va_union]
    
    
    def _rmse_np(y_true: np.ndarray, pred: np.ndarray) -> float:
        diff = pred.astype(np.float64) - y_true.astype(np.float64)
        return float(np.sqrt(np.mean(diff * diff)))
    
    
    def hill_climb_stack(
        pred_mat: np.ndarray,
        y_true: np.ndarray,
        keys: list[str],
        precision: float = 0.01,
        allow_negative: bool = True,
    ):
        """Small deterministic hill-climb stacker, package-free."""
        precision = max(float(precision), 0.001)
        weights = np.arange(-0.5, 0.5001, precision) if allow_negative else np.arange(precision, 0.5001, precision)
        model_scores = [_rmse_np(y_true, pred_mat[:, i]) for i in range(pred_mat.shape[1])]
        start = int(np.argmin(model_scores))
        coef = np.zeros(pred_mat.shape[1], dtype=np.float64)
        coef[start] = 1.0
        current = pred_mat[:, start].astype(np.float32).copy()
        current_score = model_scores[start]
        remaining = [i for i in range(pred_mat.shape[1]) if i != start]
        history = [{
            "iteration": 0,
            "model": keys[start],
            "weight": 1.0,
            "score": float(current_score),
        }]
        iteration = 0
        while remaining:
            iteration += 1
            best = (current_score, None, None, None)
            for idx in remaining:
                new_pred = pred_mat[:, idx]
                for w in weights:
                    cand = (1.0 - w) * current + w * new_pred
                    score = _rmse_np(y_true, cand)
                    if score < best[0] - 1e-7:
                        best = (score, idx, float(w), cand.astype(np.float32))
            if best[1] is None:
                break
            score, idx, w, current = best
            coef *= (1.0 - w)
            coef[idx] += w
            current_score = float(score)
            remaining.remove(idx)
            history.append({
                "iteration": iteration,
                "model": keys[idx],
                "weight": float(w),
                "score": current_score,
            })
        return coef.astype(np.float32), current.astype(np.float32), current_score, history
    
    
    avg_coef = np.ones(len(result_keys), dtype=np.float32) / max(len(result_keys), 1)
    avg_oof = Sx_val.mean(1).astype(np.float32)
    avg_test = St.mean(1).astype(np.float32)
    r_avg = _rmse_np(y_val_stack, avg_oof)
    
    ridge = Ridge(alpha=1.0, fit_intercept=False, positive=True)
    ridge.fit(Sx_val, y_val_stack)
    ridge_coef = ridge.coef_.astype(np.float32)
    ridge_oof = ridge.predict(Sx_val).astype(np.float32)
    ridge_test = ridge.predict(St).astype(np.float32)
    r_stk = _rmse_np(y_val_stack, ridge_oof)
    ridge_wts = ridge_coef / max(float(ridge_coef.sum()), 1e-9)
    
    stack_options = [
        {
            "kind": "simple_average",
            "coef": avg_coef,
            "intercept": 0.0,
            "oof": avg_oof,
            "test": avg_test,
            "rmse": r_avg,
            "history": [],
        },
        {
            "kind": "positive_ridge",
            "coef": ridge_coef,
            "intercept": float(getattr(ridge, "intercept_", 0.0)),
            "oof": ridge_oof,
            "test": ridge_test,
            "rmse": r_stk,
            "history": [],
        },
    ]
    
    if USE_HILL_STACK and len(result_keys) >= 2:
        hill_coef, hill_oof, hill_rmse, hill_history = hill_climb_stack(
            Sx_val,
            y_val_stack,
            result_keys,
            precision=HILL_PRECISION,
            allow_negative=True,
        )
        hill_test = (St @ hill_coef).astype(np.float32)
        stack_options.append({
            "kind": "hill_climb",
            "coef": hill_coef,
            "intercept": 0.0,
            "oof": hill_oof,
            "test": hill_test,
            "rmse": hill_rmse,
            "history": hill_history,
        })
    
    best_stack = min(stack_options, key=lambda item: item["rmse"])
    final_test = best_stack["test"]
    _final_oof_full = Sx.mean(1).astype(np.float32)
    _final_oof_full[_va_union] = best_stack["oof"]
    final_oof = _final_oof_full
    
    print(f"\nSimple avg OOF (val rows): {r_avg:.4f}")
    print(f"Ridge stk OOF (val rows): {r_stk:.4f}  wts={dict(zip(result_keys, ridge_wts.round(4)))}")
    if USE_HILL_STACK and len(result_keys) >= 2:
        print(f"Hill stk OOF (val rows): {stack_options[-1]['rmse']:.4f}  coefs={dict(zip(result_keys, stack_options[-1]['coef'].round(4)))}")
    print(f"Using {best_stack['kind']} predictions.")
    print(f"Final OOF RMSE (val rows): {best_stack['rmse']:.4f}")
    
    
    # ═══════════════════════════════════════════════════════════════════════════════
    # Post-Processing
    # ═══════════════════════════════════════════════════════════════════════════════
    # Grid-search alpha/tau only on val rows (where OOF is clean).
    # SavGol smoothing is REMOVED from test predictions — it introduced bias on the
    # initial rows of each well and hurt the leaderboard score vs the reference.
    
    base_val = train_df["last_known_tvt"].values[_va_union]
    ytrue_val = y.values[_va_union] + base_val
    md_val = train_df["md_since"].values[_va_union]
    oof_val = final_oof[_va_union]
    pf_val = (train_df["pf_ancc"].values[_va_union] - base_val).astype(np.float32)
    
    best_cfg, best_r = (None, None, None), np.inf
    for alpha in np.arange(0.6, 1.01, 0.05):
        for tau in [None, 30.0, 60.0, 120.0, 250.0, 500.0]:
            for w_pf in np.arange(0.0, 0.501, 0.05):
                d = ((1.0 - w_pf) * oof_val + w_pf * pf_val).astype(np.float32)
                if tau:
                    d *= 1.0 - np.exp(-np.maximum(md_val, 0.0) / tau)
                r = root_mean_squared_error(ytrue_val, base_val + d * alpha)
                if r < best_r:
                    best_r, best_cfg = r, (float(alpha), tau, float(w_pf))
    
    print(
        f"Best post-proc: alpha={best_cfg[0]:.2f} tau={best_cfg[1]} "
        f"w_pf={best_cfg[2]:.2f}  TVT RMSE={best_r:.4f}"
    )
    ALPHA, TAU, W_PF = best_cfg
    _fallback_tvt = float(
        train_df["last_known_tvt"].iloc[_va_union].mean()
        + train_df["target"].iloc[_va_union].mean()
    )
    if SAVE_ARTIFACTS:
        stacker_config = {
            "kind": best_stack["kind"],
            "coef": best_stack["coef"].astype(float).tolist(),
            "intercept": float(best_stack.get("intercept", 0.0)),
            "rmse": float(best_stack["rmse"]),
            "history": best_stack.get("history", []),
            "hill_precision": float(HILL_PRECISION),
        }
        config = {
            "result_keys": result_keys,
            "stacker": stacker_config,
            "use_ridge": bool(best_stack["kind"] == "positive_ridge"),
            "ridge_alpha": 1.0,
            "ridge_fit_intercept": False,
            "ridge_coef": ridge_coef.astype(float).tolist(),
            "ridge_intercept": float(getattr(ridge, "intercept_", 0.0)),
            "simple_avg_oof": float(r_avg),
            "ridge_oof": float(r_stk),
            "final_oof": float(best_stack["rmse"]),
            "postproc_alpha": float(ALPHA),
            "postproc_tau": None if TAU is None else float(TAU),
            "postproc_w_pf": float(W_PF),
            "postproc_oof_rmse": float(best_r),
            "fallback_tvt": _fallback_tvt,
            "exact_overlap_enabled": os.environ.get("ROGII_EXACT_OVERLAP", "1").strip().lower()
            not in {"0", "false", "no"},
            "exact_blend_weight": env_float("ROGII_EXACT_BLEND_WEIGHT", 0.28),
        }
        save_json(ARTIFACT_DIR / "inference_config.json", config)
        ARTIFACT_MANIFEST["inference_config"] = "inference_config.json"
        ARTIFACT_MANIFEST["result_keys"] = result_keys
        diag_dir = ARTIFACT_DIR / "diagnostics"
        diag_dir.mkdir(parents=True, exist_ok=True)
        np.save(diag_dir / "base_test_predictions.npy", St.astype(np.float32))
        ARTIFACT_MANIFEST["created_outputs"]["base_test_predictions"] = _artifact_rel(
            diag_dir / "base_test_predictions.npy"
        )
        np.savez_compressed(
            diag_dir / "oof_val_predictions.npz",
            predictions=Sx_val.astype(np.float32),
            target=y_val_stack.astype(np.float32),
            val_indices=_va_union.astype(np.int64),
        )
        save_json(diag_dir / "prediction_keys.json", result_keys)
        ARTIFACT_MANIFEST["created_outputs"]["oof_val_predictions"] = _artifact_rel(
            diag_dir / "oof_val_predictions.npz"
        )
        ARTIFACT_MANIFEST["created_outputs"]["prediction_keys"] = _artifact_rel(
            diag_dir / "prediction_keys.json"
        )
        meta_cols = [
            c for c in ["well", "id", "target", "last_known_tvt", "md_since", "pf_ancc", "aug_k"]
            if c in train_df.columns
        ]
        train_df.loc[_va_union, meta_cols].to_csv(
            diag_dir / "oof_val_meta.csv.gz",
            index=False,
            compression="gzip",
        )
        ARTIFACT_MANIFEST["created_outputs"]["oof_val_meta"] = _artifact_rel(
            diag_dir / "oof_val_meta.csv.gz"
        )
        test_diag = pd.DataFrame({"id": test_df["id"].values})
        for i, key in enumerate(result_keys):
            test_diag[key] = St[:, i].astype(np.float32)
        test_diag.to_csv(diag_dir / "test_base_predictions.csv.gz", index=False, compression="gzip")
        ARTIFACT_MANIFEST["created_outputs"]["test_base_predictions"] = _artifact_rel(
            diag_dir / "test_base_predictions.csv.gz"
        )
        if SAVE_FEATURE_FRAMES:
            frame_dir = ARTIFACT_DIR / "feature_frames"
            frame_dir.mkdir(parents=True, exist_ok=True)
            train_df.to_pickle(frame_dir / "train_features.pkl")
            test_df.to_pickle(frame_dir / "test_features.pkl")
            ARTIFACT_MANIFEST["created_outputs"]["train_features"] = _artifact_rel(
                frame_dir / "train_features.pkl"
            )
            ARTIFACT_MANIFEST["created_outputs"]["test_features"] = _artifact_rel(
                frame_dir / "test_features.pkl"
            )
    
    
    def apply_pp(df: pd.DataFrame, delta: np.ndarray,
                 alpha: float, tau, w_pf: float = 0.0) -> np.ndarray:
        pf_delta = (df["pf_ancc"].values - df["last_known_tvt"].values).astype(np.float32)
        d = ((1.0 - w_pf) * delta.astype(np.float32) + w_pf * pf_delta).astype(np.float32)
        if tau:
            d *= 1.0 - np.exp(-np.maximum(df["md_since"].values, 0.0) / tau)
        return d * alpha
    
    
    # Apply post-processing to test predictions — no SavGol smoothing
    test_df2 = test_df.copy()
    test_df2["pred"] = (test_df2["last_known_tvt"].values
                        + apply_pp(test_df2, final_test, ALPHA, TAU, W_PF))
    
    sample = pd.read_csv(SAMPLE)
    sub = sample[["id"]].merge(
        test_df2[["id", "pred"]].rename(columns={"pred": "tvt"}),
        on="id", how="left",
    )
    _fb_val = _fallback_tvt
    sub["tvt"] = sub["tvt"].fillna(_fb_val)
    
    n_fill = sub["tvt"].isnull().sum()
    n_fb   = (sub["tvt"] == _fb_val).sum()
    print(f"[SUB AUDIT] rows={len(sub)}  null={n_fill}  fallback_filled={n_fb}", flush=True)
    
    sub = apply_exact_train_coordinate_blend(sub[["id", "tvt"]], DATA)
    sub[["id", "tvt"]].to_csv('9.537.csv', index=False)
    if SAVE_ARTIFACTS:
        ARTIFACT_MANIFEST["created_outputs"]["submission"] = str(OUT)
        save_json(ARTIFACT_DIR / "manifest.json", ARTIFACT_MANIFEST)
        print(f"Artifact manifest -> {ARTIFACT_DIR / 'manifest.json'}", flush=True)
    
    print(f"\n✅  {OUT}  {len(sub)} rows")
    print("\n─── Final Summary ───────────────────────────")
    for k, v in results.items():
        print(f"  {k}: OOF = {v['rmse']:.4f}")
    print(f"  Best stack: {best_stack['kind']} OOF = {best_stack['rmse']:.4f}  |  PostProc TVT RMSE: {best_r:.4f}")
    print(sub.head(8).to_string(index=False))
    print(f"=== PIPELINE COMPLETE  total={time.time()-_T0_GLOBAL:.0f}s ===")
    
    
    print('\n\n','Model.6 finished its work and saved the result to a file:','9.537.csv','\n\n')

## Mark Cooper | 9.765 fallback branch

Legacy execution key: `Model.7`

Source: [imitation-tasmim-lgb-xgb](https://www.kaggle.com/code/markjcooper/imitation-tasmim-lgb-xgb) by [Mark Cooper](https://www.kaggle.com/markjcooper). This cell is disabled by default and kept only for explicit Nina-style reruns when precomputed CSVs do not match the active sample.


In [ ]:
if 'Model.7' in ensemble_of_solutions:
        
    print('\n\n','Model.7','\n\n')
    
    import subprocess, sys, os
    for p in ["numba"]:
        if subprocess.run([sys.executable,"-m","pip","show",p],capture_output=True).returncode!=0:
            subprocess.run([sys.executable,"-m","pip","install",p,"--quiet"])
    os.environ["NUMBA_CACHE_DIR"]="/kaggle/working/.numba"
    os.makedirs("/kaggle/working/.numba",exist_ok=True)
    
    from pathlib import Path
    from scipy.interpolate import interp1d
    from scipy.spatial import cKDTree
    from scipy.signal import savgol_filter
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    from sklearn.metrics import root_mean_squared_error
    from numba import njit
    from joblib import Parallel, delayed
    import lightgbm as lgb
    import xgboost as xgb
    import numpy as np, pandas as pd
    import gc, time, multiprocessing, warnings
    warnings.filterwarnings("ignore")
    
    SEED=42; np.random.seed(SEED)
    NCPU=min(4, multiprocessing.cpu_count())
    
    def _find():
        for p in [Path("/kaggle/input/rogii-wellbore-geology-prediction"),
                  Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")]:
            if (p/"train").exists(): return p
        raise FileNotFoundError("Data not found")
    
    DATA=_find(); TRAIN_DIR=DATA/"train"; TEST_DIR=DATA/"test"
    SAMPLE=DATA/"sample_submission.csv"; OUT=Path("/kaggle/working/submission.csv")
    FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
    PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5
    ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
    BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
    SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
    PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
    
    BEAMS=[(10,20.0,144.0,2,"cons"),(10,8.0,64.0,2,"loose"),(8,35.0,220.0,1,"vcons"),
           (10,14.0,90.0,5,"sm5"),(20,4.0,36.0,3,"vloose"),(12,12.0,100.0,3,"mid"),(15,25.0,180.0,2,"stiff")]
    
    PF_N=600; ANCC_N=600
    PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01; PF_IV=0.02; PF_IS=0.5; PF_RESAMP=0.5
    PF_RP=0.2; PF_RV=0.003; PF_GW=5; PF_GWT=0.3
    ANCC_A=0.998; ANCC_RN=0.002; ANCC_PN=0.005; ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001
    PF_GS_MIN=10.; PF_GS_MAX=60.; PF_GS_DEF=30.
    
    LGB_BASE=dict(boosting_type="gbdt",num_leaves=255,min_child_samples=15,
        subsample=0.8,subsample_freq=1,colsample_bytree=0.8,
        reg_lambda=3.0,reg_alpha=0.05,objective="regression",
        verbose=-1,n_jobs=-1,max_bin=255,force_row_wise=True,device="gpu")
    LGB_CONFIGS=[
        dict(learning_rate=0.025,n_estimators=8000,random_state=42),
        dict(learning_rate=0.020,n_estimators=8000,random_state=7),
        dict(learning_rate=0.030,n_estimators=8000,random_state=123),
    ]
    XGB_PARAMS = dict(
        n_estimators=3000,
        learning_rate=0.025,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=2.0,
        min_child_weight=10,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        device="cuda",            # Changed from "cpu" to "cuda" for GPU speedup
        early_stopping_rounds=200, # Moved from fit() to here
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
    
    print(f"CPUs={NCPU} | Train={len(list(TRAIN_DIR.glob('*__horizontal_well.csv')))} wells")
    
    # ── Numba JIT: beam ±2 + PF grid lookup ────────────────────────────────────
    @njit(cache=True)
    def _interp1(grid,v,vmin,step):
        i=int((v-vmin)/step)
        if i<0: return grid[0]
        n=len(grid)-1
        if i>=n: return grid[n]
        t=(v-vmin)/step-i
        return grid[i]*(1.-t)+grid[i+1]*t
    
    @njit(cache=True)
    def _resamp(pos,aux,w,N,rp,rv):
        cum=np.zeros(N+1)
        for j in range(N): cum[j+1]=cum[j]+w[j]
        u0=np.random.uniform(0.,1./N)
        np2=np.empty(N); na=np.empty(N); ci=0
        for j in range(N):
            u=u0+j/N
            while ci<N-1 and cum[ci+1]<u: ci+=1
            np2[j]=pos[ci]+rp*np.random.randn()
            na[j]=aux[ci]+rv*np.random.randn()
        return np2,na
    
    @njit(cache=True)
    def _beam_jit(sgr,tw_gr,si,BS,mc,es):
        n=len(sgr); nt=len(tw_gr); MAX=BS*6
        bidx=np.zeros(BS,np.int64); bidx[0]=si
        bcost=np.full(BS,1e30); bcost[0]=0.; bn=np.int64(1)
        hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
        cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
        for step in range(n):
            gv=sgr[step]; nc=np.int64(0)
            for bi in range(bn):
                idx=bidx[bi]; cost=bcost[bi]
                for d in range(-2,3):
                    ni=idx+d
                    if ni<0 or ni>=nt: continue
                    tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                    fnd=np.int64(-1)
                    for ci in range(nc):
                        if cI[ci]==ni: fnd=ci; break
                    if fnd>=0:
                        if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                    else:
                        if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
            kept=min(BS,nc)
            for i in range(kept):
                mi=i
                for j in range(i+1,nc):
                    if cC[j]<cC[mi]: mi=j
                if mi!=i:
                    cI[i],cI[mi]=cI[mi],cI[i]; cC[i],cC[mi]=cC[mi],cC[i]; cP[i],cP[mi]=cP[mi],cP[i]
            hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
            bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
        best=np.int64(0)
        for b in range(1,bn):
            if bcost[b]<bcost[best]: best=b
        path=np.zeros(n,np.int64); b=best
        for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
        return path
    
    @njit(cache=True)
    def _pf_ancc_jit(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,ALPHA,RN,PN,IS,RP,RR,RESAMP):
        pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
        for j in range(N):
            pos[j]=ls+IS*np.random.randn(); rate[j]=ir+0.01*np.random.randn()
        pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
        for i in range(len(md_v)):
            dm=max(md_v[i]-pm,1.)
            for j in range(N):
                rate[j]=ALPHA*rate[j]+RN*np.random.randn()
                pos[j]+=rate[j]*dm+PN*np.random.randn()
                tv=pos[j]-z_v[i]
                tv=max(tv,vmin-50.); tv=min(tv,vmin+len(gg)*step+50.); pos[j]=tv+z_v[i]
            if not np.isnan(gr_v[i]):
                ws=0.
                for j in range(N):
                    eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                    d=(gr_v[i]-eg)/gs; lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                    w[j]*=lk; ws+=w[j]
                if ws>0.:
                    for j in range(N): w[j]/=ws
                else:
                    for j in range(N): w[j]=1./N
            ne=0.
            for j in range(N): ne+=w[j]*w[j]
            if 1./ne<RESAMP*N:
                pos,rate=_resamp(pos,rate,w,N,RP,RR)
                for j in range(N): w[j]=1./N
            tv=0.
            for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
            pts[i]=tv; va=0.
            for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
            std_[i]=va**0.5; pm=md_v[i]
        return pts,std_
    
    @njit(cache=True)
    def _pf_z_jit(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,gs,ip,iv,beta,icpt,zsig,N,
                   MOM,VN,PN,GR_WT,RP,RV,RESAMP):
        pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
        for j in range(N):
            pos[j]=ip+0.5*np.random.randn(); vel[j]=iv+0.02*np.random.randn()
        pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
        for i in range(len(md_v)):
            dm=max(md_v[i]-pm,1.); dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
            for j in range(N):
                vel[j]=MOM*vel[j]+VN*np.random.randn()
                pos[j]+=vel[j]*dm+PN*np.random.randn()
                pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
            if not np.isnan(gr_v[i]):
                ws=0.
                for j in range(N):
                    ep=_interp1(gg_p,pos[j],vmin,step); dp=(gr_v[i]-ep)/gs
                    lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                    if not np.isnan(gr_sm_v[i]):
                        es=_interp1(gg_s,pos[j],vmin,step); ds=(gr_sm_v[i]-es)/(gs*1.5)
                        ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                        lk=(1.-GR_WT)*lp+GR_WT*ls
                    else: lk=lp
                    lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
                if ws>0.:
                    for j in range(N): w[j]/=ws
                else:
                    for j in range(N): w[j]=1./N
            ws2=0.
            for j in range(N):
                dv=(vel[j]-ve)/max(zsig*2.,0.005); lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
                w[j]*=lz; ws2+=w[j]
            if ws2>0.:
                for j in range(N): w[j]/=ws2
            else:
                for j in range(N): w[j]=1./N
            ne=0.
            for j in range(N): ne+=w[j]*w[j]
            if 1./ne<RESAMP*N:
                pos,vel=_resamp(pos,vel,w,N,RP,RV)
                for j in range(N): w[j]=1./N
            wm=0.
            for j in range(N): wm+=w[j]*pos[j]
            pts[i]=wm; va=0.
            for j in range(N): va+=w[j]*(pos[j]-wm)**2
            std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
        return pts,std_
    
    def _make_grid(tw_tvt,tw_gr,step=0.2):
        tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
        g=np.arange(tmin,tmax+step,step); return np.interp(g,tw_tvt,tw_gr).astype(np.float64),tmin,step
    
    # Warm-up JIT
    print("Compiling Numba JIT...")
    _dummy_gr=np.ones(10,np.float64); _dummy_tw=np.ones(20,np.float64)
    _dummy_gg,_dm,_ds=_make_grid(np.arange(20,dtype=np.float64),_dummy_tw)
    _beam_jit(_dummy_gr,_dummy_tw,5,5,10.,100.); _beam_jit(_dummy_gr,_dummy_tw,5,5,10.,100.)
    _pf_ancc_jit(np.ones(5),np.zeros(5),np.ones(5),_dummy_gg,_dm,_ds,30.,0.,0.,10,ANCC_A,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,0.5)
    print("Numba JIT ready ✓")
    
    # ============================================================
    # ROGII v5 – targeting ~9.0–9.5  (from 10.464)
    # ============================================================
    # NEW vs v4 (10.464):
    #  1. Numba JIT beam ±2 + both PFs (20× faster → N=600 PF free)
    #  2. Dense O(1) grid lookup for PF (tvt→gr in O(1) not O(log n))
    #  3. Segment b_well: early/mid/late/wls per formation (~0.3 pt)
    #  4. Score-weighted NCC softmax ensemble (~0.2 pt)
    #  5. Hybrid reference = (1-trust)*beam + trust*sc_ens
    #  6. Additional offset probes relative to beam/NCC/PF signals
    #  7. XGBoost as 2nd model + Ridge stacking (~0.3 pt)
    #  8. Post-processing grid: alpha×tau×w_pf + SG smoothing (~0.4 pt)
    # ============================================================
    
    # ── High-level wrappers ─────────────────────────────────────────────────────
    def beam_search(hgr,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
        sgr=pd.Series(hgr.astype(np.float64)).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
        if r>0: sgr=sgr.rolling(r*2+1,center=True,min_periods=1).mean()
        sgr=sgr.to_numpy(np.float64)
        si=int(np.searchsorted(tw_tvt,start_tvt))
        si=max(0,min(si,len(tw_tvt)-1))
        path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,mc,es)
        return tw_tvt[path.clip(0,len(tw_tvt)-1)].astype(np.float32)
    
    def _pf_prep(hw):
        k=hw[hw['TVT_input'].notna()]; kg=k[k['GR'].notna()]
        gs=PF_GS_DEF
        if len(kg)>=20:
            gs_val=np.std(kg['GR'].values-np.interp(kg['TVT_input'].values,
                   tw_tvt_global,tw_gr_global))
            gs=float(np.clip(gs_val,PF_GS_MIN,PF_GS_MAX))
        return gs
    
    def run_pf_ancc(hw,tw_tvt,tw_gr):
        gg,vmin,step=_make_grid(tw_tvt,tw_gr)
        k=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        k2=k[k['GR'].notna()]
        gs=PF_GS_DEF
        if len(k2)>=20:
            gs=float(np.clip(np.std(k2['GR'].values-np.interp(k2['TVT_input'].values,tw_tvt,tw_gr)),PF_GS_MIN,PF_GS_MAX))
        ls=float(k['TVT_input'].iloc[-1])+float(k['Z'].iloc[-1])
        t=k.tail(30); ir=0.
        if len(t)>=10:
            dt=np.diff(t['TVT_input'].values); dz=np.diff(t['Z'].values); dm=np.diff(t['MD'].values); m=dm>0
            if m.sum()>=3: ir=float(np.median((dt[m]+dz[m])/dm[m]))
        gr_v=ev['GR'].to_numpy(np.float64)
        pts,std_=_pf_ancc_jit(ev['MD'].to_numpy(np.float64),ev['Z'].to_numpy(np.float64),gr_v,
                               gg,vmin,step,gs,ls,ir,ANCC_N,ANCC_A,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,0.5)
        return pts,std_
    
    def run_pf_z(hw,tw_tvt,tw_gr):
        gg_p,vmin,step=_make_grid(tw_tvt,tw_gr)
        tw_sm=pd.Series(tw_gr).rolling(PF_GW,center=True,min_periods=1).mean().values
        gg_s,_,_=_make_grid(tw_tvt,tw_sm)
        k=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        k2=k[k['GR'].notna()]
        gs=PF_GS_DEF
        if len(k2)>=20:
            gs=float(np.clip(np.std(k2['GR'].values-np.interp(k2['TVT_input'].values,tw_tvt,tw_gr)),PF_GS_MIN,PF_GS_MAX))
        ktvt=k['TVT_input'].values; kmd=k['MD'].values; kz=k['Z'].values
        dz=np.diff(kz); dtvt=np.diff(ktvt); dmd_=np.diff(kmd); m=dmd_>0
        beta,intc,zsig=-1.,0.,0.1
        if m.sum()>=10:
            vz=dz[m]/dmd_[m]; vt=dtvt[m]/dmd_[m]
            c,_,_,_=np.linalg.lstsq(np.column_stack([vz,np.ones_like(vz)]),vt,rcond=None)
            beta,intc=float(c[0]),float(c[1]); zsig=max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
        iv=0.
        if len(k)>=10:
            t=k.tail(20); dt2=np.diff(t['TVT_input'].values); dm2=np.diff(t['MD'].values); m2=dm2>0
            if m2.sum()>=3: iv=float(np.median(dt2[m2]/dm2[m2]))
        ip=float(k['TVT_input'].iloc[-1])
        gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
        hw_gsm=gr_full.rolling(PF_GW,center=True,min_periods=1).mean()
        gr_v=ev['GR'].to_numpy(np.float64)
        gr_sm_v=hw_gsm.iloc[ev.index].to_numpy(np.float64)
        pts,std_=_pf_z_jit(ev['MD'].to_numpy(np.float64),ev['Z'].to_numpy(np.float64),gr_v,gr_sm_v,
                            gg_p,gg_s,vmin,step,gs,ip,iv,beta,intc,zsig,PF_N,
                            PF_MOM,PF_VN,PF_PN,PF_GWT,PF_RP,PF_RV,0.5)
        return pts,std_
    
    # ── Formation imputers ──────────────────────────────────────────────────────
    class FormationPlaneKNN:
        def __init__(self,wids,data_dir):
            rows=[]
            for wid in wids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
                except: continue
                if len(df)==0: continue
                row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
                for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
                rows.append(row)
            self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
            xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
            self.tree=cKDTree(xy/self.scale)
            self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
            self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)
        def impute(self,xy_q,self_wid=None,k=PLANE_K):
            q=xy_q/self.scale; nf=min(k+5,len(self.df))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
            ord_=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord_,1); ik=np.take_along_axis(idx,ord_,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
            xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
            A=np.zeros((len(q),3,3))
            A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
            A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
            A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
            A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
            rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
            try: coef=np.linalg.solve(A,rhs)
            except:
                coef=np.zeros((len(q),3,6))
                for r in range(len(q)):
                    try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                    except: pass
            Xq=xy_q[:,0]; Yq=xy_q[:,1]
            pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
            pred[~vk.any(1)]=self.fa.mean(0)
            return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)
    
    class DenseANCCImputer:
        def __init__(self,wids,data_dir,spw=DENSE_SPW):
            xs,ys,anccs,wids_=[],[],[],[]
            for wid in wids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
                except: continue
                if len(df)==0: continue
                ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
                xs.append(s['X'].values); ys.append(s['Y'].values)
                anccs.append(s['ANCC'].values); wids_.extend([wid]*len(s))
            self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
            self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids_)
            self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
            self.tree=cKDTree(self.xy/self.scale)
        def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
            xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
            ord_=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord_,1); ik=np.take_along_axis(idx,ord_,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
            sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
            ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
            var=((an-ap[:,None])**2*w).sum(1)/safe
            return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)
    
    hw_paths=sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
    train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
    print(f"Building imputers ({len(train_wids)} wells)..."); t0=time.time()
    FI=FormationPlaneKNN(train_wids,TRAIN_DIR)
    DI=DenseANCCImputer(train_wids,TRAIN_DIR)
    print(f"  FPK:{len(FI.df)} | Dense:{len(DI.ancc):,}  ({time.time()-t0:.0f}s)")
    
    # ── Helpers ─────────────────────────────────────────────────────────────────
    def rmse(a,b): return float(np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2)))
    def robust_slope(x,y):
        x=np.asarray(x,float); y=np.asarray(y,float); m=np.isfinite(x)&np.isfinite(y)
        if m.sum()<2 or np.std(x[m])<1e-6: return 0.
        return float(np.polyfit(x[m],y[m],1)[0])
    def affine_cal(kgr,tw_at_k,min_pts=20):
        v=np.isfinite(kgr)&np.isfinite(tw_at_k)
        if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6: return 1.,float(np.nanmean(kgr[v])-np.nanmean(tw_at_k[v])) if v.any() else 0.
        a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)
    
    def seg_b_well(ktvt,kz,form_col):
        """5 calibration constants: full, early-third, mid-third, late-50, WLS-tail-upweighted."""
        bv=ktvt+kz-form_col; n=len(bv); b_full=float(np.median(bv))
        b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
        t1,t2=n//3,2*n//3
        b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
        b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
        w=np.exp(0.02*np.arange(n)); w/=w.sum(); b_wls=float(np.dot(w,bv))
        return b_full,b_early,b_mid,b_late,b_wls
    
    def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
        out=[]
        for hw_sc in hws:
            win=2*hw_sc+1; nk=len(kgr); nh=len(hgr)
            if nk<win+1 or nh==0:
                out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
            kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
            hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
            sts=np.arange(0,nk-win+1,stride,dtype=np.int32)
            if len(sts)==0:
                out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
            C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
            Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
            hp=np.pad(hg,hw_sc,mode='edge')
            H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
            Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
            ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
            out.append((ktvt[np.clip(sts[best]+hw_sc,0,nk-1)].astype(np.float32),score))
        tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
        sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
        sc_ens=(tvts*sw).sum(1).astype(np.float32)
        return out,sc_ens
    
    # ── Feature builder ──────────────────────────────────────────────────────────
    _FI=FI; _DI=DI
    
    def build_well(hw_path,tw_path,is_train):
        global _FI,_DI
        wid=Path(hw_path).stem.replace('__horizontal_well','')
        try: hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
        except: return None
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0 or len(kn)<10: return None
        if is_train and ('TVT' not in hw.columns or hw['TVT'].isna().all()): return None
        tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
        if len(tw_tvt)<3: return None
    
        np.random.seed(SEED)
        pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
        if len(pf_a)==0: return None
        pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
        pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)
        has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))
    
        lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
        gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
        hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
        kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)
    
        # 7 beams (JIT ±2)
        bpaths={}
        for (bs,mc,es,r,tag) in BEAMS:
            bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
        beam_ref=(bpaths['cons']+bpaths['sm5'])/2.
    
        # Multi-scale NCC → score-weighted ensemble
        ktvt=kn['TVT_input'].to_numpy(np.float32)
        sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
        sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
        sc_cons=(sc8+sc15+sc25)/3.
        sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
        hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens
    
        tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
        a_cal,b_cal=affine_cal(kgr,tw_at_k)
        kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
        pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
        slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
        slp_z=robust_slope(kn['Z'].to_numpy(),ktvt)
    
        swid=wid if is_train else None
        xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
        form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
        form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
        z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)
    
        # Segment b_well per formation
        tvt_fs={}; form_rmse={}; form_list=[]
        for fi2,fn in enumerate(FORMATIONS):
            b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
            tvt_fs[f'tvtF_{fn}'] =(-z_ev+form_ev[:,fi2]+b_full).astype(np.float32)
            tvt_fs[f'tvtFw_{fn}']=(-z_ev+form_ev[:,fi2]+b_wls ).astype(np.float32)
            tvt_fs[f'tvtF50_{fn}']=(-z_ev+form_ev[:,fi2]+b_late).astype(np.float32)
            tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
            tvt_fs[f'bw50_{fn}']=np.float32(b_late)
            tvt_fs[f'bw_early_{fn}']=np.float32(b_early)  # ← NEW
            tvt_fs[f'bw_mid_{fn}'  ]=np.float32(b_mid)    # ← NEW
            form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
            form_list.append(tvt_fs[f'tvtF_{fn}'])
    
        fs=np.stack(form_list,1)
        form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
        form_std_d=fs.std(1).astype(np.float32); form_rng_d=(fs.max(1)-fs.min(1)).astype(np.float32)
    
        d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
        d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
        b_vd=ktvt+z_kn-d_kn; _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)
        b_d=float(np.median(b_vd))
        tvt_dense  =(-z_ev+d_ancc+b_d ).astype(np.float32)
        tvt_densew =(-z_ev+d_ancc+b_dw).astype(np.float32)
        tvt_dense50=(-z_ev+d_ancc+b_dl).astype(np.float32)
        d_rmse=float(np.sqrt(np.mean((ktvt+z_kn-d_kn-b_d)**2)))
        d_bias=float(np.mean(b_vd-b_d))
    
        all_sigs=[pf_use]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
        sig_mat=np.stack(all_sigs,1); sig_std=sig_mat.std(1).astype(np.float32)
        sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)
    
        gr_s=pd.Series(gr_full.values); rolls={}
        for w in [5,21,51,101]:
            r=gr_s.rolling(w,center=True,min_periods=1)
            rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
            rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
        for lag in [1,5,15,30]:
            rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
            rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
        gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
        gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
        gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
        gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)).iloc[ev.index].values.astype(np.float32)
    
        hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
        slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
        slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)
        mdd=hw['MD'].diff().replace(0,np.nan)
        dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
        dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
        dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
        nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
        def sc(v): return np.full(nh,np.float32(v),np.float32)
    
        feats={
            'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
            'last_known_tvt':sc(last_tvt),
            'pf_ancc':pf_use,'pf_ancc_std':std_use,
            'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
            'pf_z':(pf_z.astype(np.float32) if has_z else sc(last_tvt)),
            'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
            'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
            **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
            'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
            'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
            'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
            'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
            'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
            'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
            'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
            'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),
            'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
            'sig_std':sig_std,'sig_mean_d':sig_mean,
            **tvt_fs,
            **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
            'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
            'spatial_knn_dist':knn_d,
            'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
            'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
            'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
            'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
            'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),
            'pf_vs_spatial':(pf_use-tvt_fs['tvtF_ANCC']).astype(np.float32),
            'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
            'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
            'beam_vs_spatial':(bpaths['cons']-tvt_fs['tvtF_ANCC']).astype(np.float32),
            'sc_vs_beam':(sc_ens-bpaths['cons']).astype(np.float32),
            'cal_a':sc(a_cal),'cal_b':sc(b_cal),
            'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
            'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
            'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
            'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),
            'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
            'md_since':md_since,'frac':frac,'frac2':frac**2,
            'ease_frac':(3*frac**2-2*frac**3).astype(np.float32),
            'z':z_ev,'x':ev['X'].to_numpy(np.float32),'y':ev['Y'].to_numpy(np.float32),
            **rolls,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
            'gr_minus_tw_last':(gr_full.iloc[ev.index].values.astype(np.float32)-float(np.interp(last_tvt,tw_tvt,tw_gr))).astype(np.float32),
            'anchor_t_pos':sc(float((last_tvt-float(tw_tvt.min()))/max(float(tw_tvt.max()-tw_tvt.min()),1e-3))),
            'tw_tvt_range':sc(float(tw_tvt.max()-tw_tvt.min())),
            'tw_gr_mean':sc(float(tw_gr.mean())),'tw_gr_std':sc(float(tw_gr.std())),
        }
        hgr_fill=gr_full.iloc[ev.index].values.astype(np.float32)
        # Offset probes: anchor, beam, NCC, PF
        for o in ANCH_OFFS:
            feats[f'anch_diff_{int(o)}']=hgr_fill-float(np.interp(last_tvt+float(o),tw_tvt,tw_gr))
        for o in BEAM_OFFS:
            feats[f'beam_diff_{int(o)}']=hgr_fill-np.interp(bpaths['cons']+float(o),tw_tvt,tw_gr).astype(np.float32)
        for o in SC_OFFS:
            feats[f'sc_diff_{int(o)}']=hgr_fill-np.interp(sc_ens+float(o),tw_tvt,tw_gr).astype(np.float32)
        for o in PF_OFFS:
            feats[f'pf_diff_{int(o)}']=hgr_fill-np.interp(pf_use+float(o),tw_tvt,tw_gr).astype(np.float32)
        if is_train:
            feats['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
        df=pd.DataFrame(feats)
        for c in df.select_dtypes('float64').columns: df[c]=df[c].astype(np.float32)
        return df
    
    def build_dataset(paths,is_train,label):
        tw_dir=TRAIN_DIR if is_train else TEST_DIR
        print(f"  {label}: {len(paths)} wells | {NCPU} threads")
        results=Parallel(n_jobs=NCPU,backend='threading',verbose=5)(
            delayed(build_well)(str(p),str(tw_dir/p.name.replace('__horizontal_well.csv','__typewell.csv')),is_train)
            for p in paths)
        ok=[r for r in results if r is not None]
        print(f"  {label}: OK={len(ok)} skipped={len(paths)-len(ok)} | {time.time()-t0:.0f}s")
        return pd.concat(ok,ignore_index=True)
    
    print("Building train..."); t0=time.time()
    train_df=build_dataset(hw_paths,is_train=True,label="train")
    print(f"train: {train_df.shape}  {time.time()-t0:.0f}s")
    test_paths=sorted(TEST_DIR.glob('*__horizontal_well.csv'))
    print("Building test...")
    test_df=build_dataset(test_paths,is_train=False,label="test")
    print(f"test: {test_df.shape}")
    
    SKIP={'well','id','target'}
    feature_cols=[c for c in train_df.columns if c not in SKIP]
    print(f"#features: {len(feature_cols)}")
    X=train_df[feature_cols].astype(np.float32); y=train_df['target']; g=train_df['well']
    Xt=test_df[feature_cols].astype(np.float32); gc.collect()
    
    # ── GroupKFold training ─────────────────────────────────────────────────────
    cv=GroupKFold(n_splits=N_SPLITS); splits=list(cv.split(X,y,g))
    
    def run_lgb(cfg_idx):
        cfg=LGB_CONFIGS[cfg_idx]; p=dict(LGB_BASE,**cfg); n_est=p.pop('n_estimators')
        oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
        for fold,(tr,va) in enumerate(splits):
            dtr=lgb.Dataset(X.iloc[tr],label=y.iloc[tr])
            dva=lgb.Dataset(X.iloc[va],label=y.iloc[va],reference=dtr)
            m=lgb.train(p,dtr,valid_sets=[dva],num_boost_round=n_est,
                        callbacks=[lgb.early_stopping(250,verbose=False),lgb.log_evaluation(800)])
            oof[va]=m.predict(X.iloc[va],num_iteration=m.best_iteration).astype(np.float32)
            tp+=m.predict(Xt,num_iteration=m.best_iteration).astype(np.float32)/N_SPLITS
            print(f"  LGB{cfg_idx} f{fold}: {root_mean_squared_error(y.iloc[va],oof[va]):.4f} iter={m.best_iteration}")
        r=root_mean_squared_error(y,oof); print(f"  LGB{cfg_idx} OOF={r:.4f}"); return oof,tp,r
    
    def run_xgb():
        oof = np.zeros(len(train_df), np.float32)
        tp = np.zeros(len(test_df), np.float32)
        
        for fold, (tr, va) in enumerate(splits):
            # The model now knows about early stopping from XGB_PARAMS
            m = xgb.XGBRegressor(**XGB_PARAMS)
            
            # Removed early_stopping_rounds from fit()
            m.fit(
                X.iloc[tr].values, y.iloc[tr].values,
                eval_set=[(X.iloc[va].values, y.iloc[va].values)],
                verbose=500
            )
            
            # Use m.best_iteration to get the best results from early stopping
            oof[va] = m.predict(X.iloc[va].values, iteration_range=(0, m.best_iteration)).astype(np.float32)
            tp += m.predict(Xt.values, iteration_range=(0, m.best_iteration)).astype(np.float32) / N_SPLITS
            
            print(f"  XGB f{fold}: {root_mean_squared_error(y.iloc[va], oof[va]):.4f}")
            
        r = root_mean_squared_error(y, oof)
        print(f"  XGB OOF={r:.4f}")
        return oof, tp, r
    
    results={}
    for i in range(3):
        oof,tp,r=run_lgb(i); results[f'lgb{i}']={'oof':oof,'test':tp,'rmse':r}
    oof,tp,r=run_xgb(); results['xgb']={'oof':oof,'test':tp,'rmse':r}
    
    Sx=np.column_stack([v['oof'] for v in results.values()])
    St=np.column_stack([v['test'] for v in results.values()])
    ridge=Ridge(alpha=1.,fit_intercept=False,positive=True); ridge.fit(Sx,y.values)
    oof_s=ridge.predict(Sx); test_s=ridge.predict(St)
    r_avg=root_mean_squared_error(y,Sx.mean(1)); r_stk=root_mean_squared_error(y,oof_s)
    wts=ridge.coef_/max(ridge.coef_.sum(),1e-9)
    print(f"\nAvg:{r_avg:.4f} Ridge:{r_stk:.4f} wts={dict(zip(results.keys(),wts.round(4)))}")
    final_oof =oof_s if r_stk<r_avg else Sx.mean(1)
    final_test=test_s if r_stk<r_avg else St.mean(1)
    
    # ── Post-processing: alpha × tau × w_pf grid + SG smoothing ────────────────
    base=train_df['last_known_tvt'].values; ytrue=y.values+base
    pf_oof=(train_df['pf_ancc'].values-base)
    
    print("\nGrid search alpha×tau×w_pf...")
    best_cfg,best_r=(None,None,None),np.inf
    for alpha in np.arange(0.65,1.01,0.05):
        for tau in [None,25.,50.,100.,200.,350.]:
            for w_pf in [0.0,0.05,0.10,0.15]:
                d=final_oof*(1-w_pf)+pf_oof*w_pf
                if tau: d*=(1.-np.exp(-np.maximum(train_df['md_since'].values,0.)/tau))
                d*=alpha
                r=root_mean_squared_error(ytrue,base+d)
                if r<best_r: best_r,best_cfg=r,(alpha,tau,w_pf)
    ALPHA,TAU,W_PF=best_cfg
    print(f"Best: alpha={ALPHA:.2f} tau={TAU} w_pf={W_PF:.2f} | abs TVT RMSE={best_r:.4f}")
    
    def apply_pp(df,md,pd_,alpha,tau,w_pf):
        d=md*(1-w_pf)+pd_*w_pf
        if tau: d*=(1.-np.exp(-np.maximum(df['md_since'].values,0.)/tau))
        return d*alpha
    
    def sg_smooth(df,col,sg_w=17,sg_p=3):
        df=df.copy()
        for well,grp in df.groupby('well',sort=False):
            v=grp[col].values; n=len(v); wl=min(sg_w,n)
            if wl%2==0: wl-=1
            if wl>=sg_p+2: v=savgol_filter(v,wl,sg_p)
            df.loc[grp.index,col]=v
        return df
    
    test_df2=test_df.copy()
    pf_test=(test_df2['pf_ancc'].values-test_df2['last_known_tvt'].values)
    test_df2['pred']=(test_df2['last_known_tvt'].values+apply_pp(test_df2,final_test,pf_test,ALPHA,TAU,W_PF))
    test_df2=sg_smooth(test_df2,'pred')
    
    sample=pd.read_csv(SAMPLE)
    sub=(sample[['id']].merge(test_df2[['id','pred']].rename(columns={'pred':'tvt'}),on='id',how='left'))
    fb=float(train_df['last_known_tvt'].mean()+train_df['target'].mean())
    sub['tvt']=sub['tvt'].fillna(fb)
    sub[['id','tvt']].to_csv('9.765.csv',index=False)
    print(f"\n✅ {OUT}  {len(sub)} rows")
    print("\n─── Summary ──────────────────────────────────────")
    for k,v in results.items(): print(f"  {k}: OOF={v['rmse']:.4f}")
    print(f"  Stack: {min(r_avg,r_stk):.4f}")
    print(f"  PostProc: {best_r:.4f}")
    print(sub.head(5).to_string(index=False))
    
    print('\n\n','Model.7 finished its work and saved the result to a file:','9.765.csv','\n\n')

## Roman Tamrazov | 9.956 fallback branch

Legacy execution key: `Model.4`

Source: [[ROGII] BETTER SOLUTION . LB: 9.956](https://www.kaggle.com/code/romantamrazov/rogii-better-solution-lb-9-956) by [Roman Tamrazov](https://www.kaggle.com/romantamrazov). This cell is disabled by default and kept only for explicit Nina-style reruns when precomputed CSVs do not match the active sample.


In [ ]:
if 'Model.4' in ensemble_of_solutions:
    
    print('\n\n','Model.4','\n\n')
    
    import subprocess,sys,os
    for p in ["numba"]:
        if subprocess.run([sys.executable,"-m","pip","show",p],capture_output=True).returncode!=0:
            subprocess.run([sys.executable,"-m","pip","install",p,"--quiet"])
    os.environ["NUMBA_CACHE_DIR"]="/kaggle/working/.numba"
    os.makedirs("/kaggle/working/.numba",exist_ok=True)
    print("ok")
    
    from pathlib import Path
    from scipy.interpolate import interp1d
    from scipy.spatial import cKDTree
    from scipy.signal import savgol_filter
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    from sklearn.metrics import root_mean_squared_error
    from catboost import CatBoostRegressor, Pool
    from numba import njit
    from joblib import Parallel, delayed
    import lightgbm as lgb
    import numpy as np, pandas as pd
    import gc, time, multiprocessing, warnings
    warnings.filterwarnings("ignore")
    
    SEED=42; np.random.seed(SEED)
    NCPU=min(4,multiprocessing.cpu_count())
    
    def _find():
        for p in [Path("/kaggle/input/rogii-wellbore-geology-prediction"),
                   Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")]:
            if (p/"train").exists(): return p
        for p in Path("/kaggle/input").glob("*/sample_submission.csv"): return p.parent
        raise FileNotFoundError("Data not found")
    
    DATA=_find(); TRAIN_DIR=DATA/"train"; TEST_DIR=DATA/"test"
    SAMPLE=DATA/"sample_submission.csv"; OUT=Path("/kaggle/working/submission.csv")
    
    FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
    PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5
    
    # 7 beam configs
    BEAMS=[
        (10,20.0,144.0,2,"cons"),
        (10, 8.0, 64.0,2,"loose"),
        ( 8,35.0,220.0,1,"vcons"),
        (10,14.0, 90.0,5,"sm5"),
        (20, 4.0, 36.0,3,"vloose"),
        (12,12.0,100.0,3,"mid"),
        (15,25.0,180.0,2,"stiff"),
    ]
    
    # PF params — N=600 (affordable with Numba JIT)
    PF_N=600; ANCC_N=600
    PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01
    PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
    PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
    PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
    ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
    ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001
    
    # Tuned model params
    LGB_BASE=dict(
        boosting_type="gbdt", num_leaves=255, min_child_samples=15,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        reg_lambda=3.0, reg_alpha=0.05, objective="regression",
        verbose=-1, n_jobs=-1, device_type="gpu", gpu_use_dp=False, max_bin=255,
    )
    LGB_CONFIGS=[
        dict(learning_rate=0.025, n_estimators=8000, seed=42),
        dict(learning_rate=0.020, n_estimators=8000, seed=7),
        dict(learning_rate=0.030, n_estimators=8000, seed=123),
    ]
    CB_P=dict(
        iterations=8000, learning_rate=0.025, depth=7, l2_leaf_reg=2.0,
        min_data_in_leaf=15, border_count=254,
        loss_function="RMSE", random_seed=42, task_type="GPU", devices="0:1",
        od_type="Iter", od_wait=300, verbose=0,
    )
    
    import subprocess as _s
    print("GPUs:",_s.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
          capture_output=True,text=True).stdout.strip())
    print(f"CPUs={NCPU} | train={len(list(TRAIN_DIR.glob('*__horizontal_well.csv')))} wells")
    
    
    # ── Numba JIT: Beam Search ±2 + Both Particle Filters ─────────────
    # Beam JIT: ±2 delta (TVT can decrease), no GIL, cached to disk
    # PF JIT: O(1) dense-grid lookup, systematic resampling
    
    @njit(cache=True)
    def _interp1(grid, v, vmin, step):
        i = int((v - vmin) / step)
        if i < 0: return grid[0]
        n = len(grid) - 1
        if i >= n: return grid[n]
        t = (v - vmin) / step - i
        return grid[i]*(1.-t) + grid[i+1]*t
    
    @njit(cache=True)
    def _resamp(pos, aux, w, N, rp, rv):
        cum = np.zeros(N+1)
        for j in range(N): cum[j+1]=cum[j]+w[j]
        u0=np.random.uniform(0.,1./N)
        np2=np.empty(N); na=np.empty(N); ci=0
        for j in range(N):
            u=u0+j/N
            while ci<N-1 and cum[ci+1]<u: ci+=1
            np2[j]=pos[ci]+rp*np.random.randn()
            na[j] =aux[ci]+rv*np.random.randn()
        return np2,na
    
    @njit(cache=True)
    def _beam_jit(sgr, tw_gr, si, BS, mc, es):
        """Beam search ±2 delta, Numba JIT."""
        n=len(sgr); nt=len(tw_gr); MAX=BS*6
        bidx=np.zeros(BS,np.int64); bidx[0]=si
        bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
        hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
        cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
        for step in range(n):
            gv=sgr[step]; nc=np.int64(0)
            for bi in range(bn):
                idx=bidx[bi]; cost=bcost[bi]
                for d in range(-2,3):            # ±2: TVT can go down
                    ni=idx+d
                    if ni<0 or ni>=nt: continue
                    tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                    fnd=np.int64(-1)
                    for ci in range(nc):
                        if cI[ci]==ni: fnd=ci; break
                    if fnd>=0:
                        if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                    else:
                        if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
            kept=min(BS,nc)
            for i in range(kept):
                mi=i
                for j in range(i+1,nc):
                    if cC[j]<cC[mi]: mi=j
                if mi!=i:
                    cI[i],cI[mi]=cI[mi],cI[i]
                    cC[i],cC[mi]=cC[mi],cC[i]
                    cP[i],cP[mi]=cP[mi],cP[i]
            hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
            bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
        best=np.int64(0)
        for b in range(1,bn):
            if bcost[b]<bcost[best]: best=b
        path=np.zeros(n,np.int64); b=best
        for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
        return path
    
    @njit(cache=True)
    def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
                  ALPHA,RN,PN,IS,RP,RR,RESAMP):
        pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
        for j in range(N):
            pos[j]=ls+IS*np.random.randn()
            rate[j]=ir+0.01*np.random.randn()
        pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
        for i in range(len(md_v)):
            dm=md_v[i]-pm; dm=max(dm,1.)
            for j in range(N):
                rate[j]=ALPHA*rate[j]+RN*np.random.randn()
                pos[j]+=rate[j]*dm+PN*np.random.randn()
                tvt_j=pos[j]-z_v[i]
                tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
                pos[j]=tvt_j+z_v[i]
            if not np.isnan(gr_v[i]):
                ws=0.
                for j in range(N):
                    eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                    d=(gr_v[i]-eg)/gs
                    lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                    w[j]*=lk; ws+=w[j]
                if ws>0.:
                    for j in range(N): w[j]/=ws
                else:
                    for j in range(N): w[j]=1./N
            ne=0.
            for j in range(N): ne+=w[j]*w[j]
            if 1./ne<RESAMP*N:
                pos,rate=_resamp(pos,rate,w,N,RP,RR)
                for j in range(N): w[j]=1./N
            tv=0.
            for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
            pts[i]=tv; va=0.
            for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
            std_[i]=va**0.5; pm=md_v[i]
        return pts,std_
    
    @njit(cache=True)
    def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
              gs,ip,iv,beta,icpt,zsig,N,
              MOM,VN,PN,GR_WT,RP,RV,RESAMP):
        pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
        for j in range(N):
            pos[j]=ip+0.5*np.random.randn()
            vel[j]=iv+0.02*np.random.randn()
        pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
        for i in range(len(md_v)):
            dm=md_v[i]-pm; dm=max(dm,1.)
            dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
            for j in range(N):
                vel[j]=MOM*vel[j]+VN*np.random.randn()
                pos[j]+=vel[j]*dm+PN*np.random.randn()
                pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
            if not np.isnan(gr_v[i]):
                ws=0.
                for j in range(N):
                    ep=_interp1(gg_p,pos[j],vmin,step)
                    dp=(gr_v[i]-ep)/gs
                    lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                    if not np.isnan(gr_sm_v[i]):
                        es=_interp1(gg_s,pos[j],vmin,step)
                        ds=(gr_sm_v[i]-es)/(gs*1.5)
                        ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                        lk=(1.-GR_WT)*lp+GR_WT*ls
                    else: lk=lp
                    lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
                if ws>0.:
                    for j in range(N): w[j]/=ws
                else:
                    for j in range(N): w[j]=1./N
            ws2=0.
            for j in range(N):
                dv=(vel[j]-ve)/max(zsig*2.,0.005)
                lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
                w[j]*=lz; ws2+=w[j]
            if ws2>0.:
                for j in range(N): w[j]/=ws2
            else:
                for j in range(N): w[j]=1./N
            ne=0.
            for j in range(N): ne+=w[j]*w[j]
            if 1./ne<RESAMP*N:
                pos,vel=_resamp(pos,vel,w,N,RP,RV)
                for j in range(N): w[j]=1./N
            wm=0.
            for j in range(N): wm+=w[j]*pos[j]
            pts[i]=wm; va=0.
            for j in range(N): va+=w[j]*(pos[j]-wm)**2
            std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
        return pts,std_
    
    # Dense grid for O(1) typewell lookup
    def _grid(tw_tvt,tw_gr,step=0.2):
        tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
        tvt_g=np.arange(tmin,tmax+step,step)
        return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)
    
    def _gr_sig(hw,tw_tvt,tw_gr):
        kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]
        if len(kn)<20: return float(PF_GR_SIG_DEF)
        return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),
                              PF_GR_SIG_MIN,PF_GR_SIG_MAX))
    
    def _nn(arr,v):
        i=int(np.searchsorted(arr,v,'left'))
        if i>=len(arr): return len(arr)-1
        if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
        return i
    
    def _smooth(vals,fb,r):
        s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
        return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)
    
    def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
        si=_nn(tw_tvt,start_tvt)
        sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
        path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
        return tw_tvt[path].astype(np.float32)
    
    def run_pf_ancc(hw,tw_tvt,tw_gr,N=ANCC_N):
        gs=_gr_sig(hw,tw_tvt,tw_gr)
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
        tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
        dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
        ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
        gg,gmin,gst=_grid(tw_tvt,tw_gr)
        pts,std=_pf_ancc(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                          ev['GR'].values.astype(np.float64),gg,gmin,gst,
                          gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP)
        return pts.astype(np.float32),std.astype(np.float32)
    
    def run_pf_z(hw,tw_tvt,tw_gr,N=PF_N):
        gs=_gr_sig(hw,tw_tvt,tw_gr)
        tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values.astype(np.float32)
        kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
        dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
        if m2.sum()>=10:
            vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
            A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
            beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
        else: beta,icpt,zsig=-1.,0.,0.1
        t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
        iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
        gg,gmin,gst=_grid(tw_tvt,tw_gr)
        gs2,_,_=_grid(tw_tvt,tw_s)
        gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
        pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                       ev['GR'].values.astype(np.float64),
                       gr_sm.loc[ev.index].values.astype(np.float64),
                       gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                       beta,icpt,zsig,N,
                       PF_MOM,PF_VN,PF_PN,PF_GR_WT,PF_ROUGH_P,PF_ROUGH_V,PF_RESAMP)
        return pts.astype(np.float32),std.astype(np.float32)
    
    # One-time compile
    print("Compiling Numba JIT...")
    _md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)
    _gg=np.linspace(45,55,100,np.float64)
    _pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,0.998,0.002,0.005,0.3,0.1,0.001,0.5)
    _pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,0.993,0.005,0.01,0.3,0.2,0.003,0.5)
    _beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)
    print("Numba JIT ready ✓")
    
    
    def robust_slope(x,y,w=None):
        x=np.asarray(x,float); y=np.asarray(y,float)
        m=np.isfinite(x)&np.isfinite(y)
        if m.sum()<2 or np.std(x[m])<1e-6: return 0.
        return float(np.polyfit(x[m],y[m],1)[0])
    
    def affine_cal(kgr,tw_at_k,min_pts=20):
        v=np.isfinite(kgr)&np.isfinite(tw_at_k)
        if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
            return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
        a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)
    
    def seg_b_well(ktvt,kz,form_col):
        """Segment b_well: early/mid/late thirds + full prefix.
        Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness."""
        bv=ktvt+kz-form_col; n=len(bv)
        b_full=float(np.median(bv))
        b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
        t1,t2=n//3, 2*n//3
        b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
        b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
        # WLS (tail-upweighted)
        w=np.exp(0.02*np.arange(n)); w/=w.sum()
        b_wls=float(np.dot(w,bv))
        return b_full,b_early,b_mid,b_late,b_wls
    
    def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
        """Multi-scale NCC. Returns score-weighted ensemble + per-scale signals."""
        out=[]
        for hw in hws:
            win=2*hw+1; nk=len(kgr); nh=len(hgr)
            if nk<win+1 or nh==0:
                out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
            kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
            hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
            sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
            if M==0:
                out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
            C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
            Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
            hp=np.pad(hg,hw,mode='edge')
            H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
            Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
            ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
            out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
        # Score-weighted ensemble (NEW: softmax-weighted combination)
        tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
        sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
        sc_ens=(tvts*sw).sum(1).astype(np.float32)
        return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble
    
    print("Helpers OK ✓")
    
    
    class FormationPlaneKNN:
        def __init__(self,well_ids,data_dir):
            rows=[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
                except: continue
                if len(df)==0: continue
                row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
                for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
                rows.append(row)
            self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
            xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
            self.tree=cKDTree(xy/self.scale)
            self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
            self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)
    
        def impute(self,xy_q,self_wid=None,k=PLANE_K):
            q=xy_q/self.scale; nf=min(k+5,len(self.df))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
            xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
            A=np.zeros((len(q),3,3))
            A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
            A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
            A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
            A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
            rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
            try: coef=np.linalg.solve(A,rhs)
            except:
                coef=np.zeros((len(q),3,6))
                for r in range(len(q)):
                    try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                    except: pass
            Xq=xy_q[:,0]; Yq=xy_q[:,1]
            pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
            pred[~vk.any(1)]=self.fa.mean(0)
            return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)
    
    class DenseANCCImputer:
        def __init__(self,well_ids,data_dir,spw=DENSE_SPW):
            xs,ys,anccs,wids=[],[],[],[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
                except: continue
                if len(df)==0: continue
                ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
                xs.append(s['X'].values); ys.append(s['Y'].values)
                anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
            self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
            self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)
            self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
            self.tree=cKDTree(self.xy/self.scale)
    
        def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
            xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
            sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
            ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
            var=((an-ap[:,None])**2*w).sum(1)/safe
            return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)
    
    hw_paths=sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
    train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
    print(f"Building imputers ({len(train_wids)} wells)..."); t0=time.time()
    FI=FormationPlaneKNN(train_wids,TRAIN_DIR)
    DI=DenseANCCImputer(train_wids,TRAIN_DIR)
    print(f"  FPK:{len(FI.df)} | Dense:{len(DI.ancc):,}  ({time.time()-t0:.0f}s)")
    
    
    _FI=FI; _DI=DI
    ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
    BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
    SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
    PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
    
    def build_well(hw_path,tw_path,is_train):
        global _FI,_DI
        wid=Path(hw_path).stem.replace('__horizontal_well','')
        try:
            hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
        except: return None
        if is_train and 'TVT' not in hw.columns: return None
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0 or len(kn)<10: return None
        if is_train and hw['TVT'].isna().all(): return None
        tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
        if len(tw_tvt)<3: return None
    
        pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
        if len(pf_a)==0: return None
        pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
        pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)
        has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))
    
        lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
        gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
        hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
        kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)
    
        # 7 beams (Numba JIT ±2)
        bpaths={}
        for (bs,mc,es,r,tag) in BEAMS:
            bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
        beam_ref=(bpaths['cons']+bpaths['sm5'])/2.
    
        # Multi-scale NCC → score-weighted ensemble
        ktvt=kn['TVT_input'].to_numpy(np.float32)
        sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
        sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
        sc_cons=(sc8+sc15+sc25)/3.
        sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
        hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens  # use ensemble not single
    
        tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
        a_cal,b_cal=affine_cal(kgr,tw_at_k)
        kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
        pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
        slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
        slp_z=robust_slope(kz,ktvt)
    
        swid=wid if is_train else None
        xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
        form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
        form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
        z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)
    
        # Per-formation: segment b_well (early/mid/late/wls) + TVT + known-zone RMSE
        tvt_fs={}; form_rmse={}; form_list=[]
        for fi2,fn in enumerate(FORMATIONS):
            b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
            tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)
            tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)
            tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)
            tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw
            tvt_fs[f'tvtF50_{fn}']=tvt_f50
            tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
            tvt_fs[f'bw50_{fn}']=np.float32(b_late)
            tvt_fs[f'bw_early_{fn}']=np.float32(b_early)   # NEW: early segment
            tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)       # NEW: mid segment
            form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
            form_list.append(tvt_f)
    
        fs=np.stack(form_list,1)
        form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
        form_std_d =fs.std(1).astype(np.float32)
        form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)
    
        d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
        d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
        b_vd=ktvt+z_kn-d_kn
        _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)
        b_d=float(np.median(b_vd))
        tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)
        tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)
        tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)
        res_kn=ktvt+z_kn-d_kn
        d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))
    
        all_sigs=[pf_use]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
        sig_mat=np.stack(all_sigs,1)
        sig_std=sig_mat.std(1).astype(np.float32)
        sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)
    
        gr_s=pd.Series(gr_full.values); rolls={}
        for w in [5,21,51,101]:
            r=gr_s.rolling(w,center=True,min_periods=1)
            rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
            rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
        for lag in [1,5,15,30]:
            rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
            rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
        gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
        gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
        gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
        gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)
                       ).iloc[ev.index].values.astype(np.float32)
    
        hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
        slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
        slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)
    
        mdd=hw['MD'].diff().replace(0,np.nan)
        dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
        dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
        dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
    
        nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
        def sc(v): return np.full(nh,np.float32(v),np.float32)
    
        feats={
            'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
            'last_known_tvt':sc(last_tvt),
            'pf_ancc':pf_use,'pf_ancc_std':std_use,
            'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
            'pf_z':(pf_z.astype(np.float32) if has_z else sc(last_tvt)),
            'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
            'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
            **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
            'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
            'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
            'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
            'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
            'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
            'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
            'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
            'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),  # score-weighted ensemble
            'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
            'sig_std':sig_std,'sig_mean_d':sig_mean,
            **tvt_fs,
            **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
            'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
            'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),
            'spatial_knn_dist':knn_d,
            'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
            'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
            'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
            'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
            'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
            'pf_vs_spatial':(pf_use-tvt_fs['tvtF_ANCC']).astype(np.float32),
            'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
            'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
            'beam_vs_spatial':(bpaths['cons']-tvt_fs['tvtF_ANCC']).astype(np.float32),
            'sc_vs_beam':(sc_ens-bpaths['cons']).astype(np.float32),
            'cal_a':sc(a_cal),'cal_b':sc(b_cal),
            'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
            'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
            'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
            'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),
            'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
            'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
            'z':z_ev,
            'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
            'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
            'dz':(z_ev-float(lk['Z'])).astype(np.float32),
            'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
            'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
            'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
            'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
            'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
            **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
            **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
            **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
            **{f'tdpf{int(o)}':hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},
            'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
        }
        for k,v in rolls.items(): feats[k]=v
        result=pd.DataFrame(feats)
        if is_train:
            if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
            result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
        return result
    
    def build_dataset(paths,is_train,label):
        args=[(str(p),str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'),is_train)
              for p in paths
              if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
        print(f"  {label}: {len(args)} wells | {NCPU} threads")
        t0=time.time()
        res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
            delayed(build_well)(hp,tp,it) for hp,tp,it in args)
        parts=[r for r in res if r is not None]
        el=time.time()-t0
        print(f"  {label}: OK={len(parts)} skipped={len(args)-len(parts)} | {el:.0f}s ({el/max(len(args),1):.1f}s/well)")
        return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()
    
    print("Feature builder OK ✓")
    
    
    print("Building train..."); t0=time.time()
    train_df=build_dataset(hw_paths,is_train=True,label="train")
    print(f"train: {train_df.shape}  {time.time()-t0:.0f}s")
    
    test_paths=sorted(TEST_DIR.glob('*__horizontal_well.csv'))
    print("Building test...")
    test_df=build_dataset(test_paths,is_train=False,label="test")
    print(f"test: {test_df.shape}")
    
    SKIP={'well','id','target'}
    feature_cols=[c for c in train_df.columns if c not in SKIP]
    print(f"#features: {len(feature_cols)}")
    X=train_df[feature_cols]; y=train_df['target']; g=train_df['well']
    Xt=test_df[feature_cols]; gc.collect()
    
    
    cv=GroupKFold(n_splits=N_SPLITS); splits=list(cv.split(X,y,g))
    
    def run_lgb(cfg_idx):
        cfg=LGB_CONFIGS[cfg_idx]; p=dict(LGB_BASE,**cfg); n_est=p.pop('n_estimators')
        oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
        for fold,(tr,va) in enumerate(splits):
            dtr=lgb.Dataset(X.iloc[tr],label=y.iloc[tr])
            dva=lgb.Dataset(X.iloc[va],label=y.iloc[va],reference=dtr)
            m=lgb.train(p,dtr,valid_sets=[dva],num_boost_round=n_est,
                        callbacks=[lgb.early_stopping(250,verbose=False),lgb.log_evaluation(800)])
            oof[va]=m.predict(X.iloc[va],num_iteration=m.best_iteration).astype(np.float32)
            tp+=m.predict(Xt,num_iteration=m.best_iteration).astype(np.float32)/N_SPLITS
            print(f"  LGB{cfg_idx} f{fold}: {root_mean_squared_error(y.iloc[va],oof[va]):.4f} iter={m.best_iteration}")
        r=root_mean_squared_error(y,oof); print(f"  LGB{cfg_idx} OOF={r:.4f}"); return oof,tp,r
    
    def run_cb():
        oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
        for fold,(tr,va) in enumerate(splits):
            m=CatBoostRegressor(**CB_P)
            m.fit(Pool(X.iloc[tr].values,label=y.iloc[tr].values),
                  eval_set=Pool(X.iloc[va].values,label=y.iloc[va].values),use_best_model=True)
            oof[va]=m.predict(X.iloc[va].values).astype(np.float32)
            tp+=m.predict(Xt.values).astype(np.float32)/N_SPLITS
            print(f"  CB f{fold}: {root_mean_squared_error(y.iloc[va],oof[va]):.4f}")
        r=root_mean_squared_error(y,oof); print(f"  CB OOF={r:.4f}"); return oof,tp,r
    
    results={}
    for i in range(3):
        oof,tp,r=run_lgb(i); results[f'lgb{i}']={'oof':oof,'test':tp,'rmse':r}
    oof,tp,r=run_cb(); results['cb']={'oof':oof,'test':tp,'rmse':r}
    
    Sx=np.column_stack([v['oof'] for v in results.values()])
    St=np.column_stack([v['test'] for v in results.values()])
    ridge=Ridge(alpha=1.,fit_intercept=False,positive=True); ridge.fit(Sx,y.values)
    oof_s=ridge.predict(Sx); test_s=ridge.predict(St)
    r_avg=root_mean_squared_error(y,Sx.mean(1)); r_stk=root_mean_squared_error(y,oof_s)
    wts=ridge.coef_/max(ridge.coef_.sum(),1e-9)
    print(f"\nAvg:{r_avg:.4f} Ridge:{r_stk:.4f} wts={dict(zip(results.keys(),wts.round(4)))}")
    final_oof =oof_s if r_stk<r_avg else Sx.mean(1)
    final_test=test_s if r_stk<r_avg else St.mean(1)
    
    
    base=train_df['last_known_tvt'].values; ytrue=y.values+base
    pf_oof=(train_df['pf_ancc'].values-base)
    
    print("Grid search alpha×tau×w_pf...")
    best_cfg,best_r=(None,None,None),np.inf
    for alpha in np.arange(0.65,1.01,0.05):
        for tau in [None,25.,50.,100.,200.,350.]:
            for w_pf in [0.0,0.05,0.10,0.15]:
                d=final_oof*(1-w_pf)+pf_oof*w_pf
                if tau: d*=(1.-np.exp(-np.maximum(train_df['md_since'].values,0.)/tau))
                d*=alpha
                r=root_mean_squared_error(ytrue,base+d)
                if r<best_r: best_r,best_cfg=r,(alpha,tau,w_pf)
    ALPHA,TAU,W_PF=best_cfg
    print(f"Best: alpha={ALPHA:.2f} tau={TAU} w_pf={W_PF:.2f} | abs TVT RMSE={best_r:.4f}")
    
    def apply_pp(df,md,pd_,alpha,tau,w_pf):
        d=md*(1-w_pf)+pd_*w_pf
        if tau: d*=(1.-np.exp(-np.maximum(df['md_since'].values,0.)/tau))
        return d*alpha
    
    def sg_smooth(df,col,sg_w=17,sg_p=3):
        df=df.copy()
        for well,g in df.groupby('well',sort=False):
            v=g[col].values; n=len(v); wl=min(sg_w,n)
            if wl%2==0: wl-=1
            if wl>=sg_p+2: v=savgol_filter(v,wl,sg_p)
            df.loc[g.index,col]=v
        return df
    
    test_df2=test_df.copy()
    pf_test=(test_df2['pf_ancc'].values-test_df2['last_known_tvt'].values)
    test_df2['pred']=(test_df2['last_known_tvt'].values
                     +apply_pp(test_df2,final_test,pf_test,ALPHA,TAU,W_PF))
    test_df2=sg_smooth(test_df2,'pred')
    
    sample=pd.read_csv(SAMPLE)
    sub=(sample[['id']].merge(
         test_df2[['id','pred']].rename(columns={'pred':'tvt'}),on='id',how='left'))
    fb=float(train_df['last_known_tvt'].mean()+train_df['target'].mean())
    sub['tvt']=sub['tvt'].fillna(fb)
    sub[['id','tvt']].to_csv('9.956.csv',index=False)
    print(f"\n✅ {OUT}  {len(sub)} rows")
    print("\n─── Summary ──────────────────────────────────────")
    for k,v in results.items(): print(f"  {k}: OOF residual={v['rmse']:.4f}")
    print(f"  stack  : {min(r_avg,r_stk):.4f}")
    print(f"  PostProc: abs TVT={best_r:.4f}")
    print(sub.head(8).to_string(index=False))
    
    print('\n\n','Model.4 finished its work and saved the result to a file:','9.956.csv','\n\n')

In [ ]:
from pathlib import Path
import importlib.util
import inspect
import json
import pickle
import shutil
import sys
from typing import Any

import pandas as pd
import numpy as np
from scipy.optimize import minimize

WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)


def _read_table_if_exists(path: Path):
    if path.suffix == '.parquet':
        return pd.read_parquet(path)
    if path.suffix == '.csv':
        return pd.read_csv(path)
    return None


def _write_table(frame: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix == '.parquet':
        frame.to_parquet(path, index=False)
    else:
        frame.to_csv(path, index=False)
    return path


def _sidecar_roots():
    roots = []
    seen = set()
    for root in SIDECAR_ARTIFACT_ROOTS:
        root = Path(root)
        candidates = [root]
        if (root / 'rogii_artifacts').exists():
            candidates.append(root / 'rogii_artifacts')
        for candidate in candidates:
            key = candidate.as_posix()
            if key in seen:
                continue
            if (candidate / 'submissions').exists() or (candidate / 'preds').exists() or (candidate / 'metadata').exists():
                roots.append(candidate)
                seen.add(key)
    return roots


def _selected_sidecar_from_report(root: Path):
    report_paths = [
        root / 'reports' / 'candidate_submission_files.parquet',
        root / 'reports' / 'candidate_submission_files.csv',
    ]
    for report_path in report_paths:
        if not report_path.exists():
            continue
        report = _read_table_if_exists(report_path)
        if report is None or report.empty or 'file' not in report.columns:
            continue
        selected = report
        if 'selected' in selected.columns:
            selected = selected[selected['selected'].astype(bool)]
        if selected.empty:
            selected = report.sort_values('oof_rmse') if 'oof_rmse' in report.columns else report
        file_name = str(selected.iloc[0]['file'])
        candidate = root / 'submissions' / file_name
        if candidate.exists():
            return candidate
    return None


def find_existing_sidecar_submission():
    roots = _sidecar_roots()
    if SIDECAR_SUBMISSION_NAME:
        for root in roots:
            candidate = root / 'submissions' / SIDECAR_SUBMISSION_NAME
            if candidate.exists():
                return candidate
        return None

    for root in roots:
        selected = _selected_sidecar_from_report(root)
        if selected is not None:
            return selected

    for root in roots:
        for file_name in SIDECAR_PREFERRED_FILES:
            candidate = root / 'submissions' / file_name
            if candidate.exists():
                return candidate

    for root in roots:
        files = sorted((root / 'submissions').glob('submission_*.csv')) if (root / 'submissions').exists() else []
        if files:
            return files[0]
    return None


def sidecar_discovery_report(selected_path: Path | None = None) -> pd.DataFrame:
    rows = []
    selected_key = selected_path.resolve().as_posix() if selected_path is not None and Path(selected_path).exists() else ''
    for root in _sidecar_roots():
        sub_dir = root / 'submissions'
        files = sorted(sub_dir.glob('submission_*.csv')) if sub_dir.exists() else []
        for f in files:
            try:
                key = f.resolve().as_posix()
            except Exception:
                key = f.as_posix()
            rows.append({
                'root': root.as_posix(),
                'file': f.name,
                'path': f.as_posix(),
                'selected': bool(selected_key and key == selected_key),
                'preferred_rank': SIDECAR_PREFERRED_FILES.index(f.name) if f.name in SIDECAR_PREFERRED_FILES else np.nan,
            })
    out = pd.DataFrame(rows)
    out.to_csv(WORK_DIR / 'sidecar_discovery_report.csv', index=False)
    if not out.empty:
        display(out)
    return out


def _find_hblend_input_file(name: str) -> Path | None:
    file_name = f'{name}.csv'
    for root in HBLEND_INPUT_ROOTS:
        candidate = Path(root) / file_name
        if candidate.exists():
            return candidate
    # Kaggle dataset mount names can differ from the dataset title or owner path.
    # Fall back to a recursive lookup so attaching the right dataset is enough.
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.glob(f'**/{file_name}'))
        if matches:
            print(f'h-blend input {file_name} found by recursive search:', matches[0])
            return matches[0]
    return None


def copy_hblend_base_csvs_to_working():
    copied = []
    for name in ['9.537', '9.765', '9.956']:
        dst = WORK_DIR / f'{name}.csv'
        src = _find_hblend_input_file(name)
        if RUN_GENERATED_MODEL_CELLS and dst.exists():
            copied.append({'name': name, 'source': 'generated', 'destination': dst.as_posix()})
            continue
        if src is None:
            if dst.exists():
                try:
                    validated = validate_submission_ids(pd.read_csv(dst), label=f'hblend_input:{name}')
                except RuntimeError as exc:
                    if _looks_like_id_coverage_error(exc):
                        _raise_current_sample_mismatch(f'hblend_input:{name}', exc, sidecar=False)
                    raise
                validated.to_csv(dst, index=False)
                copied.append({'name': name, 'source': 'existing_working', 'destination': dst.as_posix(), 'rows': len(validated)})
                continue
            raise FileNotFoundError(f'Missing Nina h-blend input {name}.csv. Attach the rogii-03 dataset or run generated model cells.')
        shutil.copy2(src, dst)
        try:
            validated = validate_submission_ids(pd.read_csv(dst), label=f'hblend_input:{name}')
        except RuntimeError as exc:
            if _looks_like_id_coverage_error(exc):
                _raise_current_sample_mismatch(f'hblend_input:{name}', exc, sidecar=False)
            raise
        validated.to_csv(dst, index=False)
        copied.append({'name': name, 'source': src.as_posix(), 'destination': dst.as_posix(), 'rows': len(validated)})
    return pd.DataFrame(copied)


def _artifact_files(kind: str):
    files = []
    seen = set()
    for root in _sidecar_roots():
        pred_dir = root / 'preds'
        if not pred_dir.exists():
            continue
        for suffix in ('*.parquet', '*.csv'):
            for path in pred_dir.glob(suffix):
                if f'_{kind}' not in path.stem:
                    continue
                key = path.resolve().as_posix()
                if key not in seen:
                    files.append(path)
                    seen.add(key)
    return sorted(files)


def _prediction_column_name(frame: pd.DataFrame) -> str:
    first = frame.iloc[0]
    return f"pred_delta_{first['branch_name']}_{first['model_name']}"


def _coalesce_merged_metadata(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        new_col = f'{col}__new'
        if new_col not in frame.columns:
            continue
        if col in frame.columns:
            frame[col] = frame[col].combine_first(frame[new_col])
            frame = frame.drop(columns=[new_col])
        else:
            frame = frame.rename(columns={new_col: col})
    return frame


def _merge_prediction_artifacts(files, include_target: bool):
    base = None
    score_rows = []
    meta_columns = ['well_id', 'row_index', 'last_known_TVT', 'target_tvt', 'target_delta']
    for path in files:
        pred = _read_table_if_exists(path)
        if pred is None or pred.empty:
            continue
        if pred['id'].duplicated().any():
            dup = pred.loc[pred['id'].duplicated(), 'id'].head(10).tolist()
            raise RuntimeError(f'Duplicated ids in {path.name}: {dup}')
        col = _prediction_column_name(pred)
        if base is not None and col in base.columns:
            raise RuntimeError(f'Duplicate prediction column {col}. Check repeated branch artifacts across input roots.')
        keep = ['id', 'well_id', 'row_index', 'last_known_TVT', 'pred_delta']
        if include_target:
            keep += ['target_tvt', 'target_delta']
        missing = [c for c in keep if c not in pred.columns]
        if missing:
            raise RuntimeError(f'Missing required columns in {path.name}: {missing}')
        tmp = pred[keep].rename(columns={'pred_delta': col})
        if base is None:
            base = tmp
        else:
            base = base.merge(tmp, on='id', how='outer', suffixes=('', '__new'))
            base = _coalesce_merged_metadata(base, meta_columns)
        if include_target and 'target_delta' in pred.columns:
            y = pred['target_delta'].to_numpy(dtype=float)
            z = pred['pred_delta'].to_numpy(dtype=float)
            mask = np.isfinite(y) & np.isfinite(z)
            score_rows.append({
                'file': path.name,
                'branch_name': str(pred['branch_name'].iloc[0]),
                'model_name': str(pred['model_name'].iloc[0]),
                'prediction_type': str(pred['prediction_type'].iloc[0]),
                'rmse': float(np.sqrt(np.mean((y[mask] - z[mask]) ** 2))) if mask.any() else np.nan,
                'rows': int(len(pred)),
            })
    return (base if base is not None else pd.DataFrame()), pd.DataFrame(score_rows)


def _test_artifact_info(path: Path, rank: int | None = None):
    pred = _read_table_if_exists(path)
    if pred is None or pred.empty:
        return None
    col = _prediction_column_name(pred)
    ptype = str(pred['prediction_type'].iloc[0]) if 'prediction_type' in pred.columns else 'unknown'
    return col, {
        'prediction_type': ptype,
        'rank': int(rank) if rank is not None else -1,
        'file': path.name,
        'rows': int(len(pred)),
    }


def _select_test_artifacts(files):
    files = list(files)
    priority = {name: rank for rank, name in enumerate(SIDECAR_STACK_TEST_PRIORITY)}
    if SIDECAR_STACK_TEST_PREDICTION_TYPE != 'auto':
        selected = [p for p in files if p.stem.endswith(f'_{SIDECAR_STACK_TEST_PREDICTION_TYPE}')]
        info = {}
        for path in selected:
            item = _test_artifact_info(path, rank=priority.get(SIDECAR_STACK_TEST_PREDICTION_TYPE, 0))
            if item is None:
                continue
            col, row = item
            if col in info:
                raise RuntimeError(f'Duplicate fixed-mode test artifact for {col}: {info[col]["file"]} and {path.name}')
            info[col] = row
        return selected, info
    chosen = {}
    info = {}
    for path in files:
        item = _test_artifact_info(path)
        if item is None:
            continue
        col, row = item
        rank = priority.get(row['prediction_type'], len(priority) + 100)
        row['rank'] = int(rank)
        cur = info.get(col)
        if cur is None or rank < cur['rank']:
            chosen[col] = path
            info[col] = row
        elif cur is not None and rank == cur['rank']:
            raise RuntimeError(f'Duplicate test artifact for {col} at same priority: {cur["file"]} and {path.name}')
    return list(chosen.values()), info


def _fit_constrained_residual_blend(y_delta, X, l2=0.0):
    y = np.asarray(y_delta, dtype=float)
    X = np.asarray(X, dtype=float)
    valid = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
    y = y[valid]
    X = X[valid]
    n = X.shape[1]
    if n == 0 or len(y) == 0:
        raise RuntimeError('No rows/columns available for sidecar blend fitting.')
    x0 = np.full(n, min(1.0 / max(n, 1), 0.20), dtype=float)
    if x0.sum() > 1.0:
        x0 = x0 / x0.sum()

    def obj(w):
        err = y - X.dot(w)
        return float(np.mean(err * err) + float(l2) * np.sum(w * w))

    res = minimize(
        obj,
        x0,
        method='SLSQP',
        bounds=[(0.0, 1.0)] * n,
        constraints=[{'type': 'ineq', 'fun': lambda w: 1.0 - np.sum(w)}],
        options={'maxiter': 1000, 'ftol': 1e-10, 'disp': False},
    )
    if not res.success:
        print('WARNING: sidecar SLSQP did not fully converge:', res.message)
    w = np.clip(np.asarray(res.x, dtype=float), 0.0, 1.0)
    if w.sum() > 1.0:
        w = w / w.sum()
    pred = X.dot(w)
    rmse = float(np.sqrt(np.mean((y - pred) ** 2)))
    return {'weights': w.tolist(), 'anchor_weight': float(1.0 - w.sum()), 'oof_rmse': rmse, 'rows_used': int(len(y)), 'l2': float(l2)}


def _is_under_path(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except Exception:
        return False


def _sample_path_priority(path: Path) -> tuple[int, str]:
    text = path.as_posix().lower()
    parent = path.parent
    is_sidecar_sample = any(_is_under_path(path, root) for root in _sidecar_roots())
    if PUBLIC_PROBE_MODE:
        if is_sidecar_sample:
            return (0, text)
        if 'rogii-wellbore-geology-prediction' in text:
            return (5, text)
        if (parent / 'train').exists() and (parent / 'test').exists():
            return (6, text)
        return (9, text)
    if 'rogii-wellbore-geology-prediction' in text:
        return (0, text)
    if (parent / 'train').exists() and (parent / 'test').exists():
        return (1, text)
    if path.suffix == '.csv' and not is_sidecar_sample:
        return (2, text)
    return (9, text)


def _find_sample_submission():
    candidates = [
        Path('/kaggle/input/rogii-wellbore-geology-prediction/sample_submission.csv'),
        Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv'),
    ]
    if Path('/kaggle/input').exists():
        candidates.extend(sorted(Path('/kaggle/input').glob('*/sample_submission.csv')))
        candidates.extend(sorted(Path('/kaggle/input').glob('*/*/sample_submission.csv')))
        candidates.extend(sorted(Path('/kaggle/input').glob('**/sample_submission.csv')))
    for root in _sidecar_roots():
        candidates.extend([
            root / 'metadata' / 'sample_submission.parquet',
            root / 'metadata' / 'sample_submission.csv',
        ])
    seen = set()
    ordered = []
    for path in candidates:
        key = path.as_posix()
        if key in seen or not path.exists():
            continue
        seen.add(key)
        ordered.append(path)
    for path in sorted(ordered, key=_sample_path_priority):
        frame = _read_table_if_exists(path)
        if frame is not None and 'id' in frame.columns:
            print('sample_submission source:', path)
            return frame[['id']].copy()
    return None


def validate_submission_ids(df: pd.DataFrame, label: str = 'submission') -> pd.DataFrame:
    if not {'id', 'tvt'}.issubset(df.columns):
        raise RuntimeError(f"{label}: expected columns ['id', 'tvt']; got {list(df.columns)}")
    frame = df[['id', 'tvt']].copy()
    frame['id'] = frame['id'].astype(str)
    if frame['id'].duplicated().any():
        dup = frame.loc[frame['id'].duplicated(), 'id'].head(10).tolist()
        raise RuntimeError(f'{label}: duplicated ids: {dup}')
    if not np.isfinite(frame['tvt'].to_numpy(dtype=float)).all():
        raise RuntimeError(f'{label}: non-finite tvt values')
    sample = _find_sample_submission()
    if sample is None:
        print(f'WARNING: {label}: sample submission not found; id coverage cannot be verified.')
        return frame
    sample = sample[['id']].copy()
    sample['id'] = sample['id'].astype(str)
    sample_ids = set(sample['id'])
    frame_ids = set(frame['id'])
    missing = sorted(sample_ids - frame_ids)
    extra = sorted(frame_ids - sample_ids)
    if missing:
        raise RuntimeError(f'{label}: missing {len(missing)} sample ids; examples={missing[:10]}')
    if extra:
        raise RuntimeError(f'{label}: extra {len(extra)} ids not in sample; examples={extra[:10]}')
    aligned = sample[['id']].merge(frame, on='id', how='left')
    if aligned['tvt'].isna().any():
        bad = aligned.loc[aligned['tvt'].isna(), 'id'].head(10).tolist()
        raise RuntimeError(f'{label}: NaN after sample alignment; examples={bad}')
    return aligned[['id', 'tvt']]


def _looks_like_id_coverage_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return any(token in msg for token in ['missing', 'extra', 'sample ids', 'not in sample', 'nan after sample alignment'])


def _raise_current_sample_mismatch(label: str, exc: Exception, *, sidecar: bool = False):
    if sidecar:
        hint = (
            'Sidecar branch artifacts appear to target a different sample id set than this notebook rerun. '
            'That is expected if public-test artifacts are attached during a hidden rerun. '
            'Use this notebook for public probing, or set SIDECAR_INTEGRATION_MODE = "off" for a pure Nina rerun. '
            'A sidecar can only be blended when its test artifacts were generated for the current sample ids.'
        )
    else:
        hint = (
            'The precomputed Nina h-blend CSVs appear to target a different sample id set than this notebook rerun. '
            'That commonly happens on hidden reruns because the hidden sample can differ from the public sample. '
            'For a hidden-safe run, either set RUN_GENERATED_MODEL_CELLS = True so Nina model CSVs are regenerated for the current sample, '
            'or use this notebook only as a public-probing wrapper with matching public ids.'
        )
    raise RuntimeError(f'{label}: current sample id coverage mismatch. {hint} Original validation error: {exc}') from exc


def _align_to_sample(pred_frame: pd.DataFrame):
    try:
        return validate_submission_ids(pred_frame, label='built_sidecar_stack')
    except RuntimeError as exc:
        if _looks_like_id_coverage_error(exc):
            _raise_current_sample_mismatch('built_sidecar_stack', exc, sidecar=True)
        raise


def _check_sidecar_branch_set(columns):
    if not SIDECAR_STACK_REQUIRE_STRONG_BRANCHES:
        return
    text = '\n'.join(columns)
    missing = []
    if 'pred_delta_base_lgb_' not in text:
        missing.append('base_lgb')
    if 'pred_delta_plane_formation_' not in text:
        missing.append('plane_formation')
    if not any(token in text for token in ['pred_delta_super_top3_', 'pred_delta_super_top3_compact_', 'pred_delta_top2_physics_']):
        missing.append('one physics branch')
    if missing:
        raise RuntimeError(f'Sidecar stack is missing required branches: {missing}')


def build_sidecar_submission_from_branch_artifacts() -> Path:
    oof_files = _artifact_files('oof')
    test_files_all = _artifact_files('test')
    if not oof_files:
        raise FileNotFoundError('No sidecar OOF artifacts found under attached artifact roots.')
    if not test_files_all:
        raise FileNotFoundError('No sidecar test artifacts found under attached artifact roots.')
    test_files, selected_test_info = _select_test_artifacts(test_files_all)
    if not test_files:
        raise FileNotFoundError(f'No sidecar test artifacts selected for type {SIDECAR_STACK_TEST_PREDICTION_TYPE}.')

    train_stack, branch_scores = _merge_prediction_artifacts(oof_files, include_target=True)
    test_stack, _ = _merge_prediction_artifacts(test_files, include_target=False)
    pred_cols = [c for c in train_stack.columns if c.startswith('pred_delta_')]
    _check_sidecar_branch_set(pred_cols)

    coverage_rows = []
    usable_cols = []
    for col in pred_cols:
        train_rate = float(train_stack[col].notna().mean())
        test_rate = float(test_stack[col].notna().mean()) if col in test_stack.columns else 0.0
        coverage_rows.append({'prediction': col, 'train_non_null_rate': train_rate, 'test_non_null_rate': test_rate})
        if train_rate > SIDECAR_STACK_MIN_TRAIN_NON_NULL_RATE and test_rate > SIDECAR_STACK_MIN_TEST_NON_NULL_RATE:
            usable_cols.append(col)
    if not usable_cols:
        raise RuntimeError('No sidecar prediction columns have sufficient OOF/test coverage.')

    complete_mask = train_stack[usable_cols].notna().all(axis=1) & train_stack['target_delta'].notna()
    test_complete = test_stack[usable_cols].notna().all(axis=1)
    if not bool(test_complete.all()):
        bad = test_stack.loc[~test_complete, 'id'].head(10).tolist()
        raise RuntimeError(f'Missing sidecar test predictions for usable columns; examples={bad}')

    y = train_stack.loc[complete_mask, 'target_delta'].to_numpy(dtype=float)
    X = train_stack.loc[complete_mask, usable_cols].to_numpy(dtype=float)
    configs = []
    for l2 in SIDECAR_STACK_L2_GRID:
        cfg = _fit_constrained_residual_blend(y, X, l2=l2)
        cfg['prediction_columns'] = list(usable_cols)
        configs.append(cfg)
    best = min(configs, key=lambda row: row['oof_rmse'])
    weights = np.asarray(best['weights'], dtype=float)
    pred_delta = test_stack[usable_cols].to_numpy(dtype=float).dot(weights)
    pred_frame = pd.DataFrame({
        'id': test_stack['id'].to_numpy(),
        'tvt': test_stack['last_known_TVT'].to_numpy(dtype=float) + pred_delta,
    })
    sidecar = _align_to_sample(pred_frame)
    if not np.isfinite(sidecar['tvt'].to_numpy(dtype=float)).all():
        raise RuntimeError('Non-finite TVT in built sidecar stack submission.')

    out_path = WORK_DIR / f'{SIDECAR_HBLEND_NAME}.csv'
    sidecar.to_csv(out_path, index=False)

    report_dir = WORK_DIR / 'sidecar_standalone_reports'
    report_dir.mkdir(parents=True, exist_ok=True)
    _write_table(pd.DataFrame(coverage_rows), report_dir / 'sidecar_stack_coverage.parquet')
    _write_table(branch_scores.sort_values('rmse'), report_dir / 'sidecar_branch_oof_scores.parquet')
    _write_table(pd.DataFrame.from_dict(selected_test_info, orient='index').reset_index().rename(columns={'index': 'prediction'}), report_dir / 'sidecar_selected_test_artifacts.parquet')
    pd.Series({
        'sidecar_source': 'built_from_branch_preds',
        'oof_files': len(oof_files),
        'test_files_selected': len(test_files),
        'usable_prediction_columns': len(usable_cols),
        'complete_oof_rows': int(complete_mask.sum()),
        'total_oof_rows': int(len(train_stack)),
        'oof_rmse': best['oof_rmse'],
        'l2': best['l2'],
        'anchor_weight': best['anchor_weight'],
        'weight_sum': float(np.sum(weights)),
        'rows': int(len(sidecar)),
        'output_file': out_path.as_posix(),
    }).to_csv(report_dir / 'sidecar_stack_summary.csv')
    _write_table(pd.DataFrame({'prediction': usable_cols, 'weight': weights}).sort_values('weight', ascending=False), report_dir / 'sidecar_stack_weights.parquet')
    print('Built standalone sidecar stack:', out_path)
    print('Sidecar OOF RMSE:', best['oof_rmse'], 'columns:', len(usable_cols), 'anchor:', best['anchor_weight'])
    return out_path



def _find_competition_root_for_model_package() -> Path:
    candidates = [
        Path('/kaggle/input/rogii-wellbore-geology-prediction'),
        Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    ]
    for root in candidates:
        if (root / 'sample_submission.csv').exists() and (root / 'test').exists():
            return root
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for sample in input_root.glob('**/sample_submission.csv'):
            root = sample.parent
            if (root / 'test').exists():
                return root
    raise RuntimeError('Could not find competition data root with sample_submission.csv and test/.')


def _find_model_package_root() -> Path:
    for root in SIDECAR_MODEL_PACKAGE_ROOTS:
        root = Path(root)
        if (root / 'metadata' / 'model_package_manifest.json').exists():
            return root
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for manifest_path in input_root.glob('**/metadata/model_package_manifest.json'):
            return manifest_path.parents[1]
    raise FileNotFoundError(
        'No sidecar model package found. Attach a Dataset containing '
        'metadata/model_package_manifest.json, feature_builders/, stacking/, and models/.'
    )


def _read_json_file(path: Path):
    with path.open() as f:
        return json.load(f)


def _sidecar_manifest_path(manifest: dict[str, Any], key: str, default: str) -> str:
    value = manifest.get(key, default)
    if isinstance(value, str) and value.strip():
        return value
    raise RuntimeError(f'Sidecar manifest field {key!r} must be a relative file path string.')


def _sidecar_model_package_weight_source(manifest: dict[str, Any], blend_config: dict[str, Any]) -> str:
    parts = [
        blend_config.get('weight_source'),
        manifest.get('weight_source'),
        blend_config.get('fit_type'),
        manifest.get('training_feature_version'),
    ]
    return ' | '.join(str(part) for part in parts if part not in (None, ''))


def _validate_sidecar_model_package_weight_source(manifest: dict[str, Any], blend_config: dict[str, Any]) -> str:
    source = _sidecar_model_package_weight_source(manifest, blend_config)
    if bool(globals().get('SIDECAR_REQUIRE_OOF_WEIGHTED_PACKAGE', False)):
        token = str(globals().get('SIDECAR_EXPECTED_WEIGHT_SOURCE_TOKEN', 'oof')).strip().lower()
        if token and token not in source.lower():
            raise RuntimeError(
                'The attached sidecar model package does not look OOF-weighted. '
                f'Expected token {token!r} in manifest/blend_config weight source, got {source!r}. '
                'Attach the dataset version produced with build_step5_model_package --weights-json, '
                'or set SIDECAR_REQUIRE_OOF_WEIGHTED_PACKAGE = False for diagnostics.'
            )
    return source


def _sidecar_prediction_column(entry: dict[str, Any]) -> str:
    if entry.get('prediction_column'):
        return str(entry['prediction_column'])
    branch_name = entry.get('branch_name')
    model_name = entry.get('model_name')
    if not branch_name or not model_name:
        raise RuntimeError(f'Sidecar model entry needs prediction_column or branch_name/model_name: {entry}')
    return f'pred_delta_{branch_name}_{model_name}'


def _validate_model_package_manifest(package_root: Path, manifest: dict[str, Any]) -> None:
    required = ['schema_version', 'package_type', 'hidden_inference_supported', 'feature_sets', 'models', 'blend_config']
    missing = [field for field in required if field not in manifest]
    if missing:
        raise RuntimeError(f'Sidecar model package manifest missing fields: {missing}')
    if manifest.get('hidden_inference_supported') is not True:
        raise RuntimeError('Sidecar model package does not set hidden_inference_supported=true.')
    if manifest.get('package_type') not in {'rogii_hidden_model_package', 'hidden_model_package'}:
        raise RuntimeError(f'Unexpected sidecar model package type: {manifest.get("package_type")!r}')
    if not isinstance(manifest.get('models'), list) or not manifest.get('models'):
        raise RuntimeError('Sidecar model package manifest must contain a non-empty models list.')

    feature_columns_rel = _sidecar_manifest_path(manifest, 'feature_columns', 'feature_builders/feature_columns.json')
    blend_config_rel = _sidecar_manifest_path(manifest, 'blend_config', 'stacking/blend_config.json')
    for rel in [feature_columns_rel, blend_config_rel]:
        if not (package_root / rel).exists():
            raise RuntimeError(f'Sidecar model package is missing {rel}')

    blend_config = _read_json_file(package_root / blend_config_rel)
    blend_space = blend_config.get('target_space') or blend_config.get('prediction_space') or manifest.get('target_space', 'delta')
    if blend_space not in {'delta', 'tvt'}:
        raise RuntimeError(f'Unsupported sidecar blend target_space={blend_space!r}.')

    allowed_model_types = {
        'lightgbm_booster',
        'lightgbm_sklearn_pickle',
        'xgboost_json',
        'xgboost_pickle',
        'catboost_cbm',
        'sklearn_pickle',
        'direct_feature',
    }
    seen = set()
    for idx, entry in enumerate(manifest.get('models', [])):
        model_type = entry.get('model_type')
        if model_type not in allowed_model_types:
            raise RuntimeError(f'Unsupported sidecar model_type in entry {idx}: {model_type!r}')
        pred_col = _sidecar_prediction_column(entry)
        if pred_col in seen:
            raise RuntimeError(f'Missing or duplicated prediction_column in sidecar entry {idx}: {entry}')
        seen.add(pred_col)
        entry_space = entry.get('target_space', blend_space)
        if entry_space != blend_space:
            raise RuntimeError(f'Mixed sidecar target_space is not supported: {pred_col} has {entry_space!r}, blend uses {blend_space!r}')
        if model_type == 'direct_feature':
            if not entry.get('feature_column'):
                raise RuntimeError(f'direct_feature sidecar entry must define feature_column: {entry}')
            continue
        rel = entry.get('path')
        if not rel or not (package_root / rel).exists():
            raise RuntimeError(f'Sidecar model package is missing model file for entry: {entry}')


def _load_sidecar_feature_builder(package_root: Path):
    feature_dir = package_root / 'feature_builders'
    for import_root in [package_root, feature_dir]:
        key = str(import_root)
        if key not in sys.path:
            sys.path.insert(0, key)
    for path in [feature_dir / 'build_features.py', feature_dir / 'feature_builder.py']:
        if not path.exists():
            continue
        spec = importlib.util.spec_from_file_location('rogii_hblend_sidecar_feature_builder', path)
        if spec is None or spec.loader is None:
            raise RuntimeError(f'Could not import sidecar feature builder: {path}')
        module = importlib.util.module_from_spec(spec)
        sys.modules[spec.name] = module
        spec.loader.exec_module(module)
        for fn_name in ['build_features', 'build_tail_features', 'make_features']:
            if hasattr(module, fn_name):
                return getattr(module, fn_name), path
    raise RuntimeError('Sidecar model package has no feature_builders/build_features.py with build_features(...).')


def _call_sidecar_feature_builder(builder, *, data_dir: Path, sample: pd.DataFrame, package_root: Path, manifest: dict[str, Any]) -> pd.DataFrame:
    possible_kwargs = {
        'data_dir': data_dir,
        'competition_root': data_dir,
        'sample_submission': sample,
        'sample': sample,
        'package_root': package_root,
        'manifest': manifest,
        'config': manifest,
    }
    sig = inspect.signature(builder)
    kwargs = {name: value for name, value in possible_kwargs.items() if name in sig.parameters}
    features = builder(**kwargs)
    if not isinstance(features, pd.DataFrame):
        raise RuntimeError('Sidecar feature builder must return a pandas DataFrame.')
    if 'id' not in features.columns:
        raise RuntimeError('Sidecar feature frame must include id.')
    features = features.copy()
    features['id'] = features['id'].astype(str)
    if features['id'].duplicated().any():
        dup = features.loc[features['id'].duplicated(), 'id'].head(10).tolist()
        raise RuntimeError(f'Sidecar feature frame has duplicated ids: {dup}')
    sample_ids = set(sample['id'].astype(str))
    feature_ids = set(features['id'].astype(str))
    missing = sorted(sample_ids - feature_ids)
    extra = sorted(feature_ids - sample_ids)
    if missing or extra:
        raise RuntimeError(f'Sidecar feature frame id mismatch: missing={len(missing)}, extra={len(extra)}, missing_examples={missing[:10]}')
    sample_ids_frame = sample[['id']].copy()
    sample_ids_frame['id'] = sample_ids_frame['id'].astype(str)
    return sample_ids_frame.merge(features, on='id', how='left')


def _feature_columns_for_sidecar_model(feature_columns_config, entry: dict[str, Any]) -> list[str]:
    if isinstance(entry.get('feature_columns'), list):
        return list(entry['feature_columns'])
    feature_set = entry.get('feature_set')
    if isinstance(feature_columns_config, list):
        return list(feature_columns_config)
    if isinstance(feature_columns_config, dict):
        if feature_set and isinstance(feature_columns_config.get(feature_set), list):
            return list(feature_columns_config[feature_set])
        if isinstance(feature_columns_config.get('columns'), list):
            return list(feature_columns_config['columns'])
    raise RuntimeError(f'Could not resolve sidecar feature columns for model entry: {entry}')


def _load_sidecar_model(package_root: Path, entry: dict[str, Any]):
    model_type = entry.get('model_type')
    path = package_root / entry['path']
    if model_type == 'lightgbm_booster':
        import lightgbm as lgb
        return lgb.Booster(model_file=str(path))
    if model_type == 'xgboost_json':
        import xgboost as xgb
        booster = xgb.Booster()
        booster.load_model(str(path))
        return booster
    if model_type == 'catboost_cbm':
        from catboost import CatBoostRegressor
        model = CatBoostRegressor()
        model.load_model(str(path))
        return model
    if model_type in {'lightgbm_sklearn_pickle', 'xgboost_pickle', 'sklearn_pickle'}:
        try:
            import joblib
            return joblib.load(path)
        except Exception:
            with path.open('rb') as f:
                return pickle.load(f)
    raise RuntimeError(f'Unsupported sidecar model_type={model_type!r}: {entry}')


def _sidecar_feature_matrix(frame: pd.DataFrame, columns: list[str], entry: dict[str, Any]) -> pd.DataFrame:
    missing = [c for c in columns if c not in frame.columns]
    if missing:
        raise RuntimeError(f'Sidecar feature frame missing {len(missing)} columns; examples={missing[:10]}')
    X_df = frame[columns].replace([np.inf, -np.inf], np.nan)
    fill_value = entry.get('fillna', None)
    policy = str(entry.get('missing_value_policy', globals().get('_SIDECAR_MODEL_PACKAGE_MISSING_VALUE_POLICY', 'native'))).lower()
    if fill_value is not None:
        X_df = X_df.fillna(float(fill_value))
    elif policy in {'native', 'none', 'null'}:
        pass
    elif policy in {'zero', 'fill_zero'}:
        X_df = X_df.fillna(0.0)
    else:
        raise RuntimeError(f'Unsupported sidecar missing_value_policy={policy!r} for {entry.get("prediction_column")}')
    return X_df


def _predict_sidecar_model(model, model_type: str, frame: pd.DataFrame, columns: list[str], entry: dict[str, Any]) -> np.ndarray:
    X_df = _sidecar_feature_matrix(frame, columns, entry)
    if model_type == 'xgboost_json':
        import xgboost as xgb
        pred = model.predict(xgb.DMatrix(X_df.to_numpy(dtype=np.float32)))
    else:
        pred = model.predict(X_df)
    pred = np.asarray(pred, dtype=float)
    if pred.ndim > 1:
        pred = pred.reshape(len(frame), -1)[:, 0]
    if len(pred) != len(frame):
        raise RuntimeError(f'Sidecar model prediction length mismatch: got {len(pred)}, expected {len(frame)}')
    if not np.isfinite(pred).all():
        raise RuntimeError(f'Sidecar model {entry.get("prediction_column")} produced non-finite predictions.')
    return pred


def _weights_from_keys_and_coef(keys: Any, coef: Any) -> dict[str, float]:
    if keys is None or coef is None:
        return {}
    keys = list(keys)
    coef = np.asarray(coef, dtype=float).reshape(-1).tolist()
    if len(keys) != len(coef):
        raise RuntimeError(f'Sidecar blend result_keys/coef length mismatch: {len(keys)} vs {len(coef)}')
    return {str(k): float(v) for k, v in zip(keys, coef)}


def _normalize_sidecar_model_package_weights(blend_config: dict[str, Any]) -> dict[str, float]:
    if isinstance(blend_config.get('weights'), dict):
        return {str(k): float(v) for k, v in blend_config['weights'].items()}
    if isinstance(blend_config.get('model_weights'), dict):
        return {str(k): float(v) for k, v in blend_config['model_weights'].items()}
    weights = _weights_from_keys_and_coef(blend_config.get('result_keys'), blend_config.get('coef'))
    if weights:
        return weights
    stacker = blend_config.get('stacker')
    if isinstance(stacker, dict):
        weights = _weights_from_keys_and_coef(stacker.get('result_keys'), stacker.get('coef'))
        if weights:
            return weights
        if isinstance(stacker.get('weights'), dict):
            return {str(k): float(v) for k, v in stacker['weights'].items()}
        if isinstance(stacker.get('model_weights'), dict):
            return {str(k): float(v) for k, v in stacker['model_weights'].items()}
    weights = {}
    for row in blend_config.get('models', []):
        if 'prediction_column' in row and 'weight' in row:
            weights[str(row['prediction_column'])] = float(row['weight'])
    if weights:
        return weights
    raise RuntimeError('Sidecar blend_config.json must contain weights/model_weights, result_keys+coef, stacker result_keys+coef, or models[{prediction_column, weight}].')


def _sidecar_blend_intercept(blend_config: dict[str, Any]) -> float:
    for key in ('intercept', 'bias'):
        if key in blend_config and blend_config[key] is not None:
            return float(np.asarray(blend_config[key], dtype=float).reshape(-1)[0])
    stacker = blend_config.get('stacker')
    if isinstance(stacker, dict):
        for key in ('intercept', 'bias'):
            if key in stacker and stacker[key] is not None:
                return float(np.asarray(stacker[key], dtype=float).reshape(-1)[0])
    return 0.0


def _validate_sidecar_weights(weights: dict[str, float], blend_config: dict[str, Any]) -> None:
    if not weights:
        raise RuntimeError('Sidecar blend weights are empty.')
    bad_finite = {k: v for k, v in weights.items() if not np.isfinite(v)}
    if bad_finite:
        raise RuntimeError(f'Non-finite sidecar blend weights: {bad_finite}')
    if bool(blend_config.get('enforce_nonnegative', True)):
        bad_negative = {k: v for k, v in weights.items() if v < -1e-9}
        if bad_negative:
            raise RuntimeError(f'Negative sidecar blend weights are not allowed: {bad_negative}')
    weight_sum = float(sum(weights.values()))
    max_weight_sum = float(blend_config.get('max_weight_sum', 1.05))
    if weight_sum > max_weight_sum:
        raise RuntimeError(f'Sidecar blend weight sum too large: {weight_sum:.6f} > {max_weight_sum:.6f}')


def _first_existing_sidecar_column(frame: pd.DataFrame, names: list[str]) -> str | None:
    for name in names:
        if name in frame.columns:
            return name
    return None


def _apply_sidecar_delta_postprocess(delta: np.ndarray, blend_config: dict[str, Any], features: pd.DataFrame) -> np.ndarray:
    post = blend_config.get('postprocess', {}) or {}
    out = delta.astype(float).copy()
    tau = post.get('fade_tau_md', post.get('tau', None))
    if tau is not None:
        md_col = _first_existing_sidecar_column(features, ['md_since_ps', 'md_since', 'md_delta', 'MD_since', 'md_from_start'])
        if md_col is None:
            raise RuntimeError('Sidecar postprocess.fade_tau_md was set, but no md_since column is available.')
        md_since = pd.to_numeric(features[md_col], errors='coerce').to_numpy(dtype=float)
        out *= 1.0 - np.exp(-np.maximum(md_since, 0.0) / float(tau))
    out *= float(post.get('alpha', 1.0))
    return out


def _apply_sidecar_savgol_if_requested(tvt: np.ndarray, blend_config: dict[str, Any], features: pd.DataFrame) -> np.ndarray:
    post = blend_config.get('postprocess', {}) or {}
    window = int(post.get('savgol_window', 0) or 0)
    if window <= 2:
        return tvt
    if window % 2 == 0:
        window += 1
    poly = int(post.get('savgol_poly', 2) or 2)
    try:
        from scipy.signal import savgol_filter
    except Exception as exc:
        raise RuntimeError(f'Sidecar Savitzky-Golay smoothing requested but scipy is unavailable: {exc}')
    out = tvt.astype(float).copy()
    group_col = _first_existing_sidecar_column(features, ['well_id', 'well', 'WELL'])
    row_col = _first_existing_sidecar_column(features, ['row_index', 'row', 'sample_index'])
    tmp = pd.DataFrame({'_pos': np.arange(len(out)), '_tvt': out})
    tmp['_group'] = features[group_col].astype(str).to_numpy() if group_col else features['id'].astype(str).str.rsplit('_', n=1).str[0].to_numpy()
    tmp['_order'] = pd.to_numeric(features[row_col], errors='coerce').to_numpy(dtype=float) if row_col else np.arange(len(out), dtype=float)
    for _, grp in tmp.groupby('_group', sort=False):
        if len(grp) < max(window, poly + 2):
            continue
        order = grp.sort_values('_order')
        w = min(window, len(order) if len(order) % 2 == 1 else len(order) - 1)
        if w < poly + 2 or w <= 2:
            continue
        smoothed = savgol_filter(order['_tvt'].to_numpy(dtype=float), window_length=w, polyorder=min(poly, w - 1), mode='interp')
        out[order['_pos'].to_numpy(dtype=int)] = smoothed
    return out


def build_sidecar_submission_from_model_package() -> Path:
    package_root = _find_model_package_root()
    competition_root = _find_competition_root_for_model_package()
    manifest = _read_json_file(package_root / 'metadata' / 'model_package_manifest.json')
    global _SIDECAR_MODEL_PACKAGE_MISSING_VALUE_POLICY
    _SIDECAR_MODEL_PACKAGE_MISSING_VALUE_POLICY = manifest.get('missing_value_policy', 'native')
    _validate_model_package_manifest(package_root, manifest)
    sample = pd.read_csv(competition_root / 'sample_submission.csv')[['id']].copy()
    sample['id'] = sample['id'].astype(str)

    builder, builder_path = _load_sidecar_feature_builder(package_root)
    feature_frame = _call_sidecar_feature_builder(
        builder,
        data_dir=competition_root,
        sample=sample,
        package_root=package_root,
        manifest=manifest,
    )
    feature_columns_config = _read_json_file(package_root / _sidecar_manifest_path(manifest, 'feature_columns', 'feature_builders/feature_columns.json'))

    predictions = pd.DataFrame({'id': feature_frame['id'].astype(str).to_numpy()})
    report_rows = []
    for entry in manifest.get('models', []):
        pred_col = _sidecar_prediction_column(entry)
        model_type = entry.get('model_type')
        if model_type == 'direct_feature':
            source_col = entry.get('feature_column')
            if source_col not in feature_frame.columns:
                raise RuntimeError(f'Sidecar direct_feature source column missing: {source_col}')
            pred = pd.to_numeric(feature_frame[source_col], errors='coerce').to_numpy(dtype=float)
            if not np.isfinite(pred).all():
                raise RuntimeError(f'Sidecar direct_feature {source_col} produced non-finite values.')
            predictions[pred_col] = pred
            report_rows.append({
                'prediction_column': pred_col,
                'model_type': model_type,
                'feature_count': 1,
                'source_column': source_col,
                'target_space': entry.get('target_space', 'delta'),
                'pred_mean': float(np.nanmean(pred)),
                'pred_std': float(np.nanstd(pred)),
                'pred_min': float(np.nanmin(pred)),
                'pred_max': float(np.nanmax(pred)),
            })
            continue
        columns = _feature_columns_for_sidecar_model(feature_columns_config, entry)
        model = _load_sidecar_model(package_root, entry)
        pred = _predict_sidecar_model(model, model_type, feature_frame, columns, entry)
        predictions[pred_col] = pred
        report_rows.append({
            'prediction_column': pred_col,
            'model_type': model_type,
            'feature_count': len(columns),
            'source_column': '',
            'target_space': entry.get('target_space', 'delta'),
            'pred_mean': float(np.nanmean(pred)),
            'pred_std': float(np.nanstd(pred)),
            'pred_min': float(np.nanmin(pred)),
            'pred_max': float(np.nanmax(pred)),
        })

    blend_config = _read_json_file(package_root / _sidecar_manifest_path(manifest, 'blend_config', 'stacking/blend_config.json'))
    weight_source = _validate_sidecar_model_package_weight_source(manifest, blend_config)
    weights = _normalize_sidecar_model_package_weights(blend_config)
    _validate_sidecar_weights(weights, blend_config)
    missing_cols = [c for c in weights if c not in predictions.columns]
    if missing_cols:
        raise RuntimeError(f'Sidecar model package blend references missing prediction columns: {missing_cols}')

    target_space = blend_config.get('target_space') or blend_config.get('prediction_space') or manifest.get('target_space', 'delta')
    entry_spaces = {
        _sidecar_prediction_column(entry): entry.get('target_space', target_space)
        for entry in manifest.get('models', [])
    }
    wrong_spaces = {col: entry_spaces.get(col) for col in weights if entry_spaces.get(col, target_space) != target_space}
    if wrong_spaces:
        raise RuntimeError(f'Mixed sidecar target_space is not supported: {wrong_spaces}, blend={target_space!r}')

    pred_value = np.zeros(len(predictions), dtype=float)
    for col, weight in weights.items():
        pred_value += float(weight) * predictions[col].to_numpy(dtype=float)
    blend_intercept = _sidecar_blend_intercept(blend_config)
    if blend_intercept:
        pred_value += blend_intercept

    if target_space == 'delta':
        if 'last_known_TVT' not in feature_frame.columns:
            raise RuntimeError('Sidecar delta-space blend requires feature_frame["last_known_TVT"].')
        pred_value = _apply_sidecar_delta_postprocess(pred_value, blend_config, feature_frame)
        tvt = feature_frame['last_known_TVT'].to_numpy(dtype=float) + pred_value
    elif target_space == 'tvt':
        tvt = pred_value
    else:
        raise RuntimeError(f'Unsupported sidecar blend target_space={target_space!r}.')

    tvt = _apply_sidecar_savgol_if_requested(tvt, blend_config, feature_frame)

    clip_min = blend_config.get('tvt_clip_min')
    clip_max = blend_config.get('tvt_clip_max')
    if clip_min is not None or clip_max is not None:
        tvt = np.clip(tvt, -np.inf if clip_min is None else float(clip_min), np.inf if clip_max is None else float(clip_max))

    out = validate_submission_ids(pd.DataFrame({'id': feature_frame['id'].to_numpy(), 'tvt': tvt}), label='sidecar_model_package')
    out_path = WORK_DIR / f'{SIDECAR_HBLEND_NAME}.csv'
    out.to_csv(out_path, index=False)
    pd.DataFrame(report_rows).to_csv(WORK_DIR / 'sidecar_model_package_prediction_report.csv', index=False)
    weight_report = pd.DataFrame([
        {'prediction_column': k, 'weight': v, 'weight_source': weight_source}
        for k, v in weights.items()
    ])
    weight_report.to_csv(WORK_DIR / 'sidecar_model_package_blend_weights.csv', index=False)
    summary = pd.Series({
        'package_root': package_root.as_posix(),
        'competition_root': competition_root.as_posix(),
        'schema_version': manifest.get('schema_version'),
        'training_feature_version': manifest.get('training_feature_version', ''),
        'manifest_weight_source': manifest.get('weight_source', ''),
        'blend_config_weight_source': blend_config.get('weight_source', ''),
        'weight_source': weight_source,
        'feature_builder': builder_path.as_posix(),
        'rows': len(out),
        'target_space': target_space,
        'weight_sum': float(sum(weights.values())),
        'blend_intercept': float(blend_intercept),
        'postprocess': json.dumps(blend_config.get('postprocess', {}) or {}),
        'tvt_mean': float(out['tvt'].mean()),
        'tvt_std': float(out['tvt'].std()),
    })
    summary.to_csv(WORK_DIR / 'sidecar_model_package_summary.csv')
    display(summary.to_frame('value'))
    display(weight_report.sort_values('weight', ascending=False))
    return out_path


def find_or_build_sidecar_submission() -> Path | None:
    if SIDECAR_INTEGRATION_MODE == 'off':
        return None
    existing = find_existing_sidecar_submission()
    if SIDECAR_SUBMISSION_NAME and existing is None:
        raise FileNotFoundError(f'SIDECAR_SUBMISSION_NAME was set but not found: {SIDECAR_SUBMISSION_NAME}')
    if SIDECAR_SOURCE_MODE == 'existing_submission':
        return existing
    if SIDECAR_SOURCE_MODE == 'build_from_model_package':
        return build_sidecar_submission_from_model_package()
    if SIDECAR_SOURCE_MODE == 'existing_or_build' and existing is not None:
        return existing
    if SIDECAR_SOURCE_MODE in {'build_from_preds', 'existing_or_build'}:
        return build_sidecar_submission_from_branch_artifacts()
    return existing


def write_sidecar_pairwise_distance_report(sidecar_path: Path) -> pd.DataFrame:
    rows = []
    try:
        frames = []
        for name in ['9.537', '9.765', '9.956']:
            path = WORK_DIR / f'{name}.csv'
            if not path.exists():
                continue
            frame = validate_submission_ids(pd.read_csv(path), label=f'pairwise:{name}').rename(columns={'tvt': name})
            frames.append(frame[['id', name]])
        side = validate_submission_ids(pd.read_csv(sidecar_path), label='pairwise:sidecar').rename(columns={'tvt': 'sidecar'})
        frames.append(side[['id', 'sidecar']])
        if len(frames) < 2:
            return pd.DataFrame(rows)
        merged = frames[0]
        for frame in frames[1:]:
            merged = merged.merge(frame, on='id', how='inner')
        names = [c for c in merged.columns if c != 'id']
        for a_i, a in enumerate(names):
            for b in names[a_i + 1:]:
                diff = merged[a].to_numpy(dtype=float) - merged[b].to_numpy(dtype=float)
                rows.append({
                    'a': a,
                    'b': b,
                    'rows': int(len(merged)),
                    'rmse_between': float(np.sqrt(np.mean(diff * diff))),
                    'mae_between': float(np.mean(np.abs(diff))),
                    'max_abs': float(np.max(np.abs(diff))),
                })
        report = pd.DataFrame(rows)
        report.to_csv(WORK_DIR / 'sidecar_pairwise_distance_report.csv', index=False)
        if not report.empty:
            display(report)
        return report
    except Exception as exc:
        print('WARNING: sidecar pairwise distance report failed:', repr(exc))
        return pd.DataFrame(rows)


def copy_sidecar_to_working() -> Path | None:
    sidecar_path = find_or_build_sidecar_submission()
    if sidecar_path is None:
        message = 'No sidecar submission available. Attach a sidecar model package, branch preds, or set SIDECAR_INTEGRATION_MODE = "off".'
        if SIDECAR_STRICT:
            raise FileNotFoundError(message)
        print('WARNING:', message)
        return None
    try:
        side = validate_submission_ids(pd.read_csv(sidecar_path), label=f'sidecar:{Path(sidecar_path).name}')
    except RuntimeError as exc:
        if _looks_like_id_coverage_error(exc):
            _raise_current_sample_mismatch(f'sidecar:{Path(sidecar_path).name}', exc, sidecar=True)
        raise
    dst = WORK_DIR / f'{SIDECAR_HBLEND_NAME}.csv'
    if Path(sidecar_path).resolve() != dst.resolve():
        side.to_csv(dst, index=False)
    sidecar_discovery_report(Path(sidecar_path))
    write_sidecar_pairwise_distance_report(dst)
    return Path(sidecar_path)


def configure_final_hblend_params():
    params_h_blend.update({
        'path': '/kaggle/working/',
        'id_target': ['id', 'tvt'],
        'type_sort': ['asc/desc', HBLEND_ASC_WEIGHT, HBLEND_DESC_WEIGHT],
    })
    if SIDECAR_INTEGRATION_MODE == 'hblend_member':
        params_h_blend['subwts'] = list(SIDECAR_HBLEND_RANK_CORRECTION_WEIGHTS)
        params_h_blend['subm'] = [
            {'model': 'Model.6', 'name': '9.537', 'weight': SIDECAR_HBLEND_MODEL6_WEIGHT, 'color': 'navy'},
            {'model': 'Model.7', 'name': '9.765', 'weight': SIDECAR_HBLEND_MODEL7_WEIGHT, 'color': 'darkorange'},
            {'model': 'Model.4', 'name': '9.956', 'weight': SIDECAR_HBLEND_MODEL4_WEIGHT, 'color': 'darkgreen'},
            {'model': 'Sidecar', 'name': SIDECAR_HBLEND_NAME, 'weight': SIDECAR_HBLEND_WEIGHT, 'color': 'purple'},
        ]
    else:
        params_h_blend['subwts'] = list(HBLEND_RANK_CORRECTION_WEIGHTS)
        params_h_blend['subm'] = [
            {'model': 'Model.6', 'name': '9.537', 'weight': HBLEND_MODEL6_WEIGHT, 'color': 'navy'},
            {'model': 'Model.7', 'name': '9.765', 'weight': HBLEND_MODEL7_WEIGHT, 'color': 'darkorange'},
            {'model': 'Model.4', 'name': '9.956', 'weight': HBLEND_MODEL4_WEIGHT, 'color': 'darkgreen'},
        ]
    return params_h_blend


base_copy_report = copy_hblend_base_csvs_to_working()
sidecar_source = None
if SIDECAR_INTEGRATION_MODE == 'hblend_member':
    sidecar_source = copy_sidecar_to_working()

params_h_blend = configure_final_hblend_params()
if abs(sum(s['weight'] for s in params_h_blend['subm']) - 1.0) > 1e-8:
    raise RuntimeError(f"h-blend base weights must sum to 1.0: {params_h_blend['subm']}")
if abs(sum(params_h_blend['subwts'])) > 1e-8:
    raise RuntimeError(f"h-blend rank correction weights must sum to 0: {params_h_blend['subwts']}")

integration_summary = pd.Series({
    'standalone_sidecar_builder': True,
    'sidecar_integration_mode': SIDECAR_INTEGRATION_MODE,
    'sidecar_source_mode': SIDECAR_SOURCE_MODE,
    'run_generated_model_cells': RUN_GENERATED_MODEL_CELLS,
    'sidecar_source': sidecar_source.as_posix() if sidecar_source else '',
    'hblend_path': params_h_blend['path'],
    'hblend_names': ','.join(s['name'] for s in params_h_blend['subm']),
    'hblend_weights': ','.join(str(s['weight']) for s in params_h_blend['subm']),
    'rank_correction': ','.join(str(x) for x in params_h_blend['subwts']),
})
integration_summary.to_csv('sidecar_hblend_integration_summary.csv')
display(base_copy_report)
display(integration_summary.to_frame('value'))

df = h_blend(params_h_blend, details=HBLEND_DETAILS, subm='cross.csv')


In [ ]:
# Keep the staged h-blend input CSVs in /kaggle/working.
# The candidate-generation cell reuses 9.537.csv / 9.765.csv / 9.956.csv / sidecar_stack.csv.
# Only transient diagnostic outputs are cleaned here.
for file in 'cross,tida_desc,demo_submission'.split(','):
    if os.path.isfile(file + '.csv'):
        os.remove(file + '.csv')


## Submit

In [ ]:
def apply_sidecar_late_blend(main_df: pd.DataFrame, late_weight: float | None = None, label: str = 'late_linear') -> pd.DataFrame:
    weight = float(SIDECAR_LATE_BLEND_WEIGHT if late_weight is None else late_weight)
    base = validate_submission_ids(main_df[['id', 'tvt']].copy(), label=f'{label}:base_hblend')
    if weight <= 0:
        return base

    sidecar_path = find_or_build_sidecar_submission()
    if sidecar_path is None:
        message = 'No sidecar submission file found. Attach branch artifacts or set SIDECAR_LATE_BLEND_WEIGHT = 0.0.'
        if SIDECAR_STRICT:
            raise FileNotFoundError(message)
        print('WARNING:', message)
        return base

    side = validate_submission_ids(pd.read_csv(sidecar_path), label=f'{label}:sidecar')
    merged = base.rename(columns={'tvt': 'tvt_hblend'}).merge(
        side[['id', 'tvt']].rename(columns={'tvt': 'tvt_sidecar'}),
        on='id',
        how='left',
    )
    missing = int(merged['tvt_sidecar'].isna().sum())
    if missing:
        missing_ids = merged.loc[merged['tvt_sidecar'].isna(), 'id'].head(10).tolist()
        raise RuntimeError(f'Missing sidecar predictions after id merge: {missing}; examples={missing_ids}')

    merged['tvt'] = (1.0 - weight) * merged['tvt_hblend'].astype(float) + weight * merged['tvt_sidecar'].astype(float)
    out = validate_submission_ids(merged[['id', 'tvt']], label=f'{label}:final')

    summary = pd.Series({
        'sidecar_enabled': True,
        'sidecar_weight': weight,
        'sidecar_file': Path(sidecar_path).as_posix(),
        'hblend_tvt_mean': float(merged['tvt_hblend'].mean()),
        'sidecar_tvt_mean': float(merged['tvt_sidecar'].mean()),
        'final_tvt_mean': float(out['tvt'].mean()),
        'mean_abs_sidecar_diff': float(np.mean(np.abs(merged['tvt_hblend'].to_numpy(dtype=float) - merged['tvt_sidecar'].to_numpy(dtype=float)))),
        'rows': len(out),
    })
    summary.to_csv(f'sidecar_{label}_summary.csv')
    display(summary.to_frame('value'))
    return out


def apply_sidecar_gated_late_blend(
    main_df: pd.DataFrame,
    max_weight: float | None = None,
    scale: float | None = None,
    label: str = 'gated_late_linear',
) -> pd.DataFrame:
    max_w = float(SIDECAR_GATED_MAX_WEIGHT if max_weight is None else max_weight)
    gate_scale = float(SIDECAR_GATED_SCALE if scale is None else scale)
    if gate_scale <= 0:
        raise ValueError('SIDECAR_GATED_SCALE must be positive.')

    base = validate_submission_ids(main_df[['id', 'tvt']].copy(), label=f'{label}:base_hblend')
    if max_w <= 0:
        return base

    sidecar_path = find_or_build_sidecar_submission()
    if sidecar_path is None:
        message = 'No sidecar submission file found. Attach a model package or set SIDECAR_GATED_MAX_WEIGHT = 0.0.'
        if SIDECAR_STRICT:
            raise FileNotFoundError(message)
        print('WARNING:', message)
        return base

    side = validate_submission_ids(pd.read_csv(sidecar_path), label=f'{label}:sidecar')
    merged = base.rename(columns={'tvt': 'tvt_hblend'}).merge(
        side[['id', 'tvt']].rename(columns={'tvt': 'tvt_sidecar'}),
        on='id',
        how='left',
    )
    missing = int(merged['tvt_sidecar'].isna().sum())
    if missing:
        missing_ids = merged.loc[merged['tvt_sidecar'].isna(), 'id'].head(10).tolist()
        raise RuntimeError(f'Missing sidecar predictions after id merge: {missing}; examples={missing_ids}')

    h = merged['tvt_hblend'].to_numpy(dtype=float)
    s = merged['tvt_sidecar'].to_numpy(dtype=float)
    diff = np.abs(s - h)
    gate = max_w / (1.0 + (diff / gate_scale) ** 2)
    merged['sidecar_gate'] = gate
    merged['tvt'] = (1.0 - gate) * h + gate * s
    out = validate_submission_ids(merged[['id', 'tvt']], label=f'{label}:final')

    summary = pd.Series({
        'sidecar_enabled': True,
        'sidecar_mode': 'gated_late_linear',
        'sidecar_max_weight': max_w,
        'sidecar_scale': gate_scale,
        'sidecar_file': Path(sidecar_path).as_posix(),
        'gate_mean': float(np.mean(gate)),
        'gate_p95': float(np.quantile(gate, 0.95)),
        'gate_max': float(np.max(gate)),
        'hblend_tvt_mean': float(merged['tvt_hblend'].mean()),
        'sidecar_tvt_mean': float(merged['tvt_sidecar'].mean()),
        'final_tvt_mean': float(out['tvt'].mean()),
        'mean_abs_sidecar_diff': float(np.mean(diff)),
        'p95_abs_sidecar_diff': float(np.quantile(diff, 0.95)),
        'max_abs_sidecar_diff': float(np.max(diff)),
        'rows': len(out),
    })
    summary.to_csv(f'sidecar_{label}_summary.csv')
    display(summary.to_frame('value'))
    return out


def make_hblend_member_params(sidecar_weight: float, rank_correction: list[float] | None = None) -> dict:
    model7_weight = float(SIDECAR_HBLEND_MODEL7_WEIGHT)
    model4_weight = float(SIDECAR_HBLEND_MODEL4_WEIGHT)
    model6_weight = float(1.0 - model7_weight - model4_weight - float(sidecar_weight))
    if model6_weight < 0:
        raise RuntimeError(f'Invalid sidecar candidate weight {sidecar_weight}: model6 weight would be negative.')
    rank_correction = list(SIDECAR_HBLEND_RANK_CORRECTION_WEIGHTS if rank_correction is None else rank_correction)
    params = {
        'path': '/kaggle/working/',
        'id_target': ['id', 'tvt'],
        'type_sort': ['asc/desc', HBLEND_ASC_WEIGHT, HBLEND_DESC_WEIGHT],
        'subwts': rank_correction,
        'subm': [
            {'model': 'Model.6', 'name': '9.537', 'weight': model6_weight, 'color': 'navy'},
            {'model': 'Model.7', 'name': '9.765', 'weight': model7_weight, 'color': 'darkorange'},
            {'model': 'Model.4', 'name': '9.956', 'weight': model4_weight, 'color': 'darkgreen'},
            {'model': 'Sidecar', 'name': SIDECAR_HBLEND_NAME, 'weight': float(sidecar_weight), 'color': 'purple'},
        ],
    }
    if abs(sum(s['weight'] for s in params['subm']) - 1.0) > 1e-8:
        raise RuntimeError(f'Candidate h-blend weights do not sum to 1: {params["subm"]}')
    if abs(sum(params['subwts'])) > 1e-8:
        raise RuntimeError(f'Candidate rank corrections do not sum to 0: {params["subwts"]}')
    return params


def make_original_hblend_params() -> dict:
    return {
        'path': '/kaggle/working/',
        'id_target': ['id', 'tvt'],
        'type_sort': ['asc/desc', HBLEND_ASC_WEIGHT, HBLEND_DESC_WEIGHT],
        'subwts': list(HBLEND_RANK_CORRECTION_WEIGHTS),
        'subm': [
            {'model': 'Model.6', 'name': '9.537', 'weight': HBLEND_MODEL6_WEIGHT, 'color': 'navy'},
            {'model': 'Model.7', 'name': '9.765', 'weight': HBLEND_MODEL7_WEIGHT, 'color': 'darkorange'},
            {'model': 'Model.4', 'name': '9.956', 'weight': HBLEND_MODEL4_WEIGHT, 'color': 'darkgreen'},
        ],
    }


def write_candidate_outputs() -> pd.DataFrame:
    rows = []
    if not WRITE_ADDITIONAL_SUBMISSION_CANDIDATES or SIDECAR_INTEGRATION_MODE == 'off':
        return pd.DataFrame(rows)

    # h_blend() reads member CSVs from /kaggle/working. Re-stage them here because
    # earlier diagnostic cells may have cleaned cross outputs, and candidate generation
    # runs after the selected h-blend has already completed.
    copy_hblend_base_csvs_to_working()

    # Ensure sidecar_stack.csv exists before member/late candidate generation.
    working_sidecar = WORK_DIR / f'{SIDECAR_HBLEND_NAME}.csv'
    sidecar_path = working_sidecar if working_sidecar.exists() else find_or_build_sidecar_submission()
    if sidecar_path is None:
        return pd.DataFrame(rows)
    sidecar_valid = validate_submission_ids(pd.read_csv(sidecar_path), label='candidate_sidecar')
    sidecar_valid.to_csv(working_sidecar, index=False)

    for weight in SIDECAR_MEMBER_CANDIDATE_WEIGHTS:
        label = f'hblend_sidecar_member_{int(round(float(weight) * 1000)):03d}'
        params = make_hblend_member_params(float(weight), rank_correction=SIDECAR_CANDIDATE_RANK_CORRECTION_WEIGHTS)
        cand = h_blend(params, details=HBLEND_DETAILS, subm=f'cross_{label}.csv')
        cand = validate_submission_ids(cand, label=label)
        out = WORK_DIR / f'submission_{label}.csv'
        cand.to_csv(out, index=False)
        rows.append({'candidate': label, 'mode': 'hblend_member', 'weight': float(weight), 'file': out.name, 'rows': len(cand)})

    for weight in LATE_LINEAR_CANDIDATE_WEIGHTS:
        label = f'hblend_late_sidecar_{int(round(float(weight) * 1000)):03d}'
        base = h_blend(make_original_hblend_params(), details=HBLEND_DETAILS, subm=f'cross_{label}_base.csv')
        cand = apply_sidecar_late_blend(base, late_weight=float(weight), label=label)
        out = WORK_DIR / f'submission_{label}.csv'
        cand.to_csv(out, index=False)
        rows.append({'candidate': label, 'mode': 'late_linear', 'weight': float(weight), 'file': out.name, 'rows': len(cand)})

    for max_weight, scale in GATED_LATE_LINEAR_CANDIDATES:
        label = f'hblend_gated_sidecar_{int(round(float(max_weight) * 1000)):03d}_s{int(round(float(scale)))}'
        base = h_blend(make_original_hblend_params(), details=HBLEND_DETAILS, subm=f'cross_{label}_base.csv')
        cand = apply_sidecar_gated_late_blend(base, max_weight=float(max_weight), scale=float(scale), label=label)
        out = WORK_DIR / f'submission_{label}.csv'
        cand.to_csv(out, index=False)
        rows.append({
            'candidate': label,
            'mode': 'gated_late_linear',
            'weight': float(max_weight),
            'scale': float(scale),
            'file': out.name,
            'rows': len(cand),
        })

    report = pd.DataFrame(rows)
    report.to_csv(WORK_DIR / 'sidecar_candidate_submission_report.csv', index=False)
    if not report.empty:
        display(report)
    return report


if SIDECAR_INTEGRATION_MODE == 'late_linear':
    df = apply_sidecar_late_blend(df, late_weight=SIDECAR_LATE_BLEND_WEIGHT, label='selected_late_linear')
elif SIDECAR_INTEGRATION_MODE == 'gated_late_linear':
    df = apply_sidecar_gated_late_blend(
        df,
        max_weight=SIDECAR_GATED_MAX_WEIGHT,
        scale=SIDECAR_GATED_SCALE,
        label='selected_gated_late_linear',
    )
else:
    df = validate_submission_ids(df[['id', 'tvt']].copy(), label='selected_hblend_submission')
    pd.Series({
        'sidecar_enabled': SIDECAR_INTEGRATION_MODE == 'hblend_member',
        'sidecar_integration_mode': SIDECAR_INTEGRATION_MODE,
        'sidecar_late_blend_weight': 0.0,
        'sidecar_gated_max_weight': 0.0,
        'rows': len(df),
    }).to_csv('sidecar_late_blend_summary.csv')

candidate_report = write_candidate_outputs()

output_submission_name = f"submission_selected_{SIDECAR_INTEGRATION_MODE}.csv"
df = validate_submission_ids(df, label='final_hblend_sidecar_submission')
df.to_csv(output_submission_name, index=False)
df.to_csv('submission.csv', index=False)
pd.Series({
    'output_submission_name': output_submission_name,
    'submission_csv': 'submission.csv',
    'sidecar_integration_mode': SIDECAR_INTEGRATION_MODE,
    'sidecar_source_mode': SIDECAR_SOURCE_MODE,
    'sidecar_late_blend_weight': SIDECAR_LATE_BLEND_WEIGHT,
    'sidecar_gated_max_weight': SIDECAR_GATED_MAX_WEIGHT,
    'sidecar_gated_scale': SIDECAR_GATED_SCALE,
    'rows': len(df),
    'candidate_files': int(len(candidate_report)),
}).to_csv('sidecar_final_submission_summary.csv')
display(df.head())
